In [ ]:
import os
import pandas as pd
import pyreadstat

INPUT_FOLDER  = r"C:\Users\b22fa\Desktop\diplom unelgee\2008"
OUTPUT_FOLDER = r"C:\Users\b22fa\Desktop\diplom unelgee\2008_converted"
LARGE_FILE_MB = 30  

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

sav_files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(".sav")]
print(f"Нийт {len(sav_files)} файл олдлоо\n")

for filename in sav_files:
    filepath = os.path.join(INPUT_FOLDER, filename)
    basename = os.path.splitext(filename)[0]
    size_mb  = os.path.getsize(filepath) / (1024 * 1024)

    print(f"Боловсруулж байна: {filename}  ({size_mb:.1f} MB)")

    try:
        df, meta = pyreadstat.read_sav(filepath)

        if size_mb > LARGE_FILE_MB:
            out_path = os.path.join(OUTPUT_FOLDER, basename + ".csv")
            df.to_csv(out_path, index=False, encoding="utf-8-sig")
            print(f"  -> CSV: {out_path}")
        else:
            out_path = os.path.join(OUTPUT_FOLDER, basename + ".xlsx")
            df.to_excel(out_path, index=False)
            print(f"  -> Excel: {out_path}")

    except Exception as e:
        print(f"  АЛДАА: {e}")

print("\nДууслаа! Файлууд:", OUTPUT_FOLDER)

Нийт 18 файл олдлоо

Боловсруулж байна: Agriultural Equipment.sav  (0.1 MB)
  -> Excel: C:\Users\b22fa\Desktop\diplom unelgee\2008_converted\Agriultural Equipment.xlsx
Боловсруулж байна: Agriultural Expenditure.sav  (0.0 MB)
  -> Excel: C:\Users\b22fa\Desktop\diplom unelgee\2008_converted\Agriultural Expenditure.xlsx
Боловсруулж байна: basicvars (1).sav  (0.5 MB)
  -> Excel: C:\Users\b22fa\Desktop\diplom unelgee\2008_converted\basicvars (1).xlsx
Боловсруулж байна: By Product.sav  (0.6 MB)
  -> Excel: C:\Users\b22fa\Desktop\diplom unelgee\2008_converted\By Product.xlsx
Боловсруулж байна: Crop.sav  (0.1 MB)
  -> Excel: C:\Users\b22fa\Desktop\diplom unelgee\2008_converted\Crop.xlsx
Боловсруулж байна: Durable.sav  (7.1 MB)
  -> Excel: C:\Users\b22fa\Desktop\diplom unelgee\2008_converted\Durable.xlsx
Боловсруулж байна: Enterprise.sav  (0.3 MB)
  -> Excel: C:\Users\b22fa\Desktop\diplom unelgee\2008_converted\Enterprise.xlsx
Боловсруулж байна: Household (1).sav  (1.5 MB)
  -> Excel: C:\Users\

In [26]:
import os
import pandas as pd
import pyreadstat
import numpy as np

FOLDER = r"C:\Users\b22fa\Desktop\diplom unelgee\2008"
ID = "identif"

def load(filename):
    path = os.path.join(FOLDER, filename + ".sav")
    df, _ = pyreadstat.read_sav(path)
    df.columns = df.columns.str.lower()
    return df

def save(df, name):
    size = df.memory_usage(deep=True).sum() / (1024*1024)
    out = os.path.join(FOLDER, name)
    if size > 30:
        df.to_csv(out + ".csv", index=False, encoding="utf-8-sig")
        print(f"  Saved CSV: {name}.csv")
    else:
        df.to_excel(out + ".xlsx", index=False)
        print(f"  Saved Excel: {name}.xlsx")

# ── Load all files ───────────────────────────────────────────────
print("Loading files...")
wage       = load("Wage job")
other_inc  = load("Other income")
crop       = load("Crop")
agri_exp   = load("Agriultural Expenditure")
livestock  = load("Llivestock")
live_exp   = load("Livestock Expenditure")
enterprise = load("Enterprise")
remittance = load("Remittance")
nonfood    = load("Non-Food")
urban_food = load("Urban Food 30 day")
rural_food = load("Rural Food 30 day")
individual = load("Indivdual (1)")
basicvars  = load("basicvars (1)")   # hhweight энд байна

# ── 1. Цалин хөлс ───────────────────────────────────────────────
print("1. Цалин хөлс...")
w_cols = [c for c in ["q0707a","q0707b","q0707c"] if c in wage.columns]
wage["_s"] = wage[w_cols].sum(axis=1, min_count=1)
tsalin = wage.groupby(ID)["_s"].sum().reset_index()
tsalin.columns = [ID, "tsalin"]

# ── 2. Тэтгэвэр тэтгэмж бусад ───────────────────────────────────
print("2. Тэтгэвэр тэтгэмж бусад...")
t_cols = [c for c in ["q1102_t","q1103_t","q1104_t","q1105_t","q1106_t"] if c in other_inc.columns]
other_inc["_s"] = other_inc[t_cols].sum(axis=1, min_count=1)
tetgever = other_inc.groupby(ID)["_s"].sum().reset_index()
tetgever.columns = [ID, "tetgever"]

# ── 3. Үйлдвэрлэл үйлчилгээ ─────────────────────────────────────
print("3. Үйлдвэрлэл үйлчилгээ...")

crop_g  = crop.groupby(ID)["q0909t"].sum().reset_index().rename(columns={"q0909t":"crop_inc"})
agri_g  = agri_exp.groupby(ID)["q0911"].sum().reset_index().rename(columns={"q0911":"agri_exp"})
live_g  = livestock.groupby(ID)["q0918"].sum().reset_index().rename(columns={"q0918":"live_inc"})
livee_g = live_exp.groupby(ID)["q0920"].sum().reset_index().rename(columns={"q0920":"live_exp"})
ent_i   = enterprise.groupby(ID)["q1014"].sum().reset_index().rename(columns={"q1014":"ent_inc"})
ent_e   = enterprise.groupby(ID)["q1010"].sum().reset_index().rename(columns={"q1010":"ent_exp"})

uil = crop_g.merge(agri_g,  on=ID, how="outer") \
            .merge(live_g,  on=ID, how="outer") \
            .merge(livee_g, on=ID, how="outer") \
            .merge(ent_i,   on=ID, how="outer") \
            .merge(ent_e,   on=ID, how="outer").fillna(0)
uil["uildverlelal"] = (uil["crop_inc"] - uil["agri_exp"]) + \
                      (uil["live_inc"] - uil["live_exp"]) + \
                      (uil["ent_inc"]  - uil["ent_exp"])
uildverlelal = uil[[ID, "uildverlelal"]]

# ── 4. Бэлэг тусламж өөрийн аж ахуйгаас хэрэглэсэн ─────────────
print("4. Бэлэг тусламж өөрийн аж ахуй...")

# Remittance: q1111
rem_g = remittance.groupby(ID)["q1111"].sum().reset_index().rename(columns={"q1111":"remittance"})

# Non-Food: q1504
nf_g  = nonfood.groupby(ID)["q1504"].sum().reset_index().rename(columns={"q1504":"nonfood"})

# Urban Food: q1605 * q1607 (худалдаж авсан үнэ), *12
urban_food["_uf7"] = urban_food["q1605"] * urban_food["q1607"]
uf7_g = urban_food.groupby(ID)["_uf7"].sum().reset_index()
uf7_g["uf7_annual"] = uf7_g["_uf7"] * 12

# Rural Food: q1804 * q1807 (худалдаж авсан үнэ), *12
rural_food["_rf7"] = rural_food["q1804"] * rural_food["q1807"]
rf7_g = rural_food.groupby(ID)["_rf7"].sum().reset_index()
rf7_g["rf7_annual"] = rf7_g["_rf7"] * 12

# Individual: q0223 + q0227 per identif
ind_cols = [c for c in ["q0223","q0227"] if c in individual.columns]
individual["_ind"] = individual[ind_cols].sum(axis=1, min_count=1)
ind_g = individual.groupby(ID)["_ind"].sum().reset_index().rename(columns={"_ind":"ind_val"})

# Urban Food: q1605 * q1606 (өөрийн үйлдвэрлэсэн үнэ), *12
urban_food["_uf6"] = urban_food["q1605"] * urban_food["q1606"]
uf6_g = urban_food.groupby(ID)["_uf6"].sum().reset_index()
uf6_g["uf6_annual"] = uf6_g["_uf6"] * 12

# Rural Food: q1804 * q1806 (өөрийн үйлдвэрлэсэн үнэ), *12
rural_food["_rf6"] = rural_food["q1804"] * rural_food["q1806"]
rf6_g = rural_food.groupby(ID)["_rf6"].sum().reset_index()
rf6_g["rf6_annual"] = rf6_g["_rf6"] * 12

# Merge 4-р бүрэлдэхүүн бүгдийг
beleg_df = rem_g \
    .merge(nf_g,                     on=ID, how="outer") \
    .merge(uf7_g[[ID,"uf7_annual"]], on=ID, how="outer") \
    .merge(rf7_g[[ID,"rf7_annual"]], on=ID, how="outer") \
    .merge(ind_g,                    on=ID, how="outer") \
    .merge(uf6_g[[ID,"uf6_annual"]], on=ID, how="outer") \
    .merge(rf6_g[[ID,"rf6_annual"]], on=ID, how="outer") \
    .fillna(0)

beleg_df["beleg_tuslamj"] = (beleg_df["remittance"] +
                              beleg_df["nonfood"]     +
                              beleg_df["uf7_annual"]  +
                              beleg_df["rf7_annual"]  +
                              beleg_df["ind_val"]     +
                              beleg_df["uf6_annual"]  +
                              beleg_df["rf6_annual"])
beleg = beleg_df[[ID, "beleg_tuslamj"]]
# ── 4.5 Гэр бүлийн гишүүд ───────────────────────────────────────
print("4.5 Гэр бүлийн гишүүд...")

fam = individual[[ID, "q0105y"]].copy()
fam["q0105y"] = pd.to_numeric(fam["q0105y"], errors="coerce")

children_g     = fam[fam["q0105y"] < 14].groupby(ID).size().reset_index(name="children")
adults_g       = fam[fam["q0105y"] >= 14].groupby(ID).size().reset_index(name="_adults")
adults_g["leading_adult"]  = 1
adults_g["other_adults"]   = (adults_g["_adults"] - 1).clip(lower=0)
adults_g = adults_g[[ID, "leading_adult", "other_adults"]]

family = basicvars[[ID]].merge(children_g, on=ID, how="left") \
                         .merge(adults_g,   on=ID, how="left") \
                         .fillna(0)
family[["children","leading_adult","other_adults"]] = \
    family[["children","leading_adult","other_adults"]].astype(int)

# ── 5. Нийт орлого — merge all ───────────────────────────────────
print("5. Нийт орлого нэгтгэж байна...")

data2008 = basicvars.copy()
for df_ in [tsalin, tetgever, uildverlelal, beleg, family]:
    data2008 = data2008.merge(df_, on=ID, how="left")

data2008 = data2008.fillna(0)
data2008["niit_orlogo"] = (data2008["tsalin"] +
                            data2008["tetgever"] +
                            data2008["uildverlelal"] +
                            data2008["beleg_tuslamj"])

save(data2008, "data2008")
print(f"data2008: {len(data2008)} өрх")

# ── Дундаж тооцох ────────────────────────────────────────────────
print("\nДундаж тооцож байна...")

calc_cols = ["tsalin","tetgever","uildverlelal","beleg_tuslamj","niit_orlogo"]
labels    = ["1. Цалин хөлс",
             "2. Тэтгэвэр тэтгэмж бусад",
             "3. Үйлдвэрлэл үйлчилгээ",
             "4. Бэлэг тусламж өөрийн аж ахуй",
             "5. Нийт орлого"]

# Жингүй дундаж
simple_avg = [data2008[c].mean() for c in calc_cols]

# Жинлэсэн дундаж (hhweight)
weight_col = "hhweight"
if weight_col in data2008.columns:
    w = data2008[weight_col].fillna(0)
    weighted_avg = [
        np.average(data2008[c], weights=w) if w.sum() > 0 else np.nan
        for c in calc_cols
    ]
else:
    print(f"WARNING: '{weight_col}' багана олдсонгүй!")
    weighted_avg = [np.nan] * len(calc_cols)

avg_df = pd.DataFrame({
    "Орлогын төрөл":   labels,
    "Жингүй дундаж":   simple_avg,
    "Жинлэсэн дундаж": weighted_avg
})

print("\n── 2008 оны дундаж үзүүлэлтүүд ──")
print(avg_df.to_string(index=False))

save(avg_df, "averages_2008")
print("\nДууслаа!")

Loading files...
1. Цалин хөлс...
2. Тэтгэвэр тэтгэмж бусад...
3. Үйлдвэрлэл үйлчилгээ...
4. Бэлэг тусламж өөрийн аж ахуй...
4.5 Гэр бүлийн гишүүд...
5. Нийт орлого нэгтгэж байна...
  Saved Excel: data2008.xlsx
data2008: 11172 өрх

Дундаж тооцож байна...

── 2008 оны дундаж үзүүлэлтүүд ──
                  Орлогын төрөл  Жингүй дундаж  Жинлэсэн дундаж
                  1. Цалин хөлс   1.282563e+06     1.362052e+06
      2. Тэтгэвэр тэтгэмж бусад   6.897375e+05     7.079272e+05
        3. Үйлдвэрлэл үйлчилгээ   8.124604e+05     7.487661e+05
4. Бэлэг тусламж өөрийн аж ахуй   3.017128e+05     3.108644e+05
                 5. Нийт орлого   3.086473e+06     3.129610e+06
  Saved Excel: averages_2008.xlsx

Дууслаа!


In [ ]:
import os
import pandas as pd
import numpy as np

FOLDER = r"C:\Users\b22fa\Desktop\diplom unelgee\2010"
ID = "identif"

def load(filename):
    path = os.path.join(FOLDER, filename + ".dta")
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.lower()
    return df

def save(df, name):
    size = df.memory_usage(deep=True).sum() / (1024*1024)
    out = os.path.join(FOLDER, name)
    if size > 30:
        df.to_csv(out + ".csv", index=False, encoding="utf-8-sig")
        print(f"  Saved CSV: {name}.csv")
    else:
        df.to_excel(out + ".xlsx", index=False)
        print(f"  Saved Excel: {name}.xlsx")

# ── Load all files ────────────────────────────────────────────────
print("Loading files...")
indiv       = load("02_indiv (1)")
other_inc   = load("09_other_income (1)")
crop        = load("06_crop (1)")
agri_exp    = load("07_agric_exp (1)")
livestock   = load("03_livestock (1)")
live_exp    = load("04_livestock_exp (1)")
enterprise  = load("08_enterprise (1)")
remittance  = load("10_remittance (1)")
nonfood     = load("14_non_food (1)")
urban_diary = load("15_urb_diary (1)")
rural_food  = load("16_rur_food_7d (1)")
basicvars   = load("basicvars (1)")

# ════════════════════════════════════════════════════════════════
# HOUSEHOLD COMPOSITION from indiv file
# Children     : q0105y < 14
# Leading adult: 1 if any member q0105y >= 14
# Other adults : count(q0105y >= 14) - 1
# Total members: all rows per identif
# ════════════════════════════════════════════════════════════════
print("\nӨрхийн бүрэлдэхүүн тооцож байна...")

hh_comp = indiv.copy()
age_col = "q0105y"
if age_col not in hh_comp.columns:
    print(f"  WARNING: {age_col} not found in indiv!")
    hh_comp[age_col] = np.nan

hh_comp["is_child"]  = (hh_comp[age_col].fillna(0) < 14).astype(int)
hh_comp["is_adult"]  = (hh_comp[age_col].fillna(0) >= 14).astype(int)
hh_comp["is_member"] = 1

comp_g = hh_comp.groupby(ID).agg(
    children      = ("is_child",  "sum"),
    total_adults  = ("is_adult",  "sum"),
    total_members = ("is_member", "sum")
).reset_index()

comp_g["leading_adult"] = (comp_g["total_adults"] >= 1).astype(int)
comp_g["other_adults"]  = (comp_g["total_adults"] - comp_g["leading_adult"]).clip(lower=0)
comp_g = comp_g[[ID, "children", "leading_adult", "other_adults", "total_members"]]

print(f"  Дундаж хүүхэд       : {comp_g['children'].mean():.2f}")
print(f"  Дундаж бусад насанд : {comp_g['other_adults'].mean():.2f}")
print(f"  Дундаж гишүүд       : {comp_g['total_members'].mean():.2f}")

# ════════════════════════════════════════════════════════════════
# 1. ЦАЛИН ХӨЛС
#    File: 02_indiv  →  q0416b + q0416c  (horizontal)
#    then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n1. Цалин хөлс...")

wage = indiv.copy()
w_cols = ["q0416b", "q0416c"]
existing_w = [c for c in w_cols if c in wage.columns]

if existing_w:
    wage["row_sum"] = wage[existing_w].fillna(0).sum(axis=1)
else:
    print("  WARNING: q0416b / q0416c not found!")
    wage["row_sum"] = 0

tsalin = (wage.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tsalin"}))

print(f"  Нийт өрх  : {len(tsalin)}")
print(f"  Дундаж    : {tsalin['tsalin'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 2. ТЭТГЭВЭР ТЭТГЭМЖ БУСАД
#    File: 09_other_income  →  q0702_t + q0703_t + q0704_t + q0705_t + q0706_t  (horizontal)
#    then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n2. Тэтгэвэр тэтгэмж бусад...")

oi = other_inc.copy()
t_cols = ["q0702_t","q0703_t","q0704_t","q0705_t","q0706_t"]
existing_t = [c for c in t_cols if c in oi.columns]

if existing_t:
    oi["row_sum"] = oi[existing_t].fillna(0).sum(axis=1)
else:
    print("  WARNING: other_income columns not found!")
    oi["row_sum"] = 0

tetgever = (oi.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tetgever"}))

print(f"  Нийт өрх  : {len(tetgever)}")
print(f"  Дундаж    : {tetgever['tetgever'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 3. ҮЙЛДВЭРЛЭЛ ҮЙЛЧИЛГЭЭ
#    (Crop q0523t  -  AgriExp q0524)           by identif
#  + (Livestock q0505  -  LiveExp q0506)       by identif
#  + Enterprise ((q0609_1a*q0609_1b + q0609_2a*q0609_2b) - q0607_10)
# ════════════════════════════════════════════════════════════════
print("\n3. Үйлдвэрлэл үйлчилгээ...")

# Crop income
crop_g = (crop.groupby(ID)["q0523t"]
               .sum()
               .reset_index()
               .rename(columns={"q0523t": "crop_inc"}))

# Agricultural expenditure
agri_g = (agri_exp.groupby(ID)["q0524"]
                   .sum()
                   .reset_index()
                   .rename(columns={"q0524": "agri_exp"}))

# Livestock income
live_g = (livestock.groupby(ID)["q0505"]
                    .sum()
                    .reset_index()
                    .rename(columns={"q0505": "live_inc"}))

# Livestock expenditure
livee_g = (live_exp.groupby(ID)["q0506"]
                    .sum()
                    .reset_index()
                    .rename(columns={"q0506": "live_exp"}))

# Enterprise
ent = enterprise.copy()
ent["ent_inc"] = 0.0

for a, b in [("q0609_1a", "q0609_1b"),
             ("q0609_2a", "q0609_2b"),
             ("q0609_3a", "q0609_3b")]:
    if a in ent.columns and b in ent.columns:
        ent["ent_inc"] += ent[a].fillna(0) * ent[b].fillna(0)
    else:
        print(f"  WARNING: {a} or {b} not found in enterprise!")

ent["ent_exp"] = ent["q0607_10"].fillna(0) if "q0607_10" in ent.columns else 0.0
ent["ent_net"] = ent["ent_inc"] - ent["ent_exp"]

ent_g = (ent.groupby(ID)["ent_net"]
             .sum()
             .reset_index())

# Merge and calculate
uil = (crop_g
       .merge(agri_g,  on=ID, how="outer")
       .merge(live_g,  on=ID, how="outer")
       .merge(livee_g, on=ID, how="outer")
       .merge(ent_g,   on=ID, how="outer")
       .fillna(0))

uil["uildverlelal"] = ((uil["crop_inc"] - uil["agri_exp"]) +
                       (uil["live_inc"] - uil["live_exp"]) +
                        uil["ent_net"])

uildverlelal = uil[[ID, "uildverlelal"]]

print(f"  crop_inc     дундаж: {crop_g['crop_inc'].mean():,.0f}")
print(f"  agri_exp     дундаж: {agri_g['agri_exp'].mean():,.0f}")
print(f"  live_inc     дундаж: {live_g['live_inc'].mean():,.0f}")
print(f"  live_exp     дундаж: {livee_g['live_exp'].mean():,.0f}")
print(f"  ent_net      дундаж: {ent_g['ent_net'].mean():,.0f}")
print(f"  uildverlelal дундаж: {uil['uildverlelal'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 4. БЭЛЭГ ТУСЛАМЖ ӨӨРИЙН АЖ АХУй
#    Remittance   : q0712                          sum by identif
#    Individual   : q0214_3                        sum by identif
#    Non-Food     : q1004                          sum by identif
#    Urban Diary  : q1104*(q1101_3+q1101_4+
#                   q1102_3+q1102_4+q1103_3+q1103_4) horizontal
#                   then sum by identif × 12
#    Rural Food   : (q1204+q1205)*q1206            horizontal
#                   then sum by identif × 52
# ════════════════════════════════════════════════════════════════
print("\n4. Бэлэг тусламж өөрийн аж ахуй...")

# Remittance
rem_g = (remittance.groupby(ID)["q0712"]
                    .sum()
                    .reset_index()
                    .rename(columns={"q0712": "remittance"}))

# Individual q0214_3
if "q0214_3" in indiv.columns:
    indiv_g = (indiv.groupby(ID)["q0214_3"]
                     .sum()
                     .reset_index()
                     .rename(columns={"q0214_3": "indiv_inc"}))
else:
    print("  WARNING: q0214_3 not found in indiv!")
    indiv_g = pd.DataFrame({ID: basicvars[ID].unique(), "indiv_inc": 0})

# Non-Food
nf_g = (nonfood.groupby(ID)["q1004"]
                .sum()
                .reset_index()
                .rename(columns={"q1004": "nonfood"}))

# Urban Diary: q1104 × (q1101_3+q1101_4+q1102_3+q1102_4+q1103_3+q1103_4) → sum by identif → ×12
ud = urban_diary.copy()
qty_cols    = ["q1101_3","q1101_4","q1102_3","q1102_4","q1103_3","q1103_4"]
existing_ud = [c for c in qty_cols if c in ud.columns]

if existing_ud:
    ud["qty_sum"]   = ud[existing_ud].fillna(0).sum(axis=1)
    ud["row_value"] = ud["q1104"].fillna(0) * ud["qty_sum"]
else:
    print("  WARNING: Urban Diary qty columns not found!")
    ud["row_value"] = 0

ud_g = (ud.groupby(ID)["row_value"]
           .sum()
           .reset_index())
ud_g["ud_annual"] = ud_g["row_value"] * 12

# Rural Food 7-day: (q1204+q1205) × q1206 → sum by identif → ×52
rf = rural_food.copy()
for c in ["q1204","q1205","q1206"]:
    if c not in rf.columns:
        print(f"  WARNING: {c} not found in rural_food!")
        rf[c] = 0

rf["row_value"] = (rf["q1204"].fillna(0) + rf["q1205"].fillna(0)) * rf["q1206"].fillna(0)

rf_g = (rf.groupby(ID)["row_value"]
           .sum()
           .reset_index())
rf_g["rf_annual"] = rf_g["row_value"] * 52

# Merge all components
beleg_df = (rem_g
    .merge(indiv_g,                 on=ID, how="outer")
    .merge(nf_g,                    on=ID, how="outer")
    .merge(ud_g[[ID,"ud_annual"]],  on=ID, how="outer")
    .merge(rf_g[[ID,"rf_annual"]],  on=ID, how="outer")
    .fillna(0))

beleg_df["beleg_tuslamj"] = (beleg_df["remittance"] +
                              beleg_df["indiv_inc"]  +
                              beleg_df["nonfood"]    +
                              beleg_df["ud_annual"]  +
                              beleg_df["rf_annual"])

beleg = beleg_df[[ID, "beleg_tuslamj"]]

print(f"  remittance    дундаж: {beleg_df['remittance'].mean():,.0f}")
print(f"  indiv_inc     дундаж: {beleg_df['indiv_inc'].mean():,.0f}")
print(f"  nonfood       дундаж: {beleg_df['nonfood'].mean():,.0f}")
print(f"  ud_annual     дундаж: {beleg_df['ud_annual'].mean():,.0f}")
print(f"  rf_annual     дундаж: {beleg_df['rf_annual'].mean():,.0f}")
print(f"  beleg_tuslamj дундаж: {beleg_df['beleg_tuslamj'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 5. НИЙТ ОРЛОГО — merge all 4 + household composition → data2010
# ════════════════════════════════════════════════════════════════
print("\n5. Нийт орлого нэгтгэж байна...")

data2010 = basicvars.copy()

for df_ in [tsalin, tetgever, uildverlelal, beleg, comp_g]:
    data2010 = data2010.merge(df_, on=ID, how="left")

data2010 = data2010.fillna(0)

data2010["niit_orlogo"] = (data2010["tsalin"]        +
                            data2010["tetgever"]      +
                            data2010["uildverlelal"]  +
                            data2010["beleg_tuslamj"])

data2010 = data2010[data2010["niit_orlogo"] != 0].reset_index(drop=True)
print(f"  0 орлоготой өрх хасагдсаны дараа: {len(data2010)}")
print(f"  Нийт өрх   : {len(data2010)}")
print(f"  Багана     : {list(data2010.columns)}")

# ════════════════════════════════════════════════════════════════
# ДУНДАЖ — simple and weighted
# ════════════════════════════════════════════════════════════════
print("\nДундаж тооцож байна...")

calc_cols = ["tsalin","tetgever","uildverlelal","beleg_tuslamj","niit_orlogo"]
labels    = ["1. Цалин хөлс",
             "2. Тэтгэвэр тэтгэмж бусад",
             "3. Үйлдвэрлэл үйлчилгээ",
             "4. Бэлэг тусламж өөрийн аж ахуй",
             "5. Нийт орлого"]

# Simple (unweighted) average
simple_avg = [data2010[c].mean() for c in calc_cols]

# Weighted average using hhweight
weight_col = "hhweight"
if weight_col in data2010.columns:
    w = data2010[weight_col].fillna(0)
    weighted_avg = [
        np.average(data2010[c], weights=w) if w.sum() > 0 else np.nan
        for c in calc_cols
    ]
    monthly_wtd = [v / 12 for v in weighted_avg]
else:
    print(f"  WARNING: '{weight_col}' not found in data2010!")
    weighted_avg = [np.nan] * len(calc_cols)
    monthly_wtd  = [np.nan] * len(calc_cols)

# Results table
avg_df = pd.DataFrame({
    "Орлогын төрөл"           : labels,
    "Жингүй дундаж"           : simple_avg,
    "Жинлэсэн дундаж"         : weighted_avg,
    "Сарын дундаж (жинлэсэн)" : monthly_wtd
})

print("\n── 2010 оны дундаж үзүүлэлтүүд ──")
print(avg_df.to_string(index=False))

# Save final data
save(data2010, "data2010")
save(avg_df,   "avg2010")

Loading files...

Өрхийн бүрэлдэхүүн тооцож байна...
  Дундаж хүүхэд       : 0.97
  Дундаж бусад насанд : 1.90
  Дундаж гишүүд       : 3.87

1. Цалин хөлс...
  Нийт өрх  : 11198
  Дундаж    : 2,245,129

2. Тэтгэвэр тэтгэмж бусад...
  Нийт өрх  : 11191
  Дундаж    : 880,089

3. Үйлдвэрлэл үйлчилгээ...
  crop_inc     дундаж: 1,071,968
  agri_exp     дундаж: 450,870
  live_inc     дундаж: 753,389
  live_exp     дундаж: 424,982
  ent_net      дундаж: 3,675,598
  uildverlelal дундаж: 1,532,103

4. Бэлэг тусламж өөрийн аж ахуй...
  remittance    дундаж: 371,681
  indiv_inc     дундаж: 23,281
  nonfood       дундаж: 109,428
  ud_annual     дундаж: 18,900
  rf_annual     дундаж: 826
  beleg_tuslamj дундаж: 524,115

5. Нийт орлого нэгтгэж байна...
  0 орлоготой өрх хасагдсаны дараа: 11195
  Нийт өрх   : 11195
  Багана     : ['identif', 'hhsize', 'cluster', 'aimag', 'location', 'urban', 'region', 'month', 'quarter', 'hhweight', 'strata', 'stratum', 'tsalin', 'tetgever', 'uildverlelal', 'beleg_tu

In [ ]:
import os
import pandas as pd
import numpy as np

FOLDER = r"C:\Users\b22fa\Desktop\diplom unelgee\2012"
ID = "identif"

def load(filename):
    path = os.path.join(FOLDER, filename + ".dta")
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.lower()
    return df

def save(df, name):
    size = df.memory_usage(deep=True).sum() / (1024*1024)
    out = os.path.join(FOLDER, name)
    if size > 30:
        df.to_csv(out + ".csv", index=False, encoding="utf-8-sig")
        print(f"  Saved CSV: {name}.csv")
    else:
        df.to_excel(out + ".xlsx", index=False)
        print(f"  Saved Excel: {name}.xlsx")

# ── Load all files ────────────────────────────────────────────────
# ── Load all files ────────────────────────────────────────────────
print("Loading files...")
indiv       = load("02_indiv (3)")
other_inc   = load("09_other_income (3)")
crop        = load("06_crop (3)")
agri_exp    = load("07_agric_exp (3)")
livestock   = load("03_livestock (3)")
live_exp    = load("04_livestock_exp (3)")
enterprise  = load("08_enterprise (3)")
remittance  = load("10_remittance (3)")
energy      = load("12_energy")
payment     = load("13_payment_serv")
foodout     = load("19_foodout")
nonfood     = load("15_non_food")
rural_food  = load("17_rur_food_7d")
basicvars   = load("basicvars (3)")

# ════════════════════════════════════════════════════════════════
# HOUSEHOLD COMPOSITION from indiv file
# Children    : q0105y < 14
# Leading adult: 1 if any member q0105y >= 14
# Other adults : count(q0105y >= 14) - 1
# Total members: all rows per identif
# ════════════════════════════════════════════════════════════════
print("\nӨрхийн бүрэлдэхүүн тооцож байна...")

hh_comp = indiv.copy()

# Age column
age_col = "q0105y"
if age_col not in hh_comp.columns:
    print(f"  WARNING: {age_col} not found in indiv!")
    hh_comp[age_col] = np.nan

hh_comp["is_child"]       = (hh_comp[age_col].fillna(0) < 14).astype(int)
hh_comp["is_adult"]       = (hh_comp[age_col].fillna(0) >= 14).astype(int)
hh_comp["is_member"]      = 1

comp_g = hh_comp.groupby(ID).agg(
    children     = ("is_child",  "sum"),
    total_adults = ("is_adult",  "sum"),
    total_members= ("is_member", "sum")
).reset_index()

# Leading adult: 1 if there is at least one adult, else 0
comp_g["leading_adult"] = (comp_g["total_adults"] >= 1).astype(int)

# Other adults: total adults minus the leading adult (min 0)
comp_g["other_adults"]  = (comp_g["total_adults"] - comp_g["leading_adult"]).clip(lower=0)

comp_g = comp_g[[ID, "children", "leading_adult", "other_adults", "total_members"]]

print(f"  Дундаж хүүхэд        : {comp_g['children'].mean():.2f}")
print(f"  Дундаж бусад насанд  : {comp_g['other_adults'].mean():.2f}")
print(f"  Дундаж гишүүд        : {comp_g['total_members'].mean():.2f}")

# ════════════════════════════════════════════════════════════════
# 1. ЦАЛИН ХӨЛС
#    File: 02_indiv → q0617_2 + q0617_3 + q0619_2 + q0619_3 (horizontal)
#    then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n1. Цалин хөлс...")

wage = indiv.copy()
w_cols    = ["q0617_2", "q0617_3", "q0619_2", "q0619_3"]
existing_w = [c for c in w_cols if c in wage.columns]

if existing_w:
    wage["row_sum"] = wage[existing_w].fillna(0).sum(axis=1)
else:
    print("  WARNING: wage columns not found!")
    wage["row_sum"] = 0

tsalin = (wage.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tsalin"}))

print(f"  Нийт өрх  : {len(tsalin)}")
print(f"  Дундаж    : {tsalin['tsalin'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 2. ТЭТГЭВЭР ТЭТГЭМЖ БУСАД
#    File: 09_other_income → q0902_t + q0903_t + q0904_t + q0905_t + q0906_t (horizontal)
#    then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n2. Тэтгэвэр тэтгэмж бусад...")

oi = other_inc.copy()
t_cols    = ["q0902_t","q0903_t","q0904_t","q0905_t","q0906_t"]
existing_t = [c for c in t_cols if c in oi.columns]

if existing_t:
    oi["row_sum"] = oi[existing_t].fillna(0).sum(axis=1)
else:
    print("  WARNING: other_income columns not found!")
    oi["row_sum"] = 0

tetgever = (oi.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tetgever"}))

print(f"  Нийт өрх  : {len(tetgever)}")
print(f"  Дундаж    : {tetgever['tetgever'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 3. ҮЙЛДВЭРЛЭЛ ҮЙЛЧИЛГЭЭ
#    (Crop q0723t - AgriExp q0724)             by identif
#  + (Livestock q0705 - LiveExp q0706)         by identif
#  + Enterprise ((q0809_1a*q0809_1b + q0809_2a*q0809_2b+q0809_3a*q0809_q0809_3b) - q0807_10)
# ════════════════════════════════════════════════════════════════
print("\n3. Үйлдвэрлэл үйлчилгээ...")

# Crop income
crop_g = (crop.groupby(ID)["q0723t"]
               .sum()
               .reset_index()
               .rename(columns={"q0723t": "crop_inc"}))

# Agricultural expenditure
agri_g = (agri_exp.groupby(ID)["q0724"]
                   .sum()
                   .reset_index()
                   .rename(columns={"q0724": "agri_exp"}))

# Livestock income
live_g = (livestock.groupby(ID)["q0705"]
                    .sum()
                    .reset_index()
                    .rename(columns={"q0705": "live_inc"}))

# Livestock expenditure
livee_g = (live_exp.groupby(ID)["q0706"]
                    .sum()
                    .reset_index()
                    .rename(columns={"q0706": "live_exp"}))

# Enterprise
ent = enterprise.copy()
ent["ent_inc"] = 0.0

for a, b in [("q0809_1a","q0809_1b"),
             ("q0809_2a","q0809_2b"),
             ("q0809_3a","q0809_3b")]:
    if a in ent.columns and b in ent.columns:
        ent["ent_inc"] += ent[a].fillna(0) * ent[b].fillna(0)
    else:
        print(f"  WARNING: {a} or {b} not found in enterprise!")

ent["ent_exp"] = ent["q0807_10"].fillna(0) if "q0807_10" in ent.columns else 0.0
ent["ent_net"] = ent["ent_inc"] - ent["ent_exp"]

ent_g = (ent.groupby(ID)["ent_net"]
             .sum()
             .reset_index())

# Merge and calculate
uil = (crop_g
       .merge(agri_g,  on=ID, how="outer")
       .merge(live_g,  on=ID, how="outer")
       .merge(livee_g, on=ID, how="outer")
       .merge(ent_g,   on=ID, how="outer")
       .fillna(0))

uil["uildverlelal"] = ((uil["crop_inc"] - uil["agri_exp"]) +
                       (uil["live_inc"] - uil["live_exp"]) +
                        uil["ent_net"])

uildverlelal = uil[[ID, "uildverlelal"]]

print(f"  crop_inc     дундаж: {crop_g['crop_inc'].mean():,.0f}")
print(f"  agri_exp     дундаж: {agri_g['agri_exp'].mean():,.0f}")
print(f"  live_inc     дундаж: {live_g['live_inc'].mean():,.0f}")
print(f"  live_exp     дундаж: {livee_g['live_exp'].mean():,.0f}")
print(f"  ent_net      дундаж: {ent_g['ent_net'].mean():,.0f}")
print(f"  uildverlelal дундаж: {uil['uildverlelal'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 4. БЭЛЭГ ТУСЛАМЖ ӨӨРИЙН АЖ АХУй
#    Remittance      : q0913                        sum by identif
#    Foodstuffs      : q1604                        sum by identif
#    Energy          : q1141                        sum by identif
#    Payment service : q1145                        sum by identif
#    Foodout         : q1509 × 52                   sum by identif
#    Non-Food        : q1305                        sum by identif
#    Urban Diary     : q1404*(q1401_3+q1401_4+
#                      q1402_3+q1402_4+q1403_3+q1403_4) horizontal
#                      then sum by identif × 12
#    Rural Food 7d   : (q1506+q1505)*q1504          horizontal
#                      then sum by identif × 52
# ════════════════════════════════════════════════════════════════
print("\n4. Бэлэг тусламж өөрийн аж ахуй...")

# Remittance
rem_g = (remittance.groupby(ID)["q0913"]
                    .sum()
                    .reset_index()
                    .rename(columns={"q0913": "remittance"}))



# Energy
energy_g = (energy.groupby(ID)["q1141"]
                   .sum()
                   .reset_index()
                   .rename(columns={"q1141": "energy"}))

# Payment service
pay_g = (payment.groupby(ID)["q1145"]
                 .sum()
                 .reset_index()
                 .rename(columns={"q1145": "payment"}))

# Foodout: q1509 × 52
fo = foodout.copy()
if "q1509" in fo.columns:
    fo["fo_annual"] = fo["q1509"].fillna(0) * 52
else:
    print("  WARNING: q1509 not found in foodout!")
    fo["fo_annual"] = 0

fo_g = (fo.groupby(ID)["fo_annual"]
           .sum()
           .reset_index())

# Non-Food
nf_g = (nonfood.groupby(ID)["q1305"]
                .sum()
                .reset_index()
                .rename(columns={"q1305": "nonfood"}))

# Urban Diary: q1404 × (q1401_3+q1401_4+q1402_3+q1402_4+q1403_3+q1403_4) → ×12
ud = urban_diary.copy()
qty_cols    = ["q1401_3","q1401_4","q1402_3","q1402_4","q1403_3","q1403_4"]
existing_ud = [c for c in qty_cols if c in ud.columns]

if existing_ud:
    ud["qty_sum"]   = ud[existing_ud].fillna(0).sum(axis=1)
    ud["row_value"] = ud["q1404"].fillna(0) * ud["qty_sum"]
else:
    print("  WARNING: Urban Diary qty columns not found!")
    ud["row_value"] = 0

ud_g = (ud.groupby(ID)["row_value"]
           .sum()
           .reset_index())
ud_g["ud_annual"] = ud_g["row_value"] * 12

# Rural Food 7-day: (q1506+q1505)*q1504 → ×52
rf = rural_food.copy()
for c in ["q1506","q1505","q1504"]:
    if c not in rf.columns:
        print(f"  WARNING: {c} not found in rural_food!")
        rf[c] = 0

rf["row_value"] = (rf["q1506"].fillna(0) + rf["q1505"].fillna(0)) * rf["q1504"].fillna(0)

rf_g = (rf.groupby(ID)["row_value"]
           .sum()
           .reset_index())
rf_g["rf_annual"] = rf_g["row_value"] * 52

# Merge all components
beleg_df = (rem_g
    .merge(energy_g,                on=ID, how="outer")
    .merge(pay_g,                   on=ID, how="outer")
    .merge(fo_g[[ID,"fo_annual"]],  on=ID, how="outer")
    .merge(nf_g,                    on=ID, how="outer")
    .merge(ud_g[[ID,"ud_annual"]],  on=ID, how="outer")
    .merge(rf_g[[ID,"rf_annual"]],  on=ID, how="outer")
    .fillna(0))

beleg_df["beleg_tuslamj"] = (beleg_df["remittance"]  +
                              beleg_df["energy"]      +
                              beleg_df["payment"]     +
                              beleg_df["fo_annual"]   +
                              beleg_df["nonfood"]     +
                              beleg_df["ud_annual"]   +
                              beleg_df["rf_annual"])

beleg = beleg_df[[ID, "beleg_tuslamj"]]

print(f"  remittance    дундаж: {beleg_df['remittance'].mean():,.0f}")
print(f"  energy        дундаж: {beleg_df['energy'].mean():,.0f}")
print(f"  payment       дундаж: {beleg_df['payment'].mean():,.0f}")
print(f"  fo_annual     дундаж: {beleg_df['fo_annual'].mean():,.0f}")
print(f"  nonfood       дундаж: {beleg_df['nonfood'].mean():,.0f}")
print(f"  ud_annual     дундаж: {beleg_df['ud_annual'].mean():,.0f}")
print(f"  rf_annual     дундаж: {beleg_df['rf_annual'].mean():,.0f}")
print(f"  beleg_tuslamj дундаж: {beleg_df['beleg_tuslamj'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 5. НИЙТ ОРЛОГО — merge all 4 + household composition into basicvars → data2012
# ════════════════════════════════════════════════════════════════
print("\n5. Нийт орлого нэгтгэж байна...")

data2012 = basicvars.copy()

for df_ in [tsalin, tetgever, uildverlelal, beleg, comp_g]:
    data2012 = data2012.merge(df_, on=ID, how="left")

data2012 = data2012.fillna(0)

data2012["niit_orlogo"] = (data2012["tsalin"]        +
                            data2012["tetgever"]      +
                            data2012["uildverlelal"]  +
                            data2012["beleg_tuslamj"])

# Remove rows where нийт орлого is 0
data2012 = data2012[data2012["niit_orlogo"] != 0].reset_index(drop=True)
print(f"  0 орлоготой өрх хасагдсаны дараа: {len(data2012)}")

print(f"  Нийт өрх   : {len(data2012)}")
print(f"  Багана     : {list(data2012.columns)}")

# ════════════════════════════════════════════════════════════════
# ДУНДАЖ — simple and weighted
# ════════════════════════════════════════════════════════════════
print("\nДундаж тооцож байна...")

calc_cols = ["tsalin","tetgever","uildverlelal","beleg_tuslamj","niit_orlogo"]
labels    = ["1. Цалин хөлс",
             "2. Тэтгэвэр тэтгэмж бусад",
             "3. Үйлдвэрлэл үйлчилгээ",
             "4. Бэлэг тусламж өөрийн аж ахуй",
             "5. Нийт орлого"]

# Simple (unweighted) average
simple_avg = [data2012[c].mean() for c in calc_cols]

# Weighted average using hhweight
weight_col = "hhweight"
if weight_col in data2012.columns:
    w = data2012[weight_col].fillna(0)
    weighted_avg = [
        np.average(data2012[c], weights=w) if w.sum() > 0 else np.nan
        for c in calc_cols
    ]
    monthly_wtd = [v / 12 for v in weighted_avg]
else:
    print(f"  WARNING: '{weight_col}' not found in data2012!")
    weighted_avg = [np.nan] * len(calc_cols)
    monthly_wtd  = [np.nan] * len(calc_cols)

# Results table
avg_df = pd.DataFrame({
    "Орлогын төрөл"           : labels,
    "Жингүй дундаж"           : simple_avg,
    "Жинлэсэн дундаж"         : weighted_avg,
    "Сарын дундаж (жинлэсэн)" : monthly_wtd
})

print("\n── 2012 оны дундаж үзүүлэлтүүд ──")
print(avg_df.to_string(index=False))

# Save final data
save(data2012, "data2012")
save(avg_df,   "avg2012")

Loading files...

Өрхийн бүрэлдэхүүн тооцож байна...
  Дундаж хүүхэд        : 0.97
  Дундаж бусад насанд  : 1.76
  Дундаж гишүүд        : 3.74

1. Цалин хөлс...
  Нийт өрх  : 12811
  Дундаж    : 3,671,716

2. Тэтгэвэр тэтгэмж бусад...
  Нийт өрх  : 12811
  Дундаж    : 2,295,117

3. Үйлдвэрлэл үйлчилгээ...
  crop_inc     дундаж: 1,041,957
  agri_exp     дундаж: 549,453
  live_inc     дундаж: 1,285,183
  live_exp     дундаж: 546,100
  ent_net      дундаж: 12,912,800
  uildverlelal дундаж: 4,852,286

4. Бэлэг тусламж өөрийн аж ахуй...
  remittance    дундаж: 247,325
  energy        дундаж: 14,108
  payment       дундаж: 2,138
  fo_annual     дундаж: 126,193
  nonfood       дундаж: 19,695
  ud_annual     дундаж: 0
  rf_annual     дундаж: 3,075
  beleg_tuslamj дундаж: 412,533

5. Нийт орлого нэгтгэж байна...
  0 орлоготой өрх хасагдсаны дараа: 12809
  Нийт өрх   : 12809
  Багана     : ['identif', 'cluster', 'newaimag', 'location', 'urban', 'region', 'month', 'quarter', 'strata', 'hhweight',

Exception ignored in: <function ZipFile.__del__ at 0x0000022A07E1BBA0>
Traceback (most recent call last):
  File "c:\Users\b22fa\AppData\Local\Programs\Python\Python313\Lib\zipfile\__init__.py", line 1975, in __del__
    self.close()
  File "c:\Users\b22fa\AppData\Local\Programs\Python\Python313\Lib\zipfile\__init__.py", line 1992, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


  Saved Excel: data2012.xlsx
  Saved Excel: avg2012.xlsx


In [58]:
import os
import pandas as pd
import numpy as np

FOLDER = r"C:\Users\b22fa\Desktop\diplom unelgee\2014"
ID = "identif"

def load(filename):
    path = os.path.join(FOLDER, filename + ".dta")
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.lower()
    return df

def save(df, name):
    size = df.memory_usage(deep=True).sum() / (1024*1024)
    out = os.path.join(FOLDER, name)
    if size > 30:
        df.to_csv(out + ".csv", index=False, encoding="utf-8-sig")
        print(f"  Saved CSV: {name}.csv")
    else:
        df.to_excel(out + ".xlsx", index=False)
        print(f"  Saved Excel: {name}.xlsx")

# ── Load all files ────────────────────────────────────────────────
print("Loading files...")
indiv       = load("02_indiv (6)")
other_inc   = load("09_other_income (5)")
crop        = load("06_crop (5)")
agri_exp    = load("07_agric_exp (5)")
livestock   = load("03_livestock (5)")
live_exp    = load("04_livestock_exp (5)")
enterprise  = load("08_enterprise (5)")
remittance  = load("10_remittance (4)")
energy      = load("12_energy (1)")
payment     = load("13_payment_serv (1)")
foodout     = load("19_foodout (1)")
nonfood     = load("15_non_food (1)")
urban_diary = load("16_urb_diary (1)")
rural_food  = load("17_rur_food_7d (1)")
basicvars   = load("basicvars (5)")

# ════════════════════════════════════════════════════════════════
# HOUSEHOLD COMPOSITION from indiv file
# Children     : q0105y < 14
# Leading adult: 1 if any member q0105y >= 14
# Other adults : count(q0105y >= 14) - 1
# Total members: all rows per identif
# ════════════════════════════════════════════════════════════════
print("\nӨрхийн бүрэлдэхүүн тооцож байна...")

hh_comp = indiv.copy()

age_col = "q0105y"
if age_col not in hh_comp.columns:
    print(f"  WARNING: {age_col} not found in indiv!")
    hh_comp[age_col] = np.nan

hh_comp["is_child"]  = (hh_comp[age_col].fillna(0) < 14).astype(int)
hh_comp["is_adult"]  = (hh_comp[age_col].fillna(0) >= 14).astype(int)
hh_comp["is_member"] = 1

comp_g = hh_comp.groupby(ID).agg(
    children      = ("is_child",  "sum"),
    total_adults  = ("is_adult",  "sum"),
    total_members = ("is_member", "sum")
).reset_index()

comp_g["leading_adult"] = (comp_g["total_adults"] >= 1).astype(int)
comp_g["other_adults"]  = (comp_g["total_adults"] - comp_g["leading_adult"]).clip(lower=0)
comp_g = comp_g[[ID, "children", "leading_adult", "other_adults", "total_members"]]

print(f"  Дундаж хүүхэд       : {comp_g['children'].mean():.2f}")
print(f"  Дундаж бусад насанд : {comp_g['other_adults'].mean():.2f}")
print(f"  Дундаж гишүүд       : {comp_g['total_members'].mean():.2f}")

# ════════════════════════════════════════════════════════════════
# 1. ЦАЛИН ХӨЛС
#    File: 02_indiv → q0417_b + q0417_c + q0419_b + q0419_c (horizontal)
#    then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n1. Цалин хөлс...")

wage = indiv.copy()
w_cols     = ["q0417b", "q0417c", "q0419b", "q0419c"]
existing_w = [c for c in w_cols if c in wage.columns]

if existing_w:
    wage["row_sum"] = wage[existing_w].fillna(0).sum(axis=1)
else:
    print("  WARNING: wage columns not found!")
    wage["row_sum"] = 0

tsalin = (wage.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tsalin"}))

print(f"  Нийт өрх  : {len(tsalin)}")
print(f"  Дундаж    : {tsalin['tsalin'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 2. ТЭТГЭВЭР ТЭТГЭМЖ БУСАД
#    File: 09_other_income → q0702b + q0703b + q0704b + q0705b + q0706b (horizontal)
#    then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n2. Тэтгэвэр тэтгэмж бусад...")

oi = other_inc.copy()
t_cols     = ["q0702b", "q0703b", "q0704b", "q0705b", "q0706b"]
existing_t = [c for c in t_cols if c in oi.columns]

if existing_t:
    oi["row_sum"] = oi[existing_t].fillna(0).sum(axis=1)
else:
    print("  WARNING: other_income columns not found!")
    oi["row_sum"] = 0

tetgever = (oi.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tetgever"}))

print(f"  Нийт өрх  : {len(tetgever)}")
print(f"  Дундаж    : {tetgever['tetgever'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 3. ҮЙЛДВЭРЛЭЛ ҮЙЛЧИЛГЭЭ
#    (Crop q0523b - AgriExp q0524)           by identif
#  + (Livestock q0505 - LivestockExp q0506)  by identif
#  + Enterprise (q0610 - q0607_99)           by identif
# ════════════════════════════════════════════════════════════════
print("\n3. Үйлдвэрлэл үйлчилгээ...")

# Crop income
if "q0523b" in crop.columns:
    crop_g = (crop.groupby(ID)["q0523b"]
                   .sum()
                   .reset_index()
                   .rename(columns={"q0523b": "crop_inc"}))
else:
    print("  WARNING: q0523b not found in crop!")
    crop_g = crop[[ID]].drop_duplicates()
    crop_g["crop_inc"] = 0

# Agricultural expenditure
if "q0524" in agri_exp.columns:
    agri_g = (agri_exp.groupby(ID)["q0524"]
                       .sum()
                       .reset_index()
                       .rename(columns={"q0524": "agri_exp"}))
else:
    print("  WARNING: q0524 not found in agri_exp!")
    agri_g = agri_exp[[ID]].drop_duplicates()
    agri_g["agri_exp"] = 0

# Livestock income
if "q0505" in livestock.columns:
    live_g = (livestock.groupby(ID)["q0505"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0505": "live_inc"}))
else:
    print("  WARNING: q0505 not found in livestock!")
    live_g = livestock[[ID]].drop_duplicates()
    live_g["live_inc"] = 0

# Livestock expenditure
if "q0506" in live_exp.columns:
    livee_g = (live_exp.groupby(ID)["q0506"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0506": "live_exp"}))
else:
    print("  WARNING: q0506 not found in live_exp!")
    livee_g = live_exp[[ID]].drop_duplicates()
    livee_g["live_exp"] = 0

# Enterprise: q0610 - q0607_99
ent = enterprise.copy()
ent["ent_inc"] = ent["q0610"].fillna(0)    if "q0610"    in ent.columns else 0.0
ent["ent_exp"] = ent["q0607_99"].fillna(0) if "q0607_99" in ent.columns else 0.0
if "q0610"    not in ent.columns: print("  WARNING: q0610 not found in enterprise!")
if "q0607_99" not in ent.columns: print("  WARNING: q0607_99 not found in enterprise!")
ent["ent_net"] = ent["ent_inc"] - ent["ent_exp"]

ent_g = (ent.groupby(ID)["ent_net"]
             .sum()
             .reset_index())

# Merge and calculate
uil = (crop_g
       .merge(agri_g,  on=ID, how="outer")
       .merge(live_g,  on=ID, how="outer")
       .merge(livee_g, on=ID, how="outer")
       .merge(ent_g,   on=ID, how="outer")
       .fillna(0))

uil["uildverlelal"] = ((uil["crop_inc"] - uil["agri_exp"]) +
                       (uil["live_inc"] - uil["live_exp"]) +
                        uil["ent_net"])

uildverlelal = uil[[ID, "uildverlelal"]]

print(f"  crop_inc     дундаж: {crop_g['crop_inc'].mean():,.0f}")
print(f"  agri_exp     дундаж: {agri_g['agri_exp'].mean():,.0f}")
print(f"  live_inc     дундаж: {live_g['live_inc'].mean():,.0f}")
print(f"  live_exp     дундаж: {livee_g['live_exp'].mean():,.0f}")
print(f"  ent_net      дундаж: {ent_g['ent_net'].mean():,.0f}")
print(f"  uildverlelal дундаж: {uil['uildverlelal'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 4. БЭЛЭГ ТУСЛАМЖ ӨӨРИЙН АЖ АХУй
#    Remittance      : q0712                            sum by identif
#    Energy          : q0941 + q0942                    sum by identif
#    Payment service : q0946 + q0947                    sum by identif
#    Foodout         : q1309 × 52                       sum by identif
#    Non-Food        : q1104 + q1105                    sum by identif
#    Urban Diary     : q1404*(q1201_3+q1201_4+q1202_3+
#                      q1202_4+q1203_3+q1203_4) horiz   sum by identif × 12
#    Rural Food 7d   : (q1305+q1306)*q1304 horiz        sum by identif × 52
# ════════════════════════════════════════════════════════════════
print("\n4. Бэлэг тусламж өөрийн аж ахуй...")

# Remittance
if "q0712" in remittance.columns:
    rem_g = (remittance.groupby(ID)["q0712"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0712": "remittance"}))
else:
    print("  WARNING: q0712 not found in remittance!")
    rem_g = remittance[[ID]].drop_duplicates()
    rem_g["remittance"] = 0

# Energy: q0941 + q0942
en = energy.copy()
en_cols     = ["q0941", "q0942"]
existing_en = [c for c in en_cols if c in en.columns]
if existing_en:
    en["en_sum"] = en[existing_en].fillna(0).sum(axis=1)
else:
    print("  WARNING: energy columns not found!")
    en["en_sum"] = 0

energy_g = (en.groupby(ID)["en_sum"]
               .sum()
               .reset_index()
               .rename(columns={"en_sum": "energy"}))

# Payment service: q0946 + q0947
pv = payment.copy()
pv_cols     = ["q0946", "q0947"]
existing_pv = [c for c in pv_cols if c in pv.columns]
if existing_pv:
    pv["pv_sum"] = pv[existing_pv].fillna(0).sum(axis=1)
else:
    print("  WARNING: payment columns not found!")
    pv["pv_sum"] = 0

pay_g = (pv.groupby(ID)["pv_sum"]
            .sum()
            .reset_index()
            .rename(columns={"pv_sum": "payment"}))

# Foodout: q1309 × 52
fo = foodout.copy()
if "q1309" in fo.columns:
    fo["fo_annual"] = fo["q1309"].fillna(0) * 52
else:
    print("  WARNING: q1309 not found in foodout!")
    fo["fo_annual"] = 0

fo_g = (fo.groupby(ID)["fo_annual"]
           .sum()
           .reset_index())

# Non-Food: q1104 + q1105
nf = nonfood.copy()
nf_cols     = ["q1104", "q1105"]
existing_nf = [c for c in nf_cols if c in nf.columns]
if existing_nf:
    nf["nf_sum"] = nf[existing_nf].fillna(0).sum(axis=1)
else:
    print("  WARNING: nonfood columns not found!")
    nf["nf_sum"] = 0

nf_g = (nf.groupby(ID)["nf_sum"]
           .sum()
           .reset_index()
           .rename(columns={"nf_sum": "nonfood"}))

# Urban Diary: q1404 × (q1201_3+q1201_4+q1202_3+q1202_4+q1203_3+q1203_4) → × 12
ud = urban_diary.copy()
qty_cols    = ["q1201_3","q1201_4","q1202_3","q1202_4","q1203_3","q1203_4"]
existing_ud = [c for c in qty_cols if c in ud.columns]

if existing_ud and "q1204" in ud.columns:
    ud["qty_sum"]   = ud[existing_ud].fillna(0).sum(axis=1)
    ud["row_value"] = ud["q1204"].fillna(0) * ud["qty_sum"]
else:
    print("  WARNING: Urban Diary columns not found!")
    ud["row_value"] = 0

ud_g = (ud.groupby(ID)["row_value"]
           .sum()
           .reset_index())
ud_g["ud_annual"] = ud_g["row_value"] * 12

# Rural Food 7-day: (q1305+q1306)*q1304 → × 52
rf = rural_food.copy()
for c in ["q1305","q1306","q1304"]:
    if c not in rf.columns:
        print(f"  WARNING: {c} not found in rural_food!")
        rf[c] = 0

rf["row_value"] = (rf["q1305"].fillna(0) + rf["q1306"].fillna(0)) * rf["q1304"].fillna(0)

rf_g = (rf.groupby(ID)["row_value"]
           .sum()
           .reset_index())
rf_g["rf_annual"] = rf_g["row_value"] * 52

# Merge all components
beleg_df = (rem_g
    .merge(energy_g,               on=ID, how="outer")
    .merge(pay_g,                  on=ID, how="outer")
    .merge(fo_g[[ID,"fo_annual"]], on=ID, how="outer")
    .merge(nf_g,                   on=ID, how="outer")
    .merge(ud_g[[ID,"ud_annual"]], on=ID, how="outer")
    .merge(rf_g[[ID,"rf_annual"]], on=ID, how="outer")
    .fillna(0))

beleg_df["beleg_tuslamj"] = (beleg_df["remittance"] +
                              beleg_df["energy"]     +
                              beleg_df["payment"]    +
                              beleg_df["fo_annual"]  +
                              beleg_df["nonfood"]    +
                              beleg_df["ud_annual"]  +
                              beleg_df["rf_annual"])

beleg = beleg_df[[ID, "beleg_tuslamj"]]

print(f"  remittance    дундаж: {beleg_df['remittance'].mean():,.0f}")
print(f"  energy        дундаж: {beleg_df['energy'].mean():,.0f}")
print(f"  payment       дундаж: {beleg_df['payment'].mean():,.0f}")
print(f"  fo_annual     дундаж: {beleg_df['fo_annual'].mean():,.0f}")
print(f"  nonfood       дундаж: {beleg_df['nonfood'].mean():,.0f}")
print(f"  ud_annual     дундаж: {beleg_df['ud_annual'].mean():,.0f}")
print(f"  rf_annual     дундаж: {beleg_df['rf_annual'].mean():,.0f}")
print(f"  beleg_tuslamj дундаж: {beleg_df['beleg_tuslamj'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 5. НИЙТ ОРЛОГО — merge all 4 + household composition into basicvars → data2014
# ════════════════════════════════════════════════════════════════
print("\n5. Нийт орлого нэгтгэж байна...")

data2014 = basicvars.copy()

for df_ in [tsalin, tetgever, uildverlelal, beleg, comp_g]:
    data2014 = data2014.merge(df_, on=ID, how="left")

data2014 = data2014.fillna(0)

data2014["niit_orlogo"] = (data2014["tsalin"]       +
                            data2014["tetgever"]     +
                            data2014["uildverlelal"] +
                            data2014["beleg_tuslamj"])

# Remove rows where нийт орлого is 0
data2014 = data2014[data2014["niit_orlogo"] != 0].reset_index(drop=True)
print(f"  0 орлоготой өрх хасагдсаны дараа: {len(data2014)}")

print(f"  Нийт өрх   : {len(data2014)}")
print(f"  Багана     : {list(data2014.columns)}")

# ════════════════════════════════════════════════════════════════
# ДУНДАЖ — simple and weighted
# ════════════════════════════════════════════════════════════════
print("\nДундаж тооцож байна...")

calc_cols = ["tsalin","tetgever","uildverlelal","beleg_tuslamj","niit_orlogo"]
labels    = ["1. Цалин хөлс",
             "2. Тэтгэвэр тэтгэмж бусад",
             "3. Үйлдвэрлэл үйлчилгээ",
             "4. Бэлэг тусламж өөрийн аж ахуй",
             "5. Нийт орлого"]

# Simple (unweighted) average
simple_avg = [data2014[c].mean() for c in calc_cols]

# Weighted average using hhweight
weight_col = "hhweight"
if weight_col in data2014.columns:
    w = data2014[weight_col].fillna(0)
    weighted_avg = [
        np.average(data2014[c], weights=w) if w.sum() > 0 else np.nan
        for c in calc_cols
    ]
    monthly_wtd = [v / 12 for v in weighted_avg]
else:
    print(f"  WARNING: '{weight_col}' not found in data2014!")
    weighted_avg = [np.nan] * len(calc_cols)
    monthly_wtd  = [np.nan] * len(calc_cols)

# Results table
avg_df = pd.DataFrame({
    "Орлогын төрөл"           : labels,
    "Жингүй дундаж"           : simple_avg,
    "Жинлэсэн дундаж"         : weighted_avg,
    "Сарын дундаж (жинлэсэн)" : monthly_wtd
})

print("\n── 2014 оны дундаж үзүүлэлтүүд ──")
print(avg_df.to_string(index=False))

# Save final data
save(data2014, "data2014")
save(avg_df,   "avg2014")

Loading files...

Өрхийн бүрэлдэхүүн тооцож байна...
  Дундаж хүүхэд       : 0.99
  Дундаж бусад насанд : 1.65
  Дундаж гишүүд       : 3.64

1. Цалин хөлс...
  Нийт өрх  : 16174
  Дундаж    : 4,753,085

2. Тэтгэвэр тэтгэмж бусад...
  Нийт өрх  : 16174
  Дундаж    : 2,392,726

3. Үйлдвэрлэл үйлчилгээ...
  crop_inc     дундаж: 930,014
  agri_exp     дундаж: 528,187
  live_inc     дундаж: 2,483,803
  live_exp     дундаж: 796,003
  ent_net      дундаж: 7,833,296
  uildverlelal дундаж: 3,617,560

4. Бэлэг тусламж өөрийн аж ахуй...
  remittance    дундаж: 431,530
  energy        дундаж: 90,037
  payment       дундаж: 2,724
  fo_annual     дундаж: 27,148
  nonfood       дундаж: 232,865
  ud_annual     дундаж: 55,848
  rf_annual     дундаж: 2,743
  beleg_tuslamj дундаж: 842,894

5. Нийт орлого нэгтгэж байна...
  0 орлоготой өрх хасагдсаны дараа: 16172
  Нийт өрх   : 16172
  Багана     : ['identif', 'cluster', 'newaimag', 'bag', 'urban', 'region', 'month', 'quarter', 'location', 'strata', 'hhwe

In [63]:
import os
import pandas as pd
import numpy as np

FOLDER = r"C:\Users\b22fa\Desktop\diplom unelgee\2016"
ID = "identif"

def load(filename):
    path = os.path.join(FOLDER, filename + ".dta")
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.lower()
    return df

def save(df, name):
    size = df.memory_usage(deep=True).sum() / (1024*1024)
    out = os.path.join(FOLDER, name)
    if size > 30:
        df.to_csv(out + ".csv", index=False, encoding="utf-8-sig")
        print(f"  Saved CSV: {name}.csv")
    else:
        df.to_excel(out + ".xlsx", index=False)
        print(f"  Saved Excel: {name}.xlsx")

# ── Load all files ────────────────────────────────────────────────
print("Loading files...")
indiv       = load("02_indiv (8)")
other_inc   = load("09_other_income (7)")
crop        = load("06_crop (7)")
agri_exp    = load("07_agric_exp (7)")
livestock   = load("03_livestock (7)")
live_exp    = load("04_livestock_exp (7)")
enterprise  = load("08_enterprise (7)")
remittance  = load("10_remittance (5)")
energy      = load("12_energy (2)")
payment     = load("13_payment_serv (2)")
foodout     = load("19_foodout (2)")
nonfood     = load("15_non_food (2)")
urban_diary = load("16_urb_diary (2)")
rural_food  = load("17_rur_food_7d (2)")
basicvars   = load("basicvars (7)")

# ════════════════════════════════════════════════════════════════
# HOUSEHOLD COMPOSITION from indiv file
# Children     : q0105y < 14
# Leading adult: 1 if any member q0105y >= 14
# Other adults : count(q0105y >= 14) - 1
# Total members: all rows per identif
# ════════════════════════════════════════════════════════════════
print("\nӨрхийн бүрэлдэхүүн тооцож байна...")

hh_comp = indiv.copy()

age_col = "q0105y"
if age_col not in hh_comp.columns:
    print(f"  WARNING: {age_col} not found in indiv!")
    hh_comp[age_col] = np.nan

hh_comp["is_child"]  = (hh_comp[age_col].fillna(0) < 14).astype(int)
hh_comp["is_adult"]  = (hh_comp[age_col].fillna(0) >= 14).astype(int)
hh_comp["is_member"] = 1

comp_g = hh_comp.groupby(ID).agg(
    children      = ("is_child",  "sum"),
    total_adults  = ("is_adult",  "sum"),
    total_members = ("is_member", "sum")
).reset_index()

comp_g["leading_adult"] = (comp_g["total_adults"] >= 1).astype(int)
comp_g["other_adults"]  = (comp_g["total_adults"] - comp_g["leading_adult"]).clip(lower=0)
comp_g = comp_g[[ID, "children", "leading_adult", "other_adults", "total_members"]]

print(f"  Дундаж хүүхэд       : {comp_g['children'].mean():.2f}")
print(f"  Дундаж бусад насанд : {comp_g['other_adults'].mean():.2f}")
print(f"  Дундаж гишүүд       : {comp_g['total_members'].mean():.2f}")

# ════════════════════════════════════════════════════════════════
# 1. ЦАЛИН ХӨЛС
#    File: 02_indiv → q0417b + q0417c + q0419b + q0419c (horizontal)
#    then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n1. Цалин хөлс...")

wage = indiv.copy()
w_cols     = ["q0417b", "q0417c", "q0419b", "q0419c"]
existing_w = [c for c in w_cols if c in wage.columns]

if existing_w:
    wage["row_sum"] = wage[existing_w].fillna(0).sum(axis=1)
else:
    print("  WARNING: wage columns not found!")
    wage["row_sum"] = 0

tsalin = (wage.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tsalin"}))

print(f"  Нийт өрх  : {len(tsalin)}")
print(f"  Дундаж    : {tsalin['tsalin'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 2. ТЭТГЭВЭР ТЭТГЭМЖ БУСАД
#    File: 09_other_income → q0702b + q0703b + q0704b + q0705b + q0706b (horizontal)
#    then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n2. Тэтгэвэр тэтгэмж бусад...")

oi = other_inc.copy()
t_cols     = ["q0702b", "q0703b", "q0704b", "q0705b", "q0706b"]
existing_t = [c for c in t_cols if c in oi.columns]

if existing_t:
    oi["row_sum"] = oi[existing_t].fillna(0).sum(axis=1)
else:
    print("  WARNING: other_income columns not found!")
    oi["row_sum"] = 0

tetgever = (oi.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tetgever"}))

print(f"  Нийт өрх  : {len(tetgever)}")
print(f"  Дундаж    : {tetgever['tetgever'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 3. ҮЙЛДВЭРЛЭЛ ҮЙЛЧИЛГЭЭ
#    (Crop q0523b - AgriExp q0524)           by identif
#  + (Livestock q0505 - LivestockExp q0506)  by identif
#  + Enterprise (q0610 - q0607_99)           by identif
# ════════════════════════════════════════════════════════════════
print("\n3. Үйлдвэрлэл үйлчилгээ...")

# Crop income
if "q0523b" in crop.columns:
    crop_g = (crop.groupby(ID)["q0523b"]
                   .sum()
                   .reset_index()
                   .rename(columns={"q0523b": "crop_inc"}))
else:
    print("  WARNING: q0523b not found in crop!")
    crop_g = crop[[ID]].drop_duplicates()
    crop_g["crop_inc"] = 0

# Agricultural expenditure
if "q0524" in agri_exp.columns:
    agri_g = (agri_exp.groupby(ID)["q0524"]
                       .sum()
                       .reset_index()
                       .rename(columns={"q0524": "agri_exp"}))
else:
    print("  WARNING: q0524 not found in agri_exp!")
    agri_g = agri_exp[[ID]].drop_duplicates()
    agri_g["agri_exp"] = 0

# Livestock income
if "q0505" in livestock.columns:
    live_g = (livestock.groupby(ID)["q0505"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0505": "live_inc"}))
else:
    print("  WARNING: q0505 not found in livestock!")
    live_g = livestock[[ID]].drop_duplicates()
    live_g["live_inc"] = 0

# Livestock expenditure
if "q0506" in live_exp.columns:
    livee_g = (live_exp.groupby(ID)["q0506"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0506": "live_exp"}))
else:
    print("  WARNING: q0506 not found in live_exp!")
    livee_g = live_exp[[ID]].drop_duplicates()
    livee_g["live_exp"] = 0

# Enterprise: q0610 - q0607_99
ent = enterprise.copy()
ent["ent_inc"] = ent["q0610"].fillna(0)    if "q0610"    in ent.columns else 0.0
ent["ent_exp"] = ent["q0607_99"].fillna(0) if "q0607_99" in ent.columns else 0.0
if "q0610"    not in ent.columns: print("  WARNING: q0610 not found in enterprise!")
if "q0607_99" not in ent.columns: print("  WARNING: q0607_99 not found in enterprise!")
ent["ent_net"] = ent["ent_inc"] - ent["ent_exp"]

ent_g = (ent.groupby(ID)["ent_net"]
             .sum()
             .reset_index())

# Merge and calculate
uil = (crop_g
       .merge(agri_g,  on=ID, how="outer")
       .merge(live_g,  on=ID, how="outer")
       .merge(livee_g, on=ID, how="outer")
       .merge(ent_g,   on=ID, how="outer")
       .fillna(0))

uil["uildverlelal"] = ((uil["crop_inc"] - uil["agri_exp"]) +
                       (uil["live_inc"] - uil["live_exp"]) +
                        uil["ent_net"])

uildverlelal = uil[[ID, "uildverlelal"]]

print(f"  crop_inc     дундаж: {crop_g['crop_inc'].mean():,.0f}")
print(f"  agri_exp     дундаж: {agri_g['agri_exp'].mean():,.0f}")
print(f"  live_inc     дундаж: {live_g['live_inc'].mean():,.0f}")
print(f"  live_exp     дундаж: {livee_g['live_exp'].mean():,.0f}")
print(f"  ent_net      дундаж: {ent_g['ent_net'].mean():,.0f}")
print(f"  uildverlelal дундаж: {uil['uildverlelal'].mean():,.0f}")
# ════════════════════════════════════════════════════════════════


# ════════════════════════════════════════════════════════════════
# 4. БЭЛЭГ ТУСЛАМЖ ӨӨРИЙН АЖ АХУй
#    Remittance      : q0712                             sum by identif
#    Energy          : q0941 + q0942                     sum by identif
#    Payment service : q0946 + q0947                     sum by identif
#    Foodout         : q1309 × 52                        sum by identif
#    Non-Food        : q1104 + q1105                     sum by identif
#    Urban Diary     : q1204*(q1201_3+q1201_4+q1202_3+
#                      q1202_4+q1203_3+q1203_4) horiz    sum by identif × 12
#    Rural Food 7d   : (q1305+q1306)*q1304 horiz         sum by identif × 52
# ════════════════════════════════════════════════════════════════
print("\n4. Бэлэг тусламж өөрийн аж ахуй...")

# Remittance
if "q0712" in remittance.columns:
    rem_g = (remittance.groupby(ID)["q0712"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0712": "remittance"}))
else:
    print("  WARNING: q0712 not found in remittance!")
    rem_g = remittance[[ID]].drop_duplicates()
    rem_g["remittance"] = 0

# Energy: q0941 + q0942
en = energy.copy()
en_cols     = ["q0941", "q0942"]
existing_en = [c for c in en_cols if c in en.columns]
if existing_en:
    en["en_sum"] = en[existing_en].fillna(0).sum(axis=1)
else:
    print("  WARNING: energy columns not found!")
    en["en_sum"] = 0

energy_g = (en.groupby(ID)["en_sum"]
               .sum()
               .reset_index()
               .rename(columns={"en_sum": "energy"}))

# Payment service: q0946 + q0947
pv = payment.copy()
pv_cols     = ["q0946", "q0947"]
existing_pv = [c for c in pv_cols if c in pv.columns]
if existing_pv:
    pv["pv_sum"] = pv[existing_pv].fillna(0).sum(axis=1)
else:
    print("  WARNING: payment columns not found!")
    pv["pv_sum"] = 0

pay_g = (pv.groupby(ID)["pv_sum"]
            .sum()
            .reset_index()
            .rename(columns={"pv_sum": "payment"}))

# Foodout: q1309 × 52
fo = foodout.copy()
if "q1309" in fo.columns:
    fo["fo_annual"] = fo["q1309"].fillna(0) * 52
else:
    print("  WARNING: q1309 not found in foodout!")
    fo["fo_annual"] = 0

fo_g = (fo.groupby(ID)["fo_annual"]
           .sum()
           .reset_index())

# Non-Food: q1104 + q1105
nf = nonfood.copy()
nf_cols     = ["q1104", "q1105"]
existing_nf = [c for c in nf_cols if c in nf.columns]
if existing_nf:
    nf["nf_sum"] = nf[existing_nf].fillna(0).sum(axis=1)
else:
    print("  WARNING: nonfood columns not found!")
    nf["nf_sum"] = 0

nf_g = (nf.groupby(ID)["nf_sum"]
           .sum()
           .reset_index()
           .rename(columns={"nf_sum": "nonfood"}))

# Urban Diary: q1204 × (q1201_3+q1201_4+q1202_3+q1202_4+q1203_3+q1203_4) → × 12
ud = urban_diary.copy()
qty_cols    = ["q1201_3","q1201_4","q1202_3","q1202_4","q1203_3","q1203_4"]
existing_ud = [c for c in qty_cols if c in ud.columns]

if existing_ud and "q1204" in ud.columns:
    ud["qty_sum"]   = ud[existing_ud].fillna(0).sum(axis=1)
    ud["row_value"] = ud["q1204"].fillna(0) * ud["qty_sum"]
else:
    print("  WARNING: Urban Diary columns not found!")
    ud["row_value"] = 0

ud_g = (ud.groupby(ID)["row_value"]
           .sum()
           .reset_index())
ud_g["ud_annual"] = ud_g["row_value"] * 12

# Rural Food 7-day: (q1305+q1306)*q1304 → × 52
rf = rural_food.copy()
for c in ["q1305","q1306","q1304"]:
    if c not in rf.columns:
        print(f"  WARNING: {c} not found in rural_food!")
        rf[c] = 0

rf["row_value"] = (rf["q1305"].fillna(0) + rf["q1306"].fillna(0)) * rf["q1304"].fillna(0)

rf_g = (rf.groupby(ID)["row_value"]
           .sum()
           .reset_index())
rf_g["rf_annual"] = rf_g["row_value"] * 52

# Merge all components
beleg_df = (rem_g
    .merge(energy_g,               on=ID, how="outer")
    .merge(pay_g,                  on=ID, how="outer")
    .merge(fo_g[[ID,"fo_annual"]], on=ID, how="outer")
    .merge(nf_g,                   on=ID, how="outer")
    .merge(ud_g[[ID,"ud_annual"]], on=ID, how="outer")
    .merge(rf_g[[ID,"rf_annual"]], on=ID, how="outer")
    .fillna(0))

beleg_df["beleg_tuslamj"] = (beleg_df["remittance"] +
                              beleg_df["energy"]     +
                              beleg_df["payment"]    +
                              beleg_df["fo_annual"]  +
                              beleg_df["nonfood"]    +
                              beleg_df["ud_annual"]  +
                              beleg_df["rf_annual"])

beleg = beleg_df[[ID, "beleg_tuslamj"]]

print(f"  remittance    дундаж: {beleg_df['remittance'].mean():,.0f}")
print(f"  energy        дундаж: {beleg_df['energy'].mean():,.0f}")
print(f"  payment       дундаж: {beleg_df['payment'].mean():,.0f}")
print(f"  fo_annual     дундаж: {beleg_df['fo_annual'].mean():,.0f}")
print(f"  nonfood       дундаж: {beleg_df['nonfood'].mean():,.0f}")
print(f"  ud_annual     дундаж: {beleg_df['ud_annual'].mean():,.0f}")
print(f"  rf_annual     дундаж: {beleg_df['rf_annual'].mean():,.0f}")
print(f"  beleg_tuslamj дундаж: {beleg_df['beleg_tuslamj'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 5. НИЙТ ОРЛОГО — merge all 4 + household composition into basicvars → data2016
# ════════════════════════════════════════════════════════════════
print("\n5. Нийт орлого нэгтгэж байна...")

data2016 = basicvars.copy()

for df_ in [tsalin, tetgever, uildverlelal, beleg, comp_g]:
    data2016 = data2016.merge(df_, on=ID, how="left")

data2016 = data2016.fillna(0)

data2016["niit_orlogo"] = (data2016["tsalin"]       +
                            data2016["tetgever"]     +
                            data2016["uildverlelal"] +
                            data2016["beleg_tuslamj"])

# Remove rows where нийт орлого is 0
data2016 = data2016[data2016["niit_orlogo"] != 0].reset_index(drop=True)
print(f"  0 орлоготой өрх хасагдсаны дараа: {len(data2016)}")

print(f"  Нийт өрх   : {len(data2016)}")
print(f"  Багана     : {list(data2016.columns)}")

# ════════════════════════════════════════════════════════════════
# ДУНДАЖ — simple and weighted
# ════════════════════════════════════════════════════════════════
print("\nДундаж тооцож байна...")

calc_cols = ["tsalin","tetgever","uildverlelal","beleg_tuslamj","niit_orlogo"]
labels    = ["1. Цалин хөлс",
             "2. Тэтгэвэр тэтгэмж бусад",
             "3. Үйлдвэрлэл үйлчилгээ",
             "4. Бэлэг тусламж өөрийн аж ахуй",
             "5. Нийт орлого"]

# Simple (unweighted) average
simple_avg = [data2016[c].mean() for c in calc_cols]

# Weighted average using hhweight
weight_col = "hhweight"
if weight_col in data2016.columns:
    w = data2016[weight_col].fillna(0)
    weighted_avg = [
        np.average(data2016[c], weights=w) if w.sum() > 0 else np.nan
        for c in calc_cols
    ]
    monthly_wtd = [v / 12 for v in weighted_avg]
else:
    print(f"  WARNING: '{weight_col}' not found in data2016!")
    weighted_avg = [np.nan] * len(calc_cols)
    monthly_wtd  = [np.nan] * len(calc_cols)

# Results table
avg_df = pd.DataFrame({
    "Орлогын төрөл"           : labels,
    "Жингүй дундаж"           : simple_avg,
    "Жинлэсэн дундаж"         : weighted_avg,
    "Сарын дундаж (жинлэсэн)" : monthly_wtd
})

print("\n── 2016 оны дундаж үзүүлэлтүүд ──")
print(avg_df.to_string(index=False))

# Save final data
save(data2016, "data2016")
save(avg_df,   "avg2016")

Loading files...

Өрхийн бүрэлдэхүүн тооцож байна...
  Дундаж хүүхэд       : 1.02
  Дундаж бусад насанд : 1.54
  Дундаж гишүүд       : 3.56

1. Цалин хөлс...
  Нийт өрх  : 16451
  Дундаж    : 4,697,483

2. Тэтгэвэр тэтгэмж бусад...
  Нийт өрх  : 16451
  Дундаж    : 2,484,363

3. Үйлдвэрлэл үйлчилгээ...
  crop_inc     дундаж: 1,251,426
  agri_exp     дундаж: 630,552
  live_inc     дундаж: 1,825,146
  live_exp     дундаж: 980,961
  ent_net      дундаж: 6,281,482
  uildverlelal дундаж: 2,488,679

4. Бэлэг тусламж өөрийн аж ахуй...
  remittance    дундаж: 349,622
  energy        дундаж: 85,558
  payment       дундаж: 2,494
  fo_annual     дундаж: 79,502
  nonfood       дундаж: 201,461
  ud_annual     дундаж: 51,379
  rf_annual     дундаж: 2,017
  beleg_tuslamj дундаж: 772,034

5. Нийт орлого нэгтгэж байна...
  0 орлоготой өрх хасагдсаны дараа: 16446
  Нийт өрх   : 16446
  Багана     : ['identif', 'cluster', 'newaimag', 'bag', 'location', 'urban', 'region', 'month', 'quarter', 'strata', 'hh

In [66]:
import os
import pandas as pd
import numpy as np

FOLDER = r"C:\Users\b22fa\Desktop\diplom unelgee\2018"
ID = "identif"

def load(filename):
    path = os.path.join(FOLDER, filename + ".dta")
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.lower()
    return df

def save(df, name):
    size = df.memory_usage(deep=True).sum() / (1024*1024)
    out = os.path.join(FOLDER, name)
    if size > 30:
        df.to_csv(out + ".csv", index=False, encoding="utf-8-sig")
        print(f"  Saved CSV: {name}.csv")
    else:
        df.to_excel(out + ".xlsx", index=False)
        print(f"  Saved Excel: {name}.xlsx")

# ── Load all files ────────────────────────────────────────────────
print("Loading files...")
indiv       = load("02_indiv (10)")
other_inc   = load("09_other_income (10)")
crop        = load("06_crop (9)")
agri_exp    = load("07_agric_exp (9)")
livestock   = load("03_livestock (9)")
live_exp    = load("04_livestock_exp (9)")
enterprise  = load("08_enterprise (9)")
remittance  = load("10_remittance (6)")
energy      = load("12_energy (3)")
payment     = load("13_payment_serv (3)")
foodout     = load("19_foodout (3)")
nonfood     = load("15_non_food (3)")
urban_diary = load("16_urb_diary (3)")
rural_food  = load("17_rur_food_7d (3)")
basicvars   = load("basicvars (9)")

# ════════════════════════════════════════════════════════════════
# HOUSEHOLD COMPOSITION from indiv file
# Children     : q0105y < 14
# Leading adult: 1 if any member q0105y >= 14
# Other adults : count(q0105y >= 14) - 1
# Total members: all rows per identif
# ════════════════════════════════════════════════════════════════
print("\nӨрхийн бүрэлдэхүүн тооцож байна...")

hh_comp = indiv.copy()
age_col = "q0105y"
if age_col not in hh_comp.columns:
    print(f"  WARNING: {age_col} not found in indiv!")
    hh_comp[age_col] = np.nan

hh_comp["is_child"]  = (hh_comp[age_col].fillna(0) < 14).astype(int)
hh_comp["is_adult"]  = (hh_comp[age_col].fillna(0) >= 14).astype(int)
hh_comp["is_member"] = 1

comp_g = hh_comp.groupby(ID).agg(
    children      = ("is_child",  "sum"),
    total_adults  = ("is_adult",  "sum"),
    total_members = ("is_member", "sum")
).reset_index()

comp_g["leading_adult"] = (comp_g["total_adults"] >= 1).astype(int)
comp_g["other_adults"]  = (comp_g["total_adults"] - comp_g["leading_adult"]).clip(lower=0)
comp_g = comp_g[[ID, "children", "leading_adult", "other_adults", "total_members"]]

print(f"  Дундаж хүүхэд       : {comp_g['children'].mean():.2f}")
print(f"  Дундаж бусад насанд : {comp_g['other_adults'].mean():.2f}")
print(f"  Дундаж гишүүд       : {comp_g['total_members'].mean():.2f}")

# ════════════════════════════════════════════════════════════════
# 1. ЦАЛИН ХӨЛС
#    File: 02_indiv → q0417b + q0417c + q0419b + q0419c (horizontal)
#    then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n1. Цалин хөлс...")

wage = indiv.copy()
w_cols     = ["q0417b", "q0417c", "q0419b", "q0419c"]
existing_w = [c for c in w_cols if c in wage.columns]

if existing_w:
    wage["row_sum"] = wage[existing_w].fillna(0).sum(axis=1)
else:
    print("  WARNING: wage columns not found!")
    wage["row_sum"] = 0

tsalin = (wage.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tsalin"}))

print(f"  Нийт өрх  : {len(tsalin)}")
print(f"  Дундаж    : {tsalin['tsalin'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 2. ТЭТГЭВЭР ТЭТГЭМЖ БУСАД
#    File: 09_other_income
#    Columns: q0504 q0505 q0506 q0507 q0508 q0509 q0510 q0511
#             q0513 q0514 q0515 q0516 q0517 q0518 q0519 q0520
#             q0521 q0523 q0524 q0525 q0526 q0527 q0528 q0530
#             q0531 q0532 q0533 q0534 q0535 q0537 q0538 q0539
#             q0540 q0541 q0542
#    Sum horizontally per row, then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n2. Тэтгэвэр тэтгэмж бусад...")

oi = other_inc.copy()
t_cols = [
    "q0504","q0505","q0506","q0507","q0508","q0509","q0510","q0511",
    "q0513","q0514","q0515","q0516","q0517","q0518","q0519","q0520",
    "q0521","q0523","q0524","q0525","q0526","q0527","q0528",
    "q0530","q0531","q0532","q0533","q0534","q0535","q0537","q0538",
    "q0539","q0540","q0541","q0542"
]
existing_t = [c for c in t_cols if c in oi.columns]
missing_t  = [c for c in t_cols if c not in oi.columns]
if missing_t:
    print(f"  WARNING: columns not found in other_income: {missing_t}")

if existing_t:
    oi["row_sum"] = oi[existing_t].fillna(0).sum(axis=1)
else:
    print("  WARNING: no other_income columns found!")
    oi["row_sum"] = 0

tetgever = (oi.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tetgever"}))

print(f"  Нийт өрх  : {len(tetgever)}")
print(f"  Дундаж    : {tetgever['tetgever'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 3. ҮЙЛДВЭРЛЭЛ ҮЙЛЧИЛГЭЭ
#    Crop       : q0623b  - AgriExp   : q0624    by identif
#    Livestock  : q0605   - LiveExp   : q0606    by identif
#    Enterprise : q0710   - q0707_99             by identif
# ════════════════════════════════════════════════════════════════
print("\n3. Үйлдвэрлэл үйлчилгээ...")

# Crop income
if "q0623b" in crop.columns:
    crop_g = (crop.groupby(ID)["q0623b"]
                   .sum()
                   .reset_index()
                   .rename(columns={"q0623b": "crop_inc"}))
else:
    print("  WARNING: q0623b not found in crop!")
    crop_g = crop[[ID]].drop_duplicates()
    crop_g["crop_inc"] = 0

# Agricultural expenditure
if "q0624" in agri_exp.columns:
    agri_g = (agri_exp.groupby(ID)["q0624"]
                       .sum()
                       .reset_index()
                       .rename(columns={"q0624": "agri_exp"}))
else:
    print("  WARNING: q0624 not found in agri_exp!")
    agri_g = agri_exp[[ID]].drop_duplicates()
    agri_g["agri_exp"] = 0

# Livestock income
if "q0605" in livestock.columns:
    live_g = (livestock.groupby(ID)["q0605"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0605": "live_inc"}))
else:
    print("  WARNING: q0605 not found in livestock!")
    live_g = livestock[[ID]].drop_duplicates()
    live_g["live_inc"] = 0

# Livestock expenditure
if "q0606" in live_exp.columns:
    livee_g = (live_exp.groupby(ID)["q0606"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0606": "live_exp"}))
else:
    print("  WARNING: q0606 not found in live_exp!")
    livee_g = live_exp[[ID]].drop_duplicates()
    livee_g["live_exp"] = 0

# Enterprise: q0710 - q0707_99
ent = enterprise.copy()
ent["ent_inc"] = ent["q0710"].fillna(0)    if "q0710"    in ent.columns else 0.0
ent["ent_exp"] = ent["q0707_99"].fillna(0) if "q0707_99" in ent.columns else 0.0
if "q0710"    not in ent.columns: print("  WARNING: q0710 not found in enterprise!")
if "q0707_99" not in ent.columns: print("  WARNING: q0707_99 not found in enterprise!")
ent["ent_net"] = ent["ent_inc"] - ent["ent_exp"]

ent_g = (ent.groupby(ID)["ent_net"]
             .sum()
             .reset_index())

# Merge and calculate
uil = (crop_g
       .merge(agri_g,  on=ID, how="outer")
       .merge(live_g,  on=ID, how="outer")
       .merge(livee_g, on=ID, how="outer")
       .merge(ent_g,   on=ID, how="outer")
       .fillna(0))

uil["uildverlelal"] = ((uil["crop_inc"] - uil["agri_exp"]) +
                       (uil["live_inc"] - uil["live_exp"]) +
                        uil["ent_net"])

uildverlelal = uil[[ID, "uildverlelal"]]

print(f"  crop_inc     дундаж: {crop_g['crop_inc'].mean():,.0f}")
print(f"  agri_exp     дундаж: {agri_g['agri_exp'].mean():,.0f}")
print(f"  live_inc     дундаж: {live_g['live_inc'].mean():,.0f}")
print(f"  live_exp     дундаж: {livee_g['live_exp'].mean():,.0f}")
print(f"  ent_net      дундаж: {ent_g['ent_net'].mean():,.0f}")
print(f"  uildverlelal дундаж: {uil['uildverlelal'].mean():,.0f}")



# ════════════════════════════════════════════════════════════════
# 4. БЭЛЭГ ТУСЛАМЖ ӨӨРИЙН АЖ АХУй
#    Remittance      : q0548                              sum by identif
#    Energy          : q0941 + q0942                      sum by identif
#    Payment service : q0946 + q0947                      sum by identif
#    Foodout         : q1309 × 52                         sum by identif
#    Non-Food        : q1104 + q1105                      sum by identif
#    Urban Diary     : q1404*(q1201_3+q1201_4+q1202_3+
#                      q1202_4+q1203_3+q1203_4) horiz     sum by identif × 12
#    Rural Food 7d   : (q1305+q1306)*q1304 horiz          sum by identif × 52
# ════════════════════════════════════════════════════════════════
print("\n4. Бэлэг тусламж өөрийн аж ахуй...")

# Remittance: q0548
if "q0548" in remittance.columns:
    rem_g = (remittance.groupby(ID)["q0548"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0548": "remittance"}))
else:
    print("  WARNING: q0548 not found in remittance!")
    rem_g = remittance[[ID]].drop_duplicates()
    rem_g["remittance"] = 0

# Energy: q0941 + q0942
en = energy.copy()
en_cols     = ["q0941", "q0942"]
existing_en = [c for c in en_cols if c in en.columns]
if existing_en:
    en["en_sum"] = en[existing_en].fillna(0).sum(axis=1)
else:
    print("  WARNING: energy columns not found!")
    en["en_sum"] = 0

energy_g = (en.groupby(ID)["en_sum"]
               .sum()
               .reset_index()
               .rename(columns={"en_sum": "energy"}))

# Payment service: q0946 + q0947
pv = payment.copy()
pv_cols     = ["q0946", "q0947"]
existing_pv = [c for c in pv_cols if c in pv.columns]
if existing_pv:
    pv["pv_sum"] = pv[existing_pv].fillna(0).sum(axis=1)
else:
    print("  WARNING: payment columns not found!")
    pv["pv_sum"] = 0

pay_g = (pv.groupby(ID)["pv_sum"]
            .sum()
            .reset_index()
            .rename(columns={"pv_sum": "payment"}))

# Foodout: q1309 × 52
fo = foodout.copy()
if "q1309" in fo.columns:
    fo["fo_annual"] = fo["q1309"].fillna(0) * 52
else:
    print("  WARNING: q1309 not found in foodout!")
    fo["fo_annual"] = 0

fo_g = (fo.groupby(ID)["fo_annual"]
           .sum()
           .reset_index())

# Non-Food: q1104 + q1105
nf = nonfood.copy()
nf_cols     = ["q1104", "q1105"]
existing_nf = [c for c in nf_cols if c in nf.columns]
if existing_nf:
    nf["nf_sum"] = nf[existing_nf].fillna(0).sum(axis=1)
else:
    print("  WARNING: nonfood columns not found!")
    nf["nf_sum"] = 0

nf_g = (nf.groupby(ID)["nf_sum"]
           .sum()
           .reset_index()
           .rename(columns={"nf_sum": "nonfood"}))

# Urban Diary: q1404 × (q1201_3+q1201_4+q1202_3+q1202_4+q1203_3+q1203_4) → × 12
ud = urban_diary.copy()
qty_cols    = ["q1201_3","q1201_4","q1202_3","q1202_4","q1203_3","q1203_4"]
existing_ud = [c for c in qty_cols if c in ud.columns]

if existing_ud and "q1404" in ud.columns:
    ud["qty_sum"]   = ud[existing_ud].fillna(0).sum(axis=1)
    ud["row_value"] = ud["q1404"].fillna(0) * ud["qty_sum"]
else:
    print("  WARNING: Urban Diary columns not found! (q1404 or qty cols)")
    ud["row_value"] = 0

ud_g = (ud.groupby(ID)["row_value"]
           .sum()
           .reset_index())
ud_g["ud_annual"] = ud_g["row_value"] * 12

# Rural Food 7-day: (q1305+q1306)*q1304 → × 52
rf = rural_food.copy()
for c in ["q1305","q1306","q1304"]:
    if c not in rf.columns:
        print(f"  WARNING: {c} not found in rural_food!")
        rf[c] = 0

rf["row_value"] = (rf["q1305"].fillna(0) + rf["q1306"].fillna(0)) * rf["q1304"].fillna(0)

rf_g = (rf.groupby(ID)["row_value"]
           .sum()
           .reset_index())
rf_g["rf_annual"] = rf_g["row_value"] * 52

# Merge all components
beleg_df = (rem_g
    .merge(energy_g,               on=ID, how="outer")
    .merge(pay_g,                  on=ID, how="outer")
    .merge(fo_g[[ID,"fo_annual"]], on=ID, how="outer")
    .merge(nf_g,                   on=ID, how="outer")
    .merge(ud_g[[ID,"ud_annual"]], on=ID, how="outer")
    .merge(rf_g[[ID,"rf_annual"]], on=ID, how="outer")
    .fillna(0))

beleg_df["beleg_tuslamj"] = (beleg_df["remittance"] +
                              beleg_df["energy"]     +
                              beleg_df["payment"]    +
                              beleg_df["fo_annual"]  +
                              beleg_df["nonfood"]    +
                              beleg_df["ud_annual"]  +
                              beleg_df["rf_annual"])

beleg = beleg_df[[ID, "beleg_tuslamj"]]

print(f"  remittance    дундаж: {beleg_df['remittance'].mean():,.0f}")
print(f"  energy        дундаж: {beleg_df['energy'].mean():,.0f}")
print(f"  payment       дундаж: {beleg_df['payment'].mean():,.0f}")
print(f"  fo_annual     дундаж: {beleg_df['fo_annual'].mean():,.0f}")
print(f"  nonfood       дундаж: {beleg_df['nonfood'].mean():,.0f}")
print(f"  ud_annual     дундаж: {beleg_df['ud_annual'].mean():,.0f}")
print(f"  rf_annual     дундаж: {beleg_df['rf_annual'].mean():,.0f}")
print(f"  beleg_tuslamj дундаж: {beleg_df['beleg_tuslamj'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 5. НИЙТ ОРЛОГО — merge all 4 + household composition into basicvars → data2018
# ════════════════════════════════════════════════════════════════
print("\n5. Нийт орлого нэгтгэж байна...")

data2018 = basicvars.copy()

for df_ in [tsalin, tetgever, uildverlelal, beleg, comp_g]:
    data2018 = data2018.merge(df_, on=ID, how="left")

data2018 = data2018.fillna(0)

data2018["niit_orlogo"] = (data2018["tsalin"]       +
                            data2018["tetgever"]     +
                            data2018["uildverlelal"] +
                            data2018["beleg_tuslamj"])

# Remove rows where нийт орлого is 0
data2018 = data2018[data2018["niit_orlogo"] != 0].reset_index(drop=True)
print(f"  0 орлоготой өрх хасагдсаны дараа: {len(data2018)}")
print(f"  Нийт өрх   : {len(data2018)}")
print(f"  Багана     : {list(data2018.columns)}")

# ════════════════════════════════════════════════════════════════
# ДУНДАЖ — simple and weighted
# ════════════════════════════════════════════════════════════════
print("\nДундаж тооцож байна...")

calc_cols = ["tsalin","tetgever","uildverlelal","beleg_tuslamj","niit_orlogo"]
labels    = ["1. Цалин хөлс",
             "2. Тэтгэвэр тэтгэмж бусад",
             "3. Үйлдвэрлэл үйлчилгээ",
             "4. Бэлэг тусламж өөрийн аж ахуй",
             "5. Нийт орлого"]

# Simple (unweighted) average
simple_avg = [data2018[c].mean() for c in calc_cols]

# Weighted average using hhweight
weight_col = "hhweight"
if weight_col in data2018.columns:
    w = data2018[weight_col].fillna(0)
    weighted_avg = [
        np.average(data2018[c], weights=w) if w.sum() > 0 else np.nan
        for c in calc_cols
    ]
    monthly_wtd = [v / 12 for v in weighted_avg]
else:
    print(f"  WARNING: '{weight_col}' not found in data2018!")
    weighted_avg = [np.nan] * len(calc_cols)
    monthly_wtd  = [np.nan] * len(calc_cols)

# Results table
avg_df = pd.DataFrame({
    "Орлогын төрөл"           : labels,
    "Жингүй дундаж"           : simple_avg,
    "Жинлэсэн дундаж"         : weighted_avg,
    "Сарын дундаж (жинлэсэн)" : monthly_wtd
})

print("\n── 2018 оны дундаж үзүүлэлтүүд ──")
print(avg_df.to_string(index=False))

# Save final data
save(data2018, "data2018")
save(avg_df,   "avg2018")

Loading files...

Өрхийн бүрэлдэхүүн тооцож байна...
  Дундаж хүүхэд       : 1.09
  Дундаж бусад насанд : 1.55
  Дундаж гишүүд       : 3.64

1. Цалин хөлс...
  Нийт өрх  : 16454
  Дундаж    : 5,418,106

2. Тэтгэвэр тэтгэмж бусад...
  Нийт өрх  : 16454
  Дундаж    : 3,193,848

3. Үйлдвэрлэл үйлчилгээ...
  crop_inc     дундаж: 1,783,066
  agri_exp     дундаж: 999,089
  live_inc     дундаж: 2,592,966
  live_exp     дундаж: 1,646,886
  ent_net      дундаж: 7,753,865
  uildverlelal дундаж: 3,063,037

4. Бэлэг тусламж өөрийн аж ахуй...
  remittance    дундаж: 504,743
  energy        дундаж: 88,199
  payment       дундаж: 5,409
  fo_annual     дундаж: 129,561
  nonfood       дундаж: 311,984
  ud_annual     дундаж: 0
  rf_annual     дундаж: 4,247
  beleg_tuslamj дундаж: 1,044,144

5. Нийт орлого нэгтгэж байна...
  0 орлоготой өрх хасагдсаны дараа: 16451
  Нийт өрх   : 16451
  Багана     : ['identif', 'cluster', 'newaimag', 'location', 'urban', 'region', 'month', 'quarter', 'strata', 'hhweight'

In [68]:
import os
import pandas as pd
import numpy as np

FOLDER = r"C:\Users\b22fa\Desktop\diplom unelgee\2020"
ID = "identif"

def load(filename):
    path = os.path.join(FOLDER, filename + ".dta")
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.lower()
    return df

def save(df, name):
    size = df.memory_usage(deep=True).sum() / (1024*1024)
    out = os.path.join(FOLDER, name)
    if size > 30:
        df.to_csv(out + ".csv", index=False, encoding="utf-8-sig")
        print(f"  Saved CSV: {name}.csv")
    else:
        df.to_excel(out + ".xlsx", index=False)
        print(f"  Saved Excel: {name}.xlsx")

# ── Load all files ────────────────────────────────────────────────
print("Loading files...")
indiv       = load("02_indiv (12)")
other_inc   = load("09_other_income (12)")
crop        = load("06_crop (11)")
agri_exp    = load("07_agric_exp (11)")
livestock   = load("03_livestock (11)")
live_exp    = load("04_livestock_exp (11)")
enterprise  = load("08_enterprise (11)")
remittance  = load("10_remittance (7)")
energy      = load("12_energy (4)")
payment     = load("13_payment_serv (4)")
foodout     = load("19_foodout (4)")
nonfood     = load("15_non_food (4)")
urban_diary = load("16_urb_diary (4)")
rural_food  = load("17_rur_food_7d (4)")
basicvars   = load("basicvars (11)")

# ════════════════════════════════════════════════════════════════
# HOUSEHOLD COMPOSITION from indiv file
# Children     : q0105y < 14
# Leading adult: 1 if any member q0105y >= 14
# Other adults : count(q0105y >= 14) - 1
# Total members: all rows per identif
# ════════════════════════════════════════════════════════════════
print("\nӨрхийн бүрэлдэхүүн тооцож байна...")

hh_comp = indiv.copy()
age_col = "q0105y"
if age_col not in hh_comp.columns:
    print(f"  WARNING: {age_col} not found in indiv!")
    hh_comp[age_col] = np.nan

hh_comp["is_child"]  = (hh_comp[age_col].fillna(0) < 14).astype(int)
hh_comp["is_adult"]  = (hh_comp[age_col].fillna(0) >= 14).astype(int)
hh_comp["is_member"] = 1

comp_g = hh_comp.groupby(ID).agg(
    children      = ("is_child",  "sum"),
    total_adults  = ("is_adult",  "sum"),
    total_members = ("is_member", "sum")
).reset_index()

comp_g["leading_adult"] = (comp_g["total_adults"] >= 1).astype(int)
comp_g["other_adults"]  = (comp_g["total_adults"] - comp_g["leading_adult"]).clip(lower=0)
comp_g = comp_g[[ID, "children", "leading_adult", "other_adults", "total_members"]]

print(f"  Дундаж хүүхэд       : {comp_g['children'].mean():.2f}")
print(f"  Дундаж бусад насанд : {comp_g['other_adults'].mean():.2f}")
print(f"  Дундаж гишүүд       : {comp_g['total_members'].mean():.2f}")

# ════════════════════════════════════════════════════════════════
# 1. ЦАЛИН ХӨЛС
#    File: 02_indiv (wage_job)
#    Columns: q0436b + q0436c + q0450b + q0450c  (horizontal per person)
#    then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n1. Цалин хөлс...")

wage = indiv.copy()
w_cols     = ["q0436b", "q0436c", "q0450b", "q0450c"]
existing_w = [c for c in w_cols if c in wage.columns]
missing_w  = [c for c in w_cols if c not in wage.columns]
if missing_w:
    print(f"  WARNING: wage columns not found: {missing_w}")

if existing_w:
    wage["row_sum"] = wage[existing_w].fillna(0).sum(axis=1)
else:
    print("  WARNING: no wage columns found!")
    wage["row_sum"] = 0

tsalin = (wage.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tsalin"}))

print(f"  Нийт өрх  : {len(tsalin)}")
print(f"  Дундаж    : {tsalin['tsalin'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 2. ТЭТГЭВЭР ТЭТГЭМЖ БУСАД
#    File: 09_other_income
#    Columns: q0507 q0508 q0509 q0510 q0511 q0512 q0513 q0514
#             q0516 q0517 q0518 q0519 q0520 q0521 q0522 q0523 q0524 q0525
#             q0527 q0528 q0529 q0530 q0531 q0532
#             q0534 q0535 q0536 q0537 q0538 q0539
#             q0541 q0542 q0543 q0544 q0545 q0546
#    Sum horizontally per row, then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n2. Тэтгэвэр тэтгэмж бусад...")

oi = other_inc.copy()
t_cols = [
    "q0507","q0508","q0509","q0510","q0511","q0512","q0513","q0514",
    "q0516","q0517","q0518","q0519","q0520","q0521","q0522","q0523","q0524","q0525",
    "q0527","q0528","q0529","q0530","q0531","q0532",
    "q0534","q0535","q0536","q0537","q0538","q0539",
    "q0541","q0542","q0543","q0544","q0545","q0546"
]
existing_t = [c for c in t_cols if c in oi.columns]
missing_t  = [c for c in t_cols if c not in oi.columns]
if missing_t:
    print(f"  WARNING: columns not found in other_income: {missing_t}")

if existing_t:
    oi["row_sum"] = oi[existing_t].fillna(0).sum(axis=1)
else:
    print("  WARNING: no other_income columns found!")
    oi["row_sum"] = 0

tetgever = (oi.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tetgever"}))

print(f"  Нийт өрх  : {len(tetgever)}")
print(f"  Дундаж    : {tetgever['tetgever'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 3. ҮЙЛДВЭРЛЭЛ ҮЙЛЧИЛГЭЭ
#    Crop       : q0623b  - AgriExp   : q0624    by identif
#    Livestock  : q0605   - LiveExp   : q0606    by identif
#    Enterprise : q0710   - q0707_99             by identif
# ════════════════════════════════════════════════════════════════
print("\n3. Үйлдвэрлэл үйлчилгээ...")

# Crop income
if "q0623b" in crop.columns:
    crop_g = (crop.groupby(ID)["q0623b"]
                   .sum()
                   .reset_index()
                   .rename(columns={"q0623b": "crop_inc"}))
else:
    print("  WARNING: q0623b not found in crop!")
    crop_g = crop[[ID]].drop_duplicates()
    crop_g["crop_inc"] = 0

# Agricultural expenditure
if "q0624" in agri_exp.columns:
    agri_g = (agri_exp.groupby(ID)["q0624"]
                       .sum()
                       .reset_index()
                       .rename(columns={"q0624": "agri_exp"}))
else:
    print("  WARNING: q0624 not found in agri_exp!")
    agri_g = agri_exp[[ID]].drop_duplicates()
    agri_g["agri_exp"] = 0

# Livestock income
if "q0605" in livestock.columns:
    live_g = (livestock.groupby(ID)["q0605"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0605": "live_inc"}))
else:
    print("  WARNING: q0605 not found in livestock!")
    live_g = livestock[[ID]].drop_duplicates()
    live_g["live_inc"] = 0

# Livestock expenditure
if "q0606" in live_exp.columns:
    livee_g = (live_exp.groupby(ID)["q0606"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0606": "live_exp"}))
else:
    print("  WARNING: q0606 not found in live_exp!")
    livee_g = live_exp[[ID]].drop_duplicates()
    livee_g["live_exp"] = 0

# Enterprise: q0710 - q0707_99
ent = enterprise.copy()
ent["ent_inc"] = ent["q0710"].fillna(0)    if "q0710"    in ent.columns else 0.0
ent["ent_exp"] = ent["q0707_99"].fillna(0) if "q0707_99" in ent.columns else 0.0
if "q0710"    not in ent.columns: print("  WARNING: q0710 not found in enterprise!")
if "q0707_99" not in ent.columns: print("  WARNING: q0707_99 not found in enterprise!")
ent["ent_net"] = ent["ent_inc"] - ent["ent_exp"]

ent_g = (ent.groupby(ID)["ent_net"]
             .sum()
             .reset_index())

# Merge and calculate
uil = (crop_g
       .merge(agri_g,  on=ID, how="outer")
       .merge(live_g,  on=ID, how="outer")
       .merge(livee_g, on=ID, how="outer")
       .merge(ent_g,   on=ID, how="outer")
       .fillna(0))

uil["uildverlelal"] = ((uil["crop_inc"] - uil["agri_exp"]) +
                       (uil["live_inc"] - uil["live_exp"]) +
                        uil["ent_net"])

uildverlelal = uil[[ID, "uildverlelal"]]

print(f"  crop_inc     дундаж: {crop_g['crop_inc'].mean():,.0f}")
print(f"  agri_exp     дундаж: {agri_g['agri_exp'].mean():,.0f}")
print(f"  live_inc     дундаж: {live_g['live_inc'].mean():,.0f}")
print(f"  live_exp     дундаж: {livee_g['live_exp'].mean():,.0f}")
print(f"  ent_net      дундаж: {ent_g['ent_net'].mean():,.0f}")
print(f"  uildverlelal дундаж: {uil['uildverlelal'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 4. БЭЛЭГ ТУСЛАМЖ ӨӨРИЙН АЖ АХУй
#    Remittance      : q0552                                    sum by identif
#    Energy          : q0950                                    sum by identif
#    Payment service : q0954                                    sum by identif
#    Foodout         : q1309 × 52                               sum by identif
#    Non-Food        : q1204  (all rows)                        sum by identif
#    Urban Diary     : q1303 * (q1304 + q1305) per row          sum by identif × 12
#    Rural Food 7d   : (q1405 + q1406) * q1404 per row          sum by identif × 52
# ════════════════════════════════════════════════════════════════
print("\n4. Бэлэг тусламж өөрийн аж ахуй...")

# Remittance: q0552
if "q0552" in remittance.columns:
    rem_g = (remittance.groupby(ID)["q0552"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0552": "remittance"}))
else:
    print("  WARNING: q0552 not found in remittance!")
    rem_g = remittance[[ID]].drop_duplicates()
    rem_g["remittance"] = 0

# Energy: q0950
en = energy.copy()
if "q0950" in en.columns:
    en["en_sum"] = en["q0950"].fillna(0)
else:
    print("  WARNING: q0950 not found in energy!")
    en["en_sum"] = 0

energy_g = (en.groupby(ID)["en_sum"]
               .sum()
               .reset_index()
               .rename(columns={"en_sum": "energy"}))

# Payment service: q0954
pv = payment.copy()
if "q0954" in pv.columns:
    pv["pv_sum"] = pv["q0954"].fillna(0)
else:
    print("  WARNING: q0954 not found in payment!")
    pv["pv_sum"] = 0

pay_g = (pv.groupby(ID)["pv_sum"]
            .sum()
            .reset_index()
            .rename(columns={"pv_sum": "payment"}))

# Foodout: q1309 × 52
fo = foodout.copy()
if "q1309" in fo.columns:
    fo["fo_annual"] = fo["q1309"].fillna(0) * 52
else:
    print("  WARNING: q1309 not found in foodout!")
    fo["fo_annual"] = 0

fo_g = (fo.groupby(ID)["fo_annual"]
           .sum()
           .reset_index())

# Non-Food: q1204 — sum all rows per identif
nf = nonfood.copy()
if "q1204" in nf.columns:
    nf["nf_sum"] = nf["q1204"].fillna(0)
else:
    print("  WARNING: q1204 not found in nonfood!")
    nf["nf_sum"] = 0

nf_g = (nf.groupby(ID)["nf_sum"]
           .sum()
           .reset_index()
           .rename(columns={"nf_sum": "nonfood"}))

# Urban Diary: q1303 * (q1304 + q1305) per row → sum by identif × 12
ud = urban_diary.copy()
ud_price_col = "q1303"
ud_qty_cols  = ["q1304", "q1305"]
existing_uq  = [c for c in ud_qty_cols if c in ud.columns]
missing_uq   = [c for c in ud_qty_cols if c not in ud.columns]
if missing_uq:
    print(f"  WARNING: Urban Diary qty columns not found: {missing_uq}")

if existing_uq and ud_price_col in ud.columns:
    ud["qty_sum"]   = ud[existing_uq].fillna(0).sum(axis=1)
    ud["row_value"] = ud[ud_price_col].fillna(0) * ud["qty_sum"]
else:
    print(f"  WARNING: Urban Diary columns not found! ({ud_price_col} or qty cols)")
    ud["row_value"] = 0

ud_g = (ud.groupby(ID)["row_value"]
           .sum()
           .reset_index())
ud_g["ud_annual"] = ud_g["row_value"] * 12

# Rural Food 7-day: (q1405 + q1406) * q1404 per row → sum by identif × 52
rf = rural_food.copy()
for c in ["q1405", "q1406", "q1404"]:
    if c not in rf.columns:
        print(f"  WARNING: {c} not found in rural_food!")
        rf[c] = 0

rf["row_value"] = (rf["q1405"].fillna(0) + rf["q1406"].fillna(0)) * rf["q1404"].fillna(0)

rf_g = (rf.groupby(ID)["row_value"]
           .sum()
           .reset_index())
rf_g["rf_annual"] = rf_g["row_value"] * 52

# Merge all components
beleg_df = (rem_g
    .merge(energy_g,               on=ID, how="outer")
    .merge(pay_g,                  on=ID, how="outer")
    .merge(fo_g[[ID,"fo_annual"]], on=ID, how="outer")
    .merge(nf_g,                   on=ID, how="outer")
    .merge(ud_g[[ID,"ud_annual"]], on=ID, how="outer")
    .merge(rf_g[[ID,"rf_annual"]], on=ID, how="outer")
    .fillna(0))

beleg_df["beleg_tuslamj"] = (beleg_df["remittance"] +
                              beleg_df["energy"]     +
                              beleg_df["payment"]    +
                              beleg_df["fo_annual"]  +
                              beleg_df["nonfood"]    +
                              beleg_df["ud_annual"]  +
                              beleg_df["rf_annual"])

beleg = beleg_df[[ID, "beleg_tuslamj"]]

print(f"  remittance    дундаж: {beleg_df['remittance'].mean():,.0f}")
print(f"  energy        дундаж: {beleg_df['energy'].mean():,.0f}")
print(f"  payment       дундаж: {beleg_df['payment'].mean():,.0f}")
print(f"  fo_annual     дундаж: {beleg_df['fo_annual'].mean():,.0f}")
print(f"  nonfood       дундаж: {beleg_df['nonfood'].mean():,.0f}")
print(f"  ud_annual     дундаж: {beleg_df['ud_annual'].mean():,.0f}")
print(f"  rf_annual     дундаж: {beleg_df['rf_annual'].mean():,.0f}")
print(f"  beleg_tuslamj дундаж: {beleg_df['beleg_tuslamj'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 5. НИЙТ ОРЛОГО — merge all 4 + household composition into basicvars → data2020
# ════════════════════════════════════════════════════════════════
print("\n5. Нийт орлого нэгтгэж байна...")

data2020 = basicvars.copy()

for df_ in [tsalin, tetgever, uildverlelal, beleg, comp_g]:
    data2020 = data2020.merge(df_, on=ID, how="left")

data2020 = data2020.fillna(0)

data2020["niit_orlogo"] = (data2020["tsalin"]       +
                            data2020["tetgever"]     +
                            data2020["uildverlelal"] +
                            data2020["beleg_tuslamj"])

# Remove rows where нийт орлого is 0
data2020 = data2020[data2020["niit_orlogo"] != 0].reset_index(drop=True)
print(f"  0 орлоготой өрх хасагдсаны дараа: {len(data2020)}")
print(f"  Нийт өрх   : {len(data2020)}")
print(f"  Багана     : {list(data2020.columns)}")

# ════════════════════════════════════════════════════════════════
# ДУНДАЖ — simple and weighted
# ════════════════════════════════════════════════════════════════
print("\nДундаж тооцож байна...")

calc_cols = ["tsalin", "tetgever", "uildverlelal", "beleg_tuslamj", "niit_orlogo"]
labels    = ["1. Цалин хөлс",
             "2. Тэтгэвэр тэтгэмж бусад",
             "3. Үйлдвэрлэл үйлчилгээ",
             "4. Бэлэг тусламж өөрийн аж ахуй",
             "5. Нийт орлого"]

# Simple (unweighted) average
simple_avg = [data2020[c].mean() for c in calc_cols]

# Weighted average using hhweight
weight_col = "hhweight"
if weight_col in data2020.columns:
    w = data2020[weight_col].fillna(0)
    weighted_avg = [
        np.average(data2020[c], weights=w) if w.sum() > 0 else np.nan
        for c in calc_cols
    ]
    monthly_wtd = [v / 12 for v in weighted_avg]
else:
    print(f"  WARNING: '{weight_col}' not found in data2020!")
    weighted_avg = [np.nan] * len(calc_cols)
    monthly_wtd  = [np.nan] * len(calc_cols)

# Results table
avg_df = pd.DataFrame({
    "Орлогын төрөл"           : labels,
    "Жингүй дундаж"           : simple_avg,
    "Жинлэсэн дундаж"         : weighted_avg,
    "Сарын дундаж (жинлэсэн)" : monthly_wtd
})

print("\n── 2020 оны дундаж үзүүлэлтүүд ──")
print(avg_df.to_string(index=False))

# Save final data
save(data2020, "data2020")
save(avg_df,   "avg2020")

Loading files...

Өрхийн бүрэлдэхүүн тооцож байна...
  Дундаж хүүхэд       : 1.14
  Дундаж бусад насанд : 1.49
  Дундаж гишүүд       : 3.63

1. Цалин хөлс...
  Нийт өрх  : 16460
  Дундаж    : 7,146,734

2. Тэтгэвэр тэтгэмж бусад...
  Нийт өрх  : 16460
  Дундаж    : 4,561,744

3. Үйлдвэрлэл үйлчилгээ...
  crop_inc     дундаж: 1,827,950
  agri_exp     дундаж: 1,168,439
  live_inc     дундаж: 4,329,430
  live_exp     дундаж: 2,159,842
  ent_net      дундаж: 9,071,937
  uildverlelal дундаж: 4,172,748

4. Бэлэг тусламж өөрийн аж ахуй...
  remittance    дундаж: 651,491
  energy        дундаж: 96,730
  payment       дундаж: 3,925
  fo_annual     дундаж: 68,850
  nonfood       дундаж: 405,539
  ud_annual     дундаж: 7,494
  rf_annual     дундаж: 1,859
  beleg_tuslamj дундаж: 1,235,887

5. Нийт орлого нэгтгэж байна...
  0 орлоготой өрх хасагдсаны дараа: 16458
  Нийт өрх   : 16458
  Багана     : ['identif', 'cluster', 'newaimag', 'location', 'urban', 'region', 'month', 'quarter', 'strata', 'hhwe

In [69]:
import os
import pandas as pd
import numpy as np

FOLDER = r"C:\Users\b22fa\Desktop\diplom unelgee\2022"
ID = "identif"

def load(filename):
    path = os.path.join(FOLDER, filename + ".dta")
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.lower()
    return df

def save(df, name):
    size = df.memory_usage(deep=True).sum() / (1024*1024)
    out = os.path.join(FOLDER, name)
    if size > 30:
        df.to_csv(out + ".csv", index=False, encoding="utf-8-sig")
        print(f"  Saved CSV: {name}.csv")
    else:
        df.to_excel(out + ".xlsx", index=False)
        print(f"  Saved Excel: {name}.xlsx")

# ── Load all files ────────────────────────────────────────────────
print("Loading files...")
indiv       = load("02_indiv (14)")
other_inc   = load("09_other_income (14)")
crop        = load("06_crop (13)")
agri_exp    = load("07_agric_exp (13)")
livestock   = load("03_livestock (13)")
live_exp    = load("04_livestock_exp (13)")
enterprise  = load("08_enterprise (13)")
remittance  = load("10_remittance (9)")
energy      = load("12_energy (6)")
payment     = load("13_payment_serv (6)")
foodout     = load("19_foodout (6)")
nonfood     = load("15_non_food (6)")
urban_diary = load("16_urb_diary (6)")
rural_food  = load("17_rur_food_7d (6)")
basicvars   = load("basicvars (13)")

# ════════════════════════════════════════════════════════════════
# HOUSEHOLD COMPOSITION from indiv file
# Children     : q0105y < 14
# Leading adult: 1 if any member q0105y >= 14
# Other adults : count(q0105y >= 14) - 1
# Total members: all rows per identif
# ════════════════════════════════════════════════════════════════
print("\nӨрхийн бүрэлдэхүүн тооцож байна...")

hh_comp = indiv.copy()
age_col = "q0105y"
if age_col not in hh_comp.columns:
    print(f"  WARNING: {age_col} not found in indiv!")
    hh_comp[age_col] = np.nan

hh_comp["is_child"]  = (hh_comp[age_col].fillna(0) < 14).astype(int)
hh_comp["is_adult"]  = (hh_comp[age_col].fillna(0) >= 14).astype(int)
hh_comp["is_member"] = 1

comp_g = hh_comp.groupby(ID).agg(
    children      = ("is_child",  "sum"),
    total_adults  = ("is_adult",  "sum"),
    total_members = ("is_member", "sum")
).reset_index()

comp_g["leading_adult"] = (comp_g["total_adults"] >= 1).astype(int)
comp_g["other_adults"]  = (comp_g["total_adults"] - comp_g["leading_adult"]).clip(lower=0)
comp_g = comp_g[[ID, "children", "leading_adult", "other_adults", "total_members"]]

print(f"  Дундаж хүүхэд       : {comp_g['children'].mean():.2f}")
print(f"  Дундаж бусад насанд : {comp_g['other_adults'].mean():.2f}")
print(f"  Дундаж гишүүд       : {comp_g['total_members'].mean():.2f}")

# ════════════════════════════════════════════════════════════════
# 1. ЦАЛИН ХӨЛС
#    File: 02_indiv (wage_job)
#    Columns: q0436b + q0436c + q0450b + q0450c  (horizontal per person)
#    then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n1. Цалин хөлс...")

wage = indiv.copy()
w_cols     = ["q0436b", "q0436c", "q0450b", "q0450c"]
existing_w = [c for c in w_cols if c in wage.columns]
missing_w  = [c for c in w_cols if c not in wage.columns]
if missing_w:
    print(f"  WARNING: wage columns not found: {missing_w}")

if existing_w:
    wage["row_sum"] = wage[existing_w].fillna(0).sum(axis=1)
else:
    print("  WARNING: no wage columns found!")
    wage["row_sum"] = 0

tsalin = (wage.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tsalin"}))

print(f"  Нийт өрх  : {len(tsalin)}")
print(f"  Дундаж    : {tsalin['tsalin'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 2. ТЭТГЭВЭР ТЭТГЭМЖ БУСАД
#    File: 09_other_income
#    Columns: q0507 q0508 q0509 q0510 q0511 q0512 q0513 q0514
#             q0516 q0517 q0518 q0519 q0520 q0521 q0522 q0523 q0524 q0525
#             q0527 q0528 q0529 q0530 q0531 q0532
#             q0534 q0535 q0536 q0537 q0538 q0539
#             q0541 q0542 q0543 q0544 q0545 q0546
#    Sum horizontally per row, then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n2. Тэтгэвэр тэтгэмж бусад...")

oi = other_inc.copy()
t_cols = [
    "q0507","q0508","q0509","q0510","q0511","q0512","q0513","q0514",
    "q0516","q0517","q0518","q0519","q0520","q0521","q0522","q0523","q0524","q0525",
    "q0527","q0528","q0529","q0530","q0531","q0532",
    "q0534","q0535","q0536","q0537","q0538","q0539",
    "q0541","q0542","q0543","q0544","q0545","q0546"
]
existing_t = [c for c in t_cols if c in oi.columns]
missing_t  = [c for c in t_cols if c not in oi.columns]
if missing_t:
    print(f"  WARNING: columns not found in other_income: {missing_t}")

if existing_t:
    oi["row_sum"] = oi[existing_t].fillna(0).sum(axis=1)
else:
    print("  WARNING: no other_income columns found!")
    oi["row_sum"] = 0

tetgever = (oi.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tetgever"}))

print(f"  Нийт өрх  : {len(tetgever)}")
print(f"  Дундаж    : {tetgever['tetgever'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 3. ҮЙЛДВЭРЛЭЛ ҮЙЛЧИЛГЭЭ
#    Crop       : q0623b  - AgriExp   : q0624    by identif
#    Livestock  : q0605   - LiveExp   : q0606    by identif
#    Enterprise : q0710   - q0707_99             by identif
# ════════════════════════════════════════════════════════════════
print("\n3. Үйлдвэрлэл үйлчилгээ...")

# Crop income
if "q0623b" in crop.columns:
    crop_g = (crop.groupby(ID)["q0623b"]
                   .sum()
                   .reset_index()
                   .rename(columns={"q0623b": "crop_inc"}))
else:
    print("  WARNING: q0623b not found in crop!")
    crop_g = crop[[ID]].drop_duplicates()
    crop_g["crop_inc"] = 0

# Agricultural expenditure
if "q0624" in agri_exp.columns:
    agri_g = (agri_exp.groupby(ID)["q0624"]
                       .sum()
                       .reset_index()
                       .rename(columns={"q0624": "agri_exp"}))
else:
    print("  WARNING: q0624 not found in agri_exp!")
    agri_g = agri_exp[[ID]].drop_duplicates()
    agri_g["agri_exp"] = 0

# Livestock income
if "q0605" in livestock.columns:
    live_g = (livestock.groupby(ID)["q0605"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0605": "live_inc"}))
else:
    print("  WARNING: q0605 not found in livestock!")
    live_g = livestock[[ID]].drop_duplicates()
    live_g["live_inc"] = 0

# Livestock expenditure
if "q0606" in live_exp.columns:
    livee_g = (live_exp.groupby(ID)["q0606"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0606": "live_exp"}))
else:
    print("  WARNING: q0606 not found in live_exp!")
    livee_g = live_exp[[ID]].drop_duplicates()
    livee_g["live_exp"] = 0

# Enterprise: q0710 - q0707_99
ent = enterprise.copy()
ent["ent_inc"] = ent["q0710"].fillna(0)    if "q0710"    in ent.columns else 0.0
ent["ent_exp"] = ent["q0707_99"].fillna(0) if "q0707_99" in ent.columns else 0.0
if "q0710"    not in ent.columns: print("  WARNING: q0710 not found in enterprise!")
if "q0707_99" not in ent.columns: print("  WARNING: q0707_99 not found in enterprise!")
ent["ent_net"] = ent["ent_inc"] - ent["ent_exp"]

ent_g = (ent.groupby(ID)["ent_net"]
             .sum()
             .reset_index())

# Merge and calculate
uil = (crop_g
       .merge(agri_g,  on=ID, how="outer")
       .merge(live_g,  on=ID, how="outer")
       .merge(livee_g, on=ID, how="outer")
       .merge(ent_g,   on=ID, how="outer")
       .fillna(0))

uil["uildverlelal"] = ((uil["crop_inc"] - uil["agri_exp"]) +
                       (uil["live_inc"] - uil["live_exp"]) +
                        uil["ent_net"])

uildverlelal = uil[[ID, "uildverlelal"]]

print(f"  crop_inc     дундаж: {crop_g['crop_inc'].mean():,.0f}")
print(f"  agri_exp     дундаж: {agri_g['agri_exp'].mean():,.0f}")
print(f"  live_inc     дундаж: {live_g['live_inc'].mean():,.0f}")
print(f"  live_exp     дундаж: {livee_g['live_exp'].mean():,.0f}")
print(f"  ent_net      дундаж: {ent_g['ent_net'].mean():,.0f}")
print(f"  uildverlelal дундаж: {uil['uildverlelal'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 4. БЭЛЭГ ТУСЛАМЖ ӨӨРИЙН АЖ АХУй
#    Remittance      : q0552                                    sum by identif
#    Energy          : q0950                                    sum by identif
#    Payment service : q0954                                    sum by identif
#    Foodout         : q1309 × 52                               sum by identif
#    Non-Food        : q1204  (all rows)                        sum by identif
#    Urban Diary     : q1303 * (q1304 + q1305) per row          sum by identif × 12
#    Rural Food 7d   : (q1405 + q1406) * q1404 per row          sum by identif × 52
# ════════════════════════════════════════════════════════════════
print("\n4. Бэлэг тусламж өөрийн аж ахуй...")

# Remittance: q0552
if "q0552" in remittance.columns:
    rem_g = (remittance.groupby(ID)["q0552"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0552": "remittance"}))
else:
    print("  WARNING: q0552 not found in remittance!")
    rem_g = remittance[[ID]].drop_duplicates()
    rem_g["remittance"] = 0

# Energy: q0950
en = energy.copy()
if "q0950" in en.columns:
    en["en_sum"] = en["q0950"].fillna(0)
else:
    print("  WARNING: q0950 not found in energy!")
    en["en_sum"] = 0

energy_g = (en.groupby(ID)["en_sum"]
               .sum()
               .reset_index()
               .rename(columns={"en_sum": "energy"}))

# Payment service: q0954
pv = payment.copy()
if "q0954" in pv.columns:
    pv["pv_sum"] = pv["q0954"].fillna(0)
else:
    print("  WARNING: q0954 not found in payment!")
    pv["pv_sum"] = 0

pay_g = (pv.groupby(ID)["pv_sum"]
            .sum()
            .reset_index()
            .rename(columns={"pv_sum": "payment"}))

# Foodout: q1309 × 52
fo = foodout.copy()
if "q1309" in fo.columns:
    fo["fo_annual"] = fo["q1309"].fillna(0) * 52
else:
    print("  WARNING: q1309 not found in foodout!")
    fo["fo_annual"] = 0

fo_g = (fo.groupby(ID)["fo_annual"]
           .sum()
           .reset_index())

# Non-Food: q1204 — sum all rows per identif
nf = nonfood.copy()
if "q1204" in nf.columns:
    nf["nf_sum"] = nf["q1204"].fillna(0)
else:
    print("  WARNING: q1204 not found in nonfood!")
    nf["nf_sum"] = 0

nf_g = (nf.groupby(ID)["nf_sum"]
           .sum()
           .reset_index()
           .rename(columns={"nf_sum": "nonfood"}))

# Urban Diary: q1303 * (q1304 + q1305) per row → sum by identif × 12
ud = urban_diary.copy()
ud_price_col = "q1303"
ud_qty_cols  = ["q1304", "q1305"]
existing_uq  = [c for c in ud_qty_cols if c in ud.columns]
missing_uq   = [c for c in ud_qty_cols if c not in ud.columns]
if missing_uq:
    print(f"  WARNING: Urban Diary qty columns not found: {missing_uq}")

if existing_uq and ud_price_col in ud.columns:
    ud["qty_sum"]   = ud[existing_uq].fillna(0).sum(axis=1)
    ud["row_value"] = ud[ud_price_col].fillna(0) * ud["qty_sum"]
else:
    print(f"  WARNING: Urban Diary columns not found! ({ud_price_col} or qty cols)")
    ud["row_value"] = 0

ud_g = (ud.groupby(ID)["row_value"]
           .sum()
           .reset_index())
ud_g["ud_annual"] = ud_g["row_value"] * 12

# Rural Food 7-day: (q1405 + q1406) * q1404 per row → sum by identif × 52
rf = rural_food.copy()
for c in ["q1405", "q1406", "q1404"]:
    if c not in rf.columns:
        print(f"  WARNING: {c} not found in rural_food!")
        rf[c] = 0

rf["row_value"] = (rf["q1405"].fillna(0) + rf["q1406"].fillna(0)) * rf["q1404"].fillna(0)

rf_g = (rf.groupby(ID)["row_value"]
           .sum()
           .reset_index())
rf_g["rf_annual"] = rf_g["row_value"] * 52

# Merge all components
beleg_df = (rem_g
    .merge(energy_g,               on=ID, how="outer")
    .merge(pay_g,                  on=ID, how="outer")
    .merge(fo_g[[ID,"fo_annual"]], on=ID, how="outer")
    .merge(nf_g,                   on=ID, how="outer")
    .merge(ud_g[[ID,"ud_annual"]], on=ID, how="outer")
    .merge(rf_g[[ID,"rf_annual"]], on=ID, how="outer")
    .fillna(0))

beleg_df["beleg_tuslamj"] = (beleg_df["remittance"] +
                              beleg_df["energy"]     +
                              beleg_df["payment"]    +
                              beleg_df["fo_annual"]  +
                              beleg_df["nonfood"]    +
                              beleg_df["ud_annual"]  +
                              beleg_df["rf_annual"])

beleg = beleg_df[[ID, "beleg_tuslamj"]]

print(f"  remittance    дундаж: {beleg_df['remittance'].mean():,.0f}")
print(f"  energy        дундаж: {beleg_df['energy'].mean():,.0f}")
print(f"  payment       дундаж: {beleg_df['payment'].mean():,.0f}")
print(f"  fo_annual     дундаж: {beleg_df['fo_annual'].mean():,.0f}")
print(f"  nonfood       дундаж: {beleg_df['nonfood'].mean():,.0f}")
print(f"  ud_annual     дундаж: {beleg_df['ud_annual'].mean():,.0f}")
print(f"  rf_annual     дундаж: {beleg_df['rf_annual'].mean():,.0f}")
print(f"  beleg_tuslamj дундаж: {beleg_df['beleg_tuslamj'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 5. НИЙТ ОРЛОГО — merge all 4 + household composition into basicvars → data2022
# ════════════════════════════════════════════════════════════════
print("\n5. Нийт орлого нэгтгэж байна...")

data2022 = basicvars.copy()

for df_ in [tsalin, tetgever, uildverlelal, beleg, comp_g]:
    data2022 = data2022.merge(df_, on=ID, how="left")

data2022 = data2022.fillna(0)

data2022["niit_orlogo"] = (data2022["tsalin"]       +
                            data2022["tetgever"]     +
                            data2022["uildverlelal"] +
                            data2022["beleg_tuslamj"])

# Remove rows where нийт орлого is 0
data2022 = data2022[data2022["niit_orlogo"] != 0].reset_index(drop=True)
print(f"  0 орлоготой өрх хасагдсаны дараа: {len(data2022)}")
print(f"  Нийт өрх   : {len(data2022)}")
print(f"  Багана     : {list(data2022.columns)}")

# ════════════════════════════════════════════════════════════════
# ДУНДАЖ — simple and weighted
# ════════════════════════════════════════════════════════════════
print("\nДундаж тооцож байна...")

calc_cols = ["tsalin", "tetgever", "uildverlelal", "beleg_tuslamj", "niit_orlogo"]
labels    = ["1. Цалин хөлс",
             "2. Тэтгэвэр тэтгэмж бусад",
             "3. Үйлдвэрлэл үйлчилгээ",
             "4. Бэлэг тусламж өөрийн аж ахуй",
             "5. Нийт орлого"]

# Simple (unweighted) average
simple_avg = [data2022[c].mean() for c in calc_cols]

# Weighted average using hhweight
weight_col = "hhweight"
if weight_col in data2022.columns:
    w = data2022[weight_col].fillna(0)
    weighted_avg = [
        np.average(data2022[c], weights=w) if w.sum() > 0 else np.nan
        for c in calc_cols
    ]
    monthly_wtd = [v / 12 for v in weighted_avg]
else:
    print(f"  WARNING: '{weight_col}' not found in data2022!")
    weighted_avg = [np.nan] * len(calc_cols)
    monthly_wtd  = [np.nan] * len(calc_cols)

# Results table
avg_df = pd.DataFrame({
    "Орлогын төрөл"           : labels,
    "Жингүй дундаж"           : simple_avg,
    "Жинлэсэн дундаж"         : weighted_avg,
    "Сарын дундаж (жинлэсэн)" : monthly_wtd
})

print("\n── 2022 оны дундаж үзүүлэлтүүд ──")
print(avg_df.to_string(index=False))

# Save final data
save(data2022, "data2022")
save(avg_df,   "avg2022")

Loading files...

Өрхийн бүрэлдэхүүн тооцож байна...
  Дундаж хүүхэд       : 1.08
  Дундаж бусад насанд : 1.44
  Дундаж гишүүд       : 3.52

1. Цалин хөлс...
  Нийт өрх  : 22995
  Дундаж    : 8,033,824

2. Тэтгэвэр тэтгэмж бусад...
  Нийт өрх  : 22995
  Дундаж    : 6,727,957

3. Үйлдвэрлэл үйлчилгээ...
  crop_inc     дундаж: 3,619,805
  agri_exp     дундаж: 2,282,633
  live_inc     дундаж: 5,064,467
  live_exp     дундаж: 3,275,375
  ent_net      дундаж: 11,237,557
  uildverlelal дундаж: 3,981,972

4. Бэлэг тусламж өөрийн аж ахуй...
  remittance    дундаж: 598,812
  energy        дундаж: 169,159
  payment       дундаж: 19,967
  fo_annual     дундаж: 97,832
  nonfood       дундаж: 315,827
  ud_annual     дундаж: 8,672
  rf_annual     дундаж: 1,194
  beleg_tuslamj дундаж: 1,211,464

5. Нийт орлого нэгтгэж байна...
  0 орлоготой өрх хасагдсаны дараа: 22992
  Нийт өрх   : 22992
  Багана     : ['identif', 'cluster', 'newaimag', 'location', 'urban', 'region', 'month', 'quarter', 'strata', 'h

In [73]:
import os
import pandas as pd
import numpy as np

FOLDER = r"C:\Users\b22fa\Desktop\diplom unelgee\2024"
ID = "identif"

def load(filename):
    path = os.path.join(FOLDER, filename + ".dta")
    df = pd.read_stata(path, convert_categoricals=False)
    df.columns = df.columns.str.lower()
    return df

def save(df, name):
    size = df.memory_usage(deep=True).sum() / (1024*1024)
    out = os.path.join(FOLDER, name)
    if size > 30:
        df.to_csv(out + ".csv", index=False, encoding="utf-8-sig")
        print(f"  Saved CSV: {name}.csv")
    else:
        df.to_excel(out + ".xlsx", index=False)
        print(f"  Saved Excel: {name}.xlsx")

# ── Load all files ────────────────────────────────────────────────
print("Loading files...")
indiv       = load("02_indiv (1)")
other_inc   = load("09_other_income")
crop        = load("06_crop")
agri_exp    = load("07_agric_exp")
livestock   = load("03_livestock")
live_exp    = load("04_livestock_exp")
enterprise  = load("08_enterprise")
remittance  = load("10_remittance")
energy      = load("12_energy")
payment     = load("13_payment_serv")
foodout     = load("19_foodout")
nonfood     = load("15_non_food")
urban_diary = load("16_urb_diary")
rural_food  = load("17_rur_food_7d")
basicvars   = load("basicvars")

# ════════════════════════════════════════════════════════════════
# HOUSEHOLD COMPOSITION from indiv file
# Children     : q0105y < 14
# Leading adult: 1 if any member q0105y >= 14
# Other adults : count(q0105y >= 14) - 1
# Total members: all rows per identif
# ════════════════════════════════════════════════════════════════
print("\nӨрхийн бүрэлдэхүүн тооцож байна...")

hh_comp = indiv.copy()
age_col = "q0105y"
if age_col not in hh_comp.columns:
    print(f"  WARNING: {age_col} not found in indiv!")
    hh_comp[age_col] = np.nan

hh_comp["is_child"]  = (hh_comp[age_col].fillna(0) < 14).astype(int)
hh_comp["is_adult"]  = (hh_comp[age_col].fillna(0) >= 14).astype(int)
hh_comp["is_member"] = 1

comp_g = hh_comp.groupby(ID).agg(
    children      = ("is_child",  "sum"),
    total_adults  = ("is_adult",  "sum"),
    total_members = ("is_member", "sum")
).reset_index()

comp_g["leading_adult"] = (comp_g["total_adults"] >= 1).astype(int)
comp_g["other_adults"]  = (comp_g["total_adults"] - comp_g["leading_adult"]).clip(lower=0)
comp_g = comp_g[[ID, "children", "leading_adult", "other_adults", "total_members"]]

print(f"  Дундаж хүүхэд       : {comp_g['children'].mean():.2f}")
print(f"  Дундаж бусад насанд : {comp_g['other_adults'].mean():.2f}")
print(f"  Дундаж гишүүд       : {comp_g['total_members'].mean():.2f}")

# ════════════════════════════════════════════════════════════════
# 1. ЦАЛИН ХӨЛС
#    File: 02_indiv (wage_job)
#    Columns: q0436b + q0436c + q0450b + q0450c  (horizontal per person)
#    then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n1. Цалин хөлс...")

wage = indiv.copy()
w_cols     = ["q0436b", "q0436c", "q0450b", "q0450c"]
existing_w = [c for c in w_cols if c in wage.columns]
missing_w  = [c for c in w_cols if c not in wage.columns]
if missing_w:
    print(f"  WARNING: wage columns not found: {missing_w}")

if existing_w:
    wage["row_sum"] = wage[existing_w].fillna(0).sum(axis=1)
else:
    print("  WARNING: no wage columns found!")
    wage["row_sum"] = 0

tsalin = (wage.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tsalin"}))

print(f"  Нийт өрх  : {len(tsalin)}")
print(f"  Дундаж    : {tsalin['tsalin'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 2. ТЭТГЭВЭР ТЭТГЭМЖ БУСАД
#    File: 09_other_income
#    Columns: q0507 q0508 q0509 q0510 q0511 q0512 q0513 q0514
#             q0516 q0517 q0518 q0519 q0520 q0521 q0522 q0523 q0524 q0525
#             q0527 q0528 q0529 q0530 q0531 q0532
#             q0534 q0535 q0536 q0537 q0538 q0539
#             q0541 q0542 q0543 q0544 q0545 q0546
#    Sum horizontally per row, then sum by identif
# ════════════════════════════════════════════════════════════════
print("\n2. Тэтгэвэр тэтгэмж бусад...")

oi = other_inc.copy()
t_cols = [
    "q0507","q0508","q0509","q0510","q0511","q0512","q0513","q0514",
    "q0516","q0517","q0518","q0519","q0520","q0521","q0522","q0523","q0524","q0525",
    "q0527","q0528","q0529","q0530","q0531","q0532",
    "q0534","q0535","q0536","q0537","q0538","q0539",
    "q0541","q0542","q0543","q0544","q0545","q0546"
]
existing_t = [c for c in t_cols if c in oi.columns]
missing_t  = [c for c in t_cols if c not in oi.columns]
if missing_t:
    print(f"  WARNING: columns not found in other_income: {missing_t}")

if existing_t:
    oi["row_sum"] = oi[existing_t].fillna(0).sum(axis=1)
else:
    print("  WARNING: no other_income columns found!")
    oi["row_sum"] = 0

tetgever = (oi.groupby(ID)["row_sum"]
               .sum()
               .reset_index()
               .rename(columns={"row_sum": "tetgever"}))

print(f"  Нийт өрх  : {len(tetgever)}")
print(f"  Дундаж    : {tetgever['tetgever'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 3. ҮЙЛДВЭРЛЭЛ ҮЙЛЧИЛГЭЭ
#    Crop       : q0623b  - AgriExp   : q0624    by identif
#    Livestock  : q0605   - LiveExp   : q0606    by identif
#    Enterprise : q0710   - q0707_99             by identif
# ════════════════════════════════════════════════════════════════
print("\n3. Үйлдвэрлэл үйлчилгээ...")

# Crop income
if "q0623b" in crop.columns:
    crop_g = (crop.groupby(ID)["q0623b"]
                   .sum()
                   .reset_index()
                   .rename(columns={"q0623b": "crop_inc"}))
else:
    print("  WARNING: q0623b not found in crop!")
    crop_g = crop[[ID]].drop_duplicates()
    crop_g["crop_inc"] = 0

# Agricultural expenditure
if "q0624" in agri_exp.columns:
    agri_g = (agri_exp.groupby(ID)["q0624"]
                       .sum()
                       .reset_index()
                       .rename(columns={"q0624": "agri_exp"}))
else:
    print("  WARNING: q0624 not found in agri_exp!")
    agri_g = agri_exp[[ID]].drop_duplicates()
    agri_g["agri_exp"] = 0

# Livestock income
if "q0605" in livestock.columns:
    live_g = (livestock.groupby(ID)["q0605"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0605": "live_inc"}))
else:
    print("  WARNING: q0605 not found in livestock!")
    live_g = livestock[[ID]].drop_duplicates()
    live_g["live_inc"] = 0

# Livestock expenditure
if "q0606" in live_exp.columns:
    livee_g = (live_exp.groupby(ID)["q0606"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0606": "live_exp"}))
else:
    print("  WARNING: q0606 not found in live_exp!")
    livee_g = live_exp[[ID]].drop_duplicates()
    livee_g["live_exp"] = 0

# Enterprise: q0710 - q0707_99
ent = enterprise.copy()
ent["ent_inc"] = ent["q0710"].fillna(0)    if "q0710"    in ent.columns else 0.0
ent["ent_exp"] = ent["q0707_99"].fillna(0) if "q0707_99" in ent.columns else 0.0
if "q0710"    not in ent.columns: print("  WARNING: q0710 not found in enterprise!")
if "q0707_99" not in ent.columns: print("  WARNING: q0707_99 not found in enterprise!")
ent["ent_net"] = ent["ent_inc"] - ent["ent_exp"]

ent_g = (ent.groupby(ID)["ent_net"]
             .sum()
             .reset_index())

# Merge and calculate
uil = (crop_g
       .merge(agri_g,  on=ID, how="outer")
       .merge(live_g,  on=ID, how="outer")
       .merge(livee_g, on=ID, how="outer")
       .merge(ent_g,   on=ID, how="outer")
       .fillna(0))

uil["uildverlelal"] = ((uil["crop_inc"] - uil["agri_exp"]) +
                       (uil["live_inc"] - uil["live_exp"]) +
                        uil["ent_net"])

uildverlelal = uil[[ID, "uildverlelal"]]

print(f"  crop_inc     дундаж: {crop_g['crop_inc'].mean():,.0f}")
print(f"  agri_exp     дундаж: {agri_g['agri_exp'].mean():,.0f}")
print(f"  live_inc     дундаж: {live_g['live_inc'].mean():,.0f}")
print(f"  live_exp     дундаж: {livee_g['live_exp'].mean():,.0f}")
print(f"  ent_net      дундаж: {ent_g['ent_net'].mean():,.0f}")
print(f"  uildverlelal дундаж: {uil['uildverlelal'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 4. БЭЛЭГ ТУСЛАМЖ ӨӨРИЙН АЖ АХУй
#    Remittance      : q0559                                    sum by identif   ← CHANGED (was q0552)
#    Energy          : q0966                                    sum by identif   ← CHANGED (was q0950)
#    Payment service : q0970                                    sum by identif   ← CHANGED (was q0954)
#    Foodout         : q1309 × 52                               sum by identif
#    Non-Food        : q1204  (all rows)                        sum by identif
#    Urban Diary     : q1303 * (q1304 + q1305) per row          sum by identif × 12
#    Rural Food 7d   : (q1405 + q1406) * q1410 per row          sum by identif × 52  ← CHANGED (was q1404)
# ════════════════════════════════════════════════════════════════
print("\n4. Бэлэг тусламж өөрийн аж ахуй...")

# Remittance: q0559  ← CHANGED
if "q0559" in remittance.columns:
    rem_g = (remittance.groupby(ID)["q0559"]
                        .sum()
                        .reset_index()
                        .rename(columns={"q0559": "remittance"}))
else:
    print("  WARNING: q0559 not found in remittance!")
    rem_g = remittance[[ID]].drop_duplicates()
    rem_g["remittance"] = 0

# Energy: q0966  ← CHANGED
en = energy.copy()
if "q0966" in en.columns:
    en["en_sum"] = en["q0966"].fillna(0)
else:
    print("  WARNING: q0966 not found in energy!")
    en["en_sum"] = 0

energy_g = (en.groupby(ID)["en_sum"]
               .sum()
               .reset_index()
               .rename(columns={"en_sum": "energy"}))

# Payment service: q0970  ← CHANGED
pv = payment.copy()
if "q0970" in pv.columns:
    pv["pv_sum"] = pv["q0970"].fillna(0)
else:
    print("  WARNING: q0970 not found in payment!")
    pv["pv_sum"] = 0

pay_g = (pv.groupby(ID)["pv_sum"]
            .sum()
            .reset_index()
            .rename(columns={"pv_sum": "payment"}))

# Foodout: q1309 × 52
fo = foodout.copy()
if "q1309" in fo.columns:
    fo["fo_annual"] = fo["q1309"].fillna(0) * 52
else:
    print("  WARNING: q1309 not found in foodout!")
    fo["fo_annual"] = 0

fo_g = (fo.groupby(ID)["fo_annual"]
           .sum()
           .reset_index())

# Non-Food: q1204 — sum all rows per identif
nf = nonfood.copy()
if "q1204" in nf.columns:
    nf["nf_sum"] = nf["q1204"].fillna(0)
else:
    print("  WARNING: q1204 not found in nonfood!")
    nf["nf_sum"] = 0

nf_g = (nf.groupby(ID)["nf_sum"]
           .sum()
           .reset_index()
           .rename(columns={"nf_sum": "nonfood"}))

# Urban Diary: q1303 * (q1304 + q1305) per row → sum by identif × 12
ud = urban_diary.copy()
ud_price_col = "q1303"
ud_qty_cols  = ["q1304", "q1305"]
existing_uq  = [c for c in ud_qty_cols if c in ud.columns]
missing_uq   = [c for c in ud_qty_cols if c not in ud.columns]
if missing_uq:
    print(f"  WARNING: Urban Diary qty columns not found: {missing_uq}")

if existing_uq and ud_price_col in ud.columns:
    ud["qty_sum"]   = ud[existing_uq].fillna(0).sum(axis=1)
    ud["row_value"] = ud[ud_price_col].fillna(0) * ud["qty_sum"]
else:
    print(f"  WARNING: Urban Diary columns not found! ({ud_price_col} or qty cols)")
    ud["row_value"] = 0

ud_g = (ud.groupby(ID)["row_value"]
           .sum()
           .reset_index())
ud_g["ud_annual"] = ud_g["row_value"] * 12

# Rural Food 7-day: (q1405 + q1406) * q1410 / q1409 per row → sum by identif × 52
rf = rural_food.copy()
for c in ["q1405", "q1406", "q1410", "q1409"]:
    if c not in rf.columns:
        print(f"  WARNING: {c} not found in rural_food!")
        rf[c] = 0

rf["row_value"] = (
    (rf["q1405"].fillna(0) + rf["q1406"].fillna(0))
    * rf["q1410"].fillna(0)
    / rf["q1409"].replace(0, np.nan)  # avoid division by zero
).fillna(0)

rf_g = (rf.groupby(ID)["row_value"]
           .sum()
           .reset_index())
rf_g["rf_annual"] = rf_g["row_value"] * 12

# Merge all components
beleg_df = (rem_g
    .merge(energy_g,               on=ID, how="outer")
    .merge(pay_g,                  on=ID, how="outer")
    .merge(fo_g[[ID,"fo_annual"]], on=ID, how="outer")
    .merge(nf_g,                   on=ID, how="outer")
    .merge(ud_g[[ID,"ud_annual"]], on=ID, how="outer")
    .merge(rf_g[[ID,"rf_annual"]], on=ID, how="outer")
    .fillna(0))

beleg_df["beleg_tuslamj"] = (beleg_df["remittance"] +
                              beleg_df["energy"]     +
                              beleg_df["payment"]    +
                              beleg_df["fo_annual"]  +
                              beleg_df["nonfood"]    +
                              beleg_df["ud_annual"]  +
                              beleg_df["rf_annual"])

beleg = beleg_df[[ID, "beleg_tuslamj"]]

print(f"  remittance    дундаж: {beleg_df['remittance'].mean():,.0f}")
print(f"  energy        дундаж: {beleg_df['energy'].mean():,.0f}")
print(f"  payment       дундаж: {beleg_df['payment'].mean():,.0f}")
print(f"  fo_annual     дундаж: {beleg_df['fo_annual'].mean():,.0f}")
print(f"  nonfood       дундаж: {beleg_df['nonfood'].mean():,.0f}")
print(f"  ud_annual     дундаж: {beleg_df['ud_annual'].mean():,.0f}")
print(f"  rf_annual     дундаж: {beleg_df['rf_annual'].mean():,.0f}")
print(f"  beleg_tuslamj дундаж: {beleg_df['beleg_tuslamj'].mean():,.0f}")

# ════════════════════════════════════════════════════════════════
# 5. НИЙТ ОРЛОГО — merge all 4 + household composition into basicvars → data2024
# ════════════════════════════════════════════════════════════════
print("\n5. Нийт орлого нэгтгэж байна...")

data2024 = basicvars.copy()

for df_ in [tsalin, tetgever, uildverlelal, beleg, comp_g]:
    data2024 = data2024.merge(df_, on=ID, how="left")

data2024 = data2024.fillna(0)

data2024["niit_orlogo"] = (data2024["tsalin"]       +
                            data2024["tetgever"]     +
                            data2024["uildverlelal"] +
                            data2024["beleg_tuslamj"])

# Remove rows where нийт орлого is 0
data2024 = data2024[data2024["niit_orlogo"] != 0].reset_index(drop=True)
print(f"  0 орлоготой өрх хасагдсаны дараа: {len(data2024)}")
print(f"  Нийт өрх   : {len(data2024)}")
print(f"  Багана     : {list(data2024.columns)}")

# ════════════════════════════════════════════════════════════════
# ДУНДАЖ — simple and weighted
# ════════════════════════════════════════════════════════════════
print("\nДундаж тооцож байна...")

calc_cols = ["tsalin", "tetgever", "uildverlelal", "beleg_tuslamj", "niit_orlogo"]
labels    = ["1. Цалин хөлс",
             "2. Тэтгэвэр тэтгэмж бусад",
             "3. Үйлдвэрлэл үйлчилгээ",
             "4. Бэлэг тусламж өөрийн аж ахуй",
             "5. Нийт орлого"]

# Simple (unweighted) average
simple_avg = [data2024[c].mean() for c in calc_cols]

# Weighted average using hhweight_final
weight_col1 = "hhweight_final"
if weight_col1 in data2024.columns:
    w1 = data2024[weight_col1].fillna(0)
    weighted_avg1 = [
        np.average(data2024[c], weights=w1) if w1.sum() > 0 else np.nan
        for c in calc_cols
    ]
    monthly_wtd1 = [v / 12 for v in weighted_avg1]
else:
    print(f"  WARNING: '{weight_col1}' not found in data2024!")
    weighted_avg1 = [np.nan] * len(calc_cols)
    monthly_wtd1  = [np.nan] * len(calc_cols)

# Weighted average using hhweight_pa_final
weight_col2 = "hhweight_pa_final"
if weight_col2 in data2024.columns:
    w2 = data2024[weight_col2].fillna(0)
    weighted_avg2 = [
        np.average(data2024[c], weights=w2) if w2.sum() > 0 else np.nan
        for c in calc_cols
    ]
    monthly_wtd2 = [v / 12 for v in weighted_avg2]
else:
    print(f"  WARNING: '{weight_col2}' not found in data2024!")
    weighted_avg2 = [np.nan] * len(calc_cols)
    monthly_wtd2  = [np.nan] * len(calc_cols)

# Results table
avg_df = pd.DataFrame({
    "Орлогын төрөл"                        : labels,
    "Жингүй дундаж"                        : simple_avg,
    "Жинлэсэн дундаж (hhweight_final)"     : weighted_avg1,
    "Сарын дундаж (hhweight_final)"        : monthly_wtd1,
    "Жинлэсэн дундаж (hhweight_pa_final)"  : weighted_avg2,
    "Сарын дундаж (hhweight_pa_final)"     : monthly_wtd2,
})

print("\n── 2024 оны дундаж үзүүлэлтүүд ──")
print(avg_df.to_string(index=False))

# Save final data
save(data2024, "data2024")
save(avg_df,   "avg2024")

Loading files...

Өрхийн бүрэлдэхүүн тооцож байна...
  Дундаж хүүхэд       : 1.02
  Дундаж бусад насанд : 1.45
  Дундаж гишүүд       : 3.47

1. Цалин хөлс...
  Нийт өрх  : 15513
  Дундаж    : 12,555,551

2. Тэтгэвэр тэтгэмж бусад...
  Нийт өрх  : 15513
  Дундаж    : 8,903,154

3. Үйлдвэрлэл үйлчилгээ...
  crop_inc     дундаж: 4,095,725
  agri_exp     дундаж: 3,128,376
  live_inc     дундаж: 6,936,443
  live_exp     дундаж: 5,847,387
  ent_net      дундаж: 15,849,673
  uildverlelal дундаж: 4,256,586

4. Бэлэг тусламж өөрийн аж ахуй...
  remittance    дундаж: 507,983
  energy        дундаж: 132,615
  payment       дундаж: 7,540
  fo_annual     дундаж: 65,409
  nonfood       дундаж: 333,360
  ud_annual     дундаж: 6,052
  rf_annual     дундаж: 306,265
  beleg_tuslamj дундаж: 1,359,225

5. Нийт орлого нэгтгэж байна...
  0 орлоготой өрх хасагдсаны дараа: 15511
  Нийт өрх   : 15511
  Багана     : ['identif', 'cluster', 'newaimag', 'newsoum', 'bag', 'urban', 'region', 'location', 'strata', 'm

In [74]:
import os
import glob
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Configuration ─────────────────────────────────────────────────────────────
BASE   = r"C:\Users\b22fa\Desktop\diplom unelgee"
YEARS  = [2008, 2010, 2012, 2014, 2016, 2018, 2020, 2022, 2024]
ID     = "identif"

INCOME_COLS = ["tsalin", "tetgever", "uildverlelal", "beleg_tuslamj", "niit_orlogo"]
COMP_COLS   = ["children", "leading_adult", "other_adults", "total_members"]

# ── Helpers ───────────────────────────────────────────────────────────────────
def folder(year):
    return os.path.join(BASE, str(year))

def find_data_file(year):
    """Find data{year}.xlsx or data{year}.csv in the year folder."""
    d = folder(year)
    for ext in ["xlsx", "csv"]:
        pattern = os.path.join(d, f"data{year}.{ext}")
        matches = glob.glob(pattern)
        if matches:
            return matches[0], ext
    raise FileNotFoundError(f"No data{year}.xlsx/.csv found in {d}")

def find_avg_file(year):
    """Find avg{year}.xlsx or avg{year}.csv in the year folder."""
    d = folder(year)
    for ext in ["xlsx", "csv"]:
        pattern = os.path.join(d, f"avg{year}.{ext}")
        matches = glob.glob(pattern)
        if matches:
            return matches[0], ext
    raise FileNotFoundError(f"No avg{year}.xlsx/.csv found in {d}")

def read_file(path, ext):
    if ext == "csv":
        return pd.read_csv(path, encoding="utf-8-sig")
    return pd.read_excel(path)

# ── Sheet 1: All household-level data ─────────────────────────────────────────
print("Sheet 1: өрхийн түвшний мэдээлэл уншиж байна...")

all_frames = []
for year in YEARS:
    try:
        path, ext = find_data_file(year)
        df = read_file(path, ext)
        df.columns = df.columns.str.lower()

        keep = [ID] + COMP_COLS + INCOME_COLS
        existing = [c for c in keep if c in df.columns]
        missing  = [c for c in keep if c not in df.columns]
        if missing:
            print(f"  {year} WARNING: columns not found: {missing}")
            for c in missing:
                df[c] = np.nan

        df = df[keep].copy()
        df.insert(0, "year", year)
        all_frames.append(df)
        print(f"  {year}: {len(df):,} өрх уншлаа")
    except FileNotFoundError as e:
        print(f"  {year} SKIP: {e}")

alldata = pd.concat(all_frames, ignore_index=True)
print(f"\nНийт өрх: {len(alldata):,}")

# ── Sheet 2: Weighted averages ─────────────────────────────────────────────────
print("\nSheet 2: жинлэсэн дундаж уншиж байна...")

avg_frames = []
for year in YEARS:
    try:
        path, ext = find_avg_file(year)
        df = read_file(path, ext)
        df.insert(0, "year", year)
        avg_frames.append(df)
        print(f"  {year}: avg файл уншлаа")
    except FileNotFoundError as e:
        print(f"  {year} SKIP: {e}")

avg_all = pd.concat(avg_frames, ignore_index=True)

# Ensure monthly column exists (annual / 12)
if "Сарын дундаж (жинлэсэн)" not in avg_all.columns and "Жинлэсэн дундаж" in avg_all.columns:
    avg_all["Сарын дундаж (жинлэсэн)"] = avg_all["Жинлэсэн дундаж"] / 12

# ── Build Excel ────────────────────────────────────────────────────────────────
print("\nExcel файл үүсгэж байна...")

OUT = os.path.join(BASE, "alldata.xlsx")

# Style helpers
HEADER_FILL   = PatternFill("solid", fgColor="1F4E79")
SUBHDR_FILL   = PatternFill("solid", fgColor="2E75B6")
ALT_FILL      = PatternFill("solid", fgColor="D6E4F0")
WHITE_FILL    = PatternFill("solid", fgColor="FFFFFF")
HDR_FONT      = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT     = Font(name="Arial", size=10)
BOLD_FONT     = Font(name="Arial", bold=True, size=10)
CENTER        = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT          = Alignment(horizontal="left",   vertical="center")
RIGHT         = Alignment(horizontal="right",  vertical="center")

thin = Side(style="thin", color="BFBFBF")
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)

def style_header(cell, fill=HEADER_FILL):
    cell.font      = HDR_FONT
    cell.fill      = fill
    cell.alignment = CENTER
    cell.border    = BORDER

def style_body(cell, align=RIGHT, alt=False):
    cell.font      = BODY_FONT
    cell.fill      = ALT_FILL if alt else WHITE_FILL
    cell.alignment = align
    cell.border    = BORDER

wb = Workbook()

# ════════════════════════════════════════════════════════════════
# SHEET 1
# ════════════════════════════════════════════════════════════════
ws1 = wb.active
ws1.title = "Өрхийн мэдээлэл"

col_defs = [
    ("year",          "Он",                         10),
    (ID,              "Өрхийн дугаар",               16),
    ("children",      "Хүүхэд (14-с доош)",          16),
    ("leading_adult", "Тэргүүлэх насанд хүрсэн",     20),
    ("other_adults",  "Бусад насанд хүрсэн",          18),
    ("total_members", "Нийт гишүүд",                  14),
    ("tsalin",        "Цалин хөлс",                  18),
    ("tetgever",      "Тэтгэвэр тэтгэмж бусад",       22),
    ("uildverlelal",  "Үйлдвэрлэл үйлчилгээ",         22),
    ("beleg_tuslamj", "Бэлэг тусламж өөрийн аж ахуй", 26),
    ("niit_orlogo",   "Нийт орлого",                  16),
]

keys   = [c[0] for c in col_defs]
labels = [c[1] for c in col_defs]
widths = [c[2] for c in col_defs]

# Title row
ws1.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(keys))
title_cell = ws1.cell(row=1, column=1, value="2008–2024 Өрхийн орлогын нэгдсэн мэдээлэл")
title_cell.font      = Font(name="Arial", bold=True, size=13, color="1F4E79")
title_cell.alignment = CENTER
ws1.row_dimensions[1].height = 28

# Header row
for ci, (label, w) in enumerate(zip(labels, widths), start=1):
    cell = ws1.cell(row=2, column=ci, value=label)
    style_header(cell)
    ws1.column_dimensions[get_column_letter(ci)].width = w
ws1.row_dimensions[2].height = 36

# Data rows
num_fmt_int  = "#,##0"
num_fmt_id   = "0"

for ri, row in enumerate(alldata.itertuples(index=False), start=3):
    alt = (ri % 2 == 0)
    for ci, key in enumerate(keys, start=1):
        val  = getattr(row, key, None)
        cell = ws1.cell(row=ri, column=ci, value=val if pd.notna(val) else None)
        if key == "year":
            cell.font      = BOLD_FONT
            cell.fill      = ALT_FILL if alt else WHITE_FILL
            cell.alignment = CENTER
            cell.border    = BORDER
            cell.number_format = "0"
        elif key == ID:
            style_body(cell, align=CENTER, alt=alt)
            cell.number_format = num_fmt_id
        elif key in COMP_COLS:
            style_body(cell, align=CENTER, alt=alt)
            cell.number_format = num_fmt_int
        else:
            style_body(cell, align=RIGHT, alt=alt)
            cell.number_format = num_fmt_int

# Freeze panes
ws1.freeze_panes = "A3"

print(f"  Sheet 1: {len(alldata):,} мөр бичлаа")

# ════════════════════════════════════════════════════════════════
# SHEET 2 — Weighted averages
# ════════════════════════════════════════════════════════════════
ws2 = wb.create_sheet("Жинлэсэн дундаж")

# Detect columns from avg_all
avg_cols = list(avg_all.columns)  # year, Орлогын төрөл, Жингүй дундаж, Жинлэсэн дундаж, Сарын дундаж (жинлэсэн)

col_widths2 = {
    "year":                       8,
    "Орлогын төрөл":             32,
    "Жингүй дундаж":             20,
    "Жинлэсэн дундаж":           20,
    "Сарын дундаж (жинлэсэн)":   24,
}

display_labels2 = {
    "year":                       "Он",
    "Орлогын төрөл":             "Орлогын төрөл",
    "Жингүй дундаж":             "Жингүй дундаж (жилийн)",
    "Жинлэсэн дундаж":           "Жинлэсэн дундаж (жилийн)",
    "Сарын дундаж (жинлэсэн)":   "Сарын дундаж (жинлэсэн)",
}

# Title
ws2.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(avg_cols))
t2 = ws2.cell(row=1, column=1, value="2008–2024 Жинлэсэн дундаж орлогын үзүүлэлт")
t2.font      = Font(name="Arial", bold=True, size=13, color="1F4E79")
t2.alignment = CENTER
ws2.row_dimensions[1].height = 28

# Header
for ci, col in enumerate(avg_cols, start=1):
    label = display_labels2.get(col, col)
    cell  = ws2.cell(row=2, column=ci, value=label)
    style_header(cell)
    ws2.column_dimensions[get_column_letter(ci)].width = col_widths2.get(col, 18)
ws2.row_dimensions[2].height = 36

# Data
prev_year = None
year_alt  = False
for ri, row in enumerate(avg_all.itertuples(index=False), start=3):
    yr = getattr(row, "year", None)
    if yr != prev_year:
        year_alt = not year_alt
        prev_year = yr
    fill = ALT_FILL if year_alt else WHITE_FILL

    for ci, col in enumerate(avg_cols, start=1):
        val  = getattr(row, col.replace(" ", "_").replace("(", "").replace(")", ""), None)
        # fallback: access by position
        try:
            val = row[avg_cols.index(col)]
        except Exception:
            val = None

        cell = ws2.cell(row=ri, column=ci, value=val if (val is not None and pd.notna(val)) else None)
        cell.border = BORDER
        cell.fill   = fill

        if col == "year":
            cell.font         = BOLD_FONT
            cell.alignment    = CENTER
            cell.number_format = "0"
        elif col == "Орлогын төрөл":
            cell.font         = BODY_FONT
            cell.alignment    = LEFT
        else:
            cell.font          = BODY_FONT
            cell.alignment     = RIGHT
            cell.number_format = "#,##0"

ws2.freeze_panes = "A3"

print(f"  Sheet 2: {len(avg_all):,} мөр бичлаа")

# ── Save ──────────────────────────────────────────────────────────
wb.save(OUT)
print(f"\n✓ Хадгалагдлаа: {OUT}")

Sheet 1: өрхийн түвшний мэдээлэл уншиж байна...
  2008 WARNING: columns not found: ['total_members']
  2008: 11,172 өрх уншлаа
  2010 WARNING: columns not found: ['children', 'leading_adult', 'other_adults', 'total_members']
  2010: 11,195 өрх уншлаа
  2012: 12,809 өрх уншлаа
  2014: 16,172 өрх уншлаа
  2016: 16,446 өрх уншлаа
  2018: 16,451 өрх уншлаа
  2020: 16,458 өрх уншлаа
  2022: 22,992 өрх уншлаа
  2024: 15,511 өрх уншлаа

Нийт өрх: 139,206

Sheet 2: жинлэсэн дундаж уншиж байна...
  2008 SKIP: No avg2008.xlsx/.csv found in C:\Users\b22fa\Desktop\diplom unelgee\2008
  2010: avg файл уншлаа
  2012: avg файл уншлаа
  2014: avg файл уншлаа
  2016: avg файл уншлаа
  2018: avg файл уншлаа
  2020: avg файл уншлаа
  2022: avg файл уншлаа
  2024: avg файл уншлаа

Excel файл үүсгэж байна...
  Sheet 1: 139,206 мөр бичлаа
  Sheet 2: 40 мөр бичлаа

✓ Хадгалагдлаа: C:\Users\b22fa\Desktop\diplom unelgee\alldata.xlsx


In [78]:
import os
import glob
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Configuration ─────────────────────────────────────────────────────────────
BASE   = r"C:\Users\b22fa\Desktop\diplom unelgee"
YEARS  = [2008, 2010, 2012, 2014, 2016, 2018, 2020, 2022, 2024]
ID     = "identif"

INCOME_COLS = ["tsalin", "tetgever", "uildverlelal", "beleg_tuslamj", "niit_orlogo"]
COMP_COLS   = ["children", "leading_adult", "other_adults", "total_members", "hhweight"]

# ── Helpers ───────────────────────────────────────────────────────────────────
def folder(year):
    return os.path.join(BASE, str(year))

def find_data_file(year):
    """Find data{year}.xlsx or data{year}.csv in the year folder."""
    d = folder(year)
    for ext in ["xlsx", "csv"]:
        pattern = os.path.join(d, f"data{year}.{ext}")
        matches = glob.glob(pattern)
        if matches:
            return matches[0], ext
    raise FileNotFoundError(f"No data{year}.xlsx/.csv found in {d}")

def find_avg_file(year):
    """Find avg{year}.xlsx or avg{year}.csv in the year folder."""
    d = folder(year)
    for ext in ["xlsx", "csv"]:
        pattern = os.path.join(d, f"avg{year}.{ext}")
        matches = glob.glob(pattern)
        if matches:
            return matches[0], ext
    raise FileNotFoundError(f"No avg{year}.xlsx/.csv found in {d}")

def read_file(path, ext):
    if ext == "csv":
        return pd.read_csv(path, encoding="utf-8-sig")
    return pd.read_excel(path)

# ── Sheet 1: All household-level data ─────────────────────────────────────────
print("Sheet 1: өрхийн түвшний мэдээлэл уншиж байна...")

all_frames = []
for year in YEARS:
    try:
        path, ext = find_data_file(year)
        df = read_file(path, ext)
        df.columns = df.columns.str.lower()

        keep = [ID] + COMP_COLS + INCOME_COLS
        existing = [c for c in keep if c in df.columns]
        missing  = [c for c in keep if c not in df.columns]
        if missing:
            print(f"  {year} WARNING: columns not found: {missing}")
            for c in missing:
                df[c] = np.nan

        df = df[keep].copy()
        df.insert(0, "year", year)
        all_frames.append(df)
        print(f"  {year}: {len(df):,} өрх уншлаа")
    except FileNotFoundError as e:
        print(f"  {year} SKIP: {e}")

alldata = pd.concat(all_frames, ignore_index=True)
print(f"\nНийт өрх: {len(alldata):,}")

# ── Sheet 2: Weighted averages ─────────────────────────────────────────────────
print("\nSheet 2: жинлэсэн дундаж уншиж байна...")

avg_frames = []
for year in YEARS:
    try:
        path, ext = find_avg_file(year)
        df = read_file(path, ext)
        df.insert(0, "year", year)
        avg_frames.append(df)
        print(f"  {year}: avg файл уншлаа")
    except FileNotFoundError as e:
        print(f"  {year} SKIP: {e}")

avg_all = pd.concat(avg_frames, ignore_index=True)

# Ensure monthly column exists (annual / 12)
if "Сарын дундаж (жинлэсэн)" not in avg_all.columns and "Жинлэсэн дундаж" in avg_all.columns:
    avg_all["Сарын дундаж (жинлэсэн)"] = avg_all["Жинлэсэн дундаж"] / 12

# ── Build Excel ────────────────────────────────────────────────────────────────
print("\nExcel файл үүсгэж байна...")

OUT = os.path.join(BASE, "alldata.xlsx")

# Style helpers
HEADER_FILL   = PatternFill("solid", fgColor="1F4E79")
SUBHDR_FILL   = PatternFill("solid", fgColor="2E75B6")
ALT_FILL      = PatternFill("solid", fgColor="D6E4F0")
WHITE_FILL    = PatternFill("solid", fgColor="FFFFFF")
HDR_FONT      = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT     = Font(name="Arial", size=10)
BOLD_FONT     = Font(name="Arial", bold=True, size=10)
CENTER        = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT          = Alignment(horizontal="left",   vertical="center")
RIGHT         = Alignment(horizontal="right",  vertical="center")

thin = Side(style="thin", color="BFBFBF")
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)

def style_header(cell, fill=HEADER_FILL):
    cell.font      = HDR_FONT
    cell.fill      = fill
    cell.alignment = CENTER
    cell.border    = BORDER

def style_body(cell, align=RIGHT, alt=False):
    cell.font      = BODY_FONT
    cell.fill      = ALT_FILL if alt else WHITE_FILL
    cell.alignment = align
    cell.border    = BORDER

wb = Workbook()

# ════════════════════════════════════════════════════════════════
# SHEET 1
# ════════════════════════════════════════════════════════════════
ws1 = wb.active
ws1.title = "Өрхийн мэдээлэл"

col_defs = [
    ("year",          "Он",                          10),
    (ID,              "Өрхийн дугаар",                16),
    ("children",      "Хүүхэд (14-с доош)",           16),
    ("leading_adult", "Тэргүүлэх насанд хүрсэн",      20),
    ("other_adults",  "Бусад насанд хүрсэн",           18),
    ("total_members", "Нийт гишүүд",                   14),
    ("hhweight",      "Өрхийн жин (hhweight)",         20),
    ("tsalin",        "Цалин хөлс",                   18),
    ("tetgever",      "Тэтгэвэр тэтгэмж бусад",        22),
    ("uildverlelal",  "Үйлдвэрлэл үйлчилгээ",          22),
    ("beleg_tuslamj", "Бэлэг тусламж өөрийн аж ахуй",  26),
    ("niit_orlogo",   "Нийт орлого",                   16),
]

keys   = [c[0] for c in col_defs]
labels = [c[1] for c in col_defs]
widths = [c[2] for c in col_defs]

# Title row
ws1.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(keys))
title_cell = ws1.cell(row=1, column=1, value="2008–2024 Өрхийн орлогын нэгдсэн мэдээлэл")
title_cell.font      = Font(name="Arial", bold=True, size=13, color="1F4E79")
title_cell.alignment = CENTER
ws1.row_dimensions[1].height = 28

# Header row
for ci, (label, w) in enumerate(zip(labels, widths), start=1):
    cell = ws1.cell(row=2, column=ci, value=label)
    style_header(cell)
    ws1.column_dimensions[get_column_letter(ci)].width = w
ws1.row_dimensions[2].height = 36

# Data rows
num_fmt_int  = "#,##0"
num_fmt_id   = "0"
num_fmt_wt   = "#,##0.0000"

for ri, row in enumerate(alldata.itertuples(index=False), start=3):
    alt = (ri % 2 == 0)
    for ci, key in enumerate(keys, start=1):
        val  = getattr(row, key, None)
        cell = ws1.cell(row=ri, column=ci, value=val if pd.notna(val) else None)

        if key == "year":
            cell.font          = BOLD_FONT
            cell.fill          = ALT_FILL if alt else WHITE_FILL
            cell.alignment     = CENTER
            cell.border        = BORDER
            cell.number_format = "0"

        elif key == ID:
            style_body(cell, align=CENTER, alt=alt)
            cell.number_format = num_fmt_id

        elif key == "hhweight":
            # hhweight: жингийн утга — бутархай тоо байж болно
            style_body(cell, align=RIGHT, alt=alt)
            cell.number_format = num_fmt_wt

        elif key in COMP_COLS:
            # children, leading_adult, other_adults, total_members
            style_body(cell, align=CENTER, alt=alt)
            cell.number_format = num_fmt_int

        else:
            # орлогын баганууд
            style_body(cell, align=RIGHT, alt=alt)
            cell.number_format = num_fmt_int

# Freeze panes
ws1.freeze_panes = "A3"

print(f"  Sheet 1: {len(alldata):,} мөр бичлаа")

# ════════════════════════════════════════════════════════════════
# SHEET 2 — Weighted averages
# ════════════════════════════════════════════════════════════════
ws2 = wb.create_sheet("Жинлэсэн дундаж")

# Detect columns from avg_all
avg_cols = list(avg_all.columns)  # year, Орлогын төрөл, Жингүй дундаж, Жинлэсэн дундаж, Сарын дундаж (жинлэсэн)

col_widths2 = {
    "year":                       8,
    "Орлогын төрөл":             32,
    "Жингүй дундаж":             20,
    "Жинлэсэн дундаж":           20,
    "Сарын дундаж (жинлэсэн)":   24,
}

display_labels2 = {
    "year":                       "Он",
    "Орлогын төрөл":             "Орлогын төрөл",
    "Жингүй дундаж":             "Жингүй дундаж (жилийн)",
    "Жинлэсэн дундаж":           "Жинлэсэн дундаж (жилийн)",
    "Сарын дундаж (жинлэсэн)":   "Сарын дундаж (жинлэсэн)",
}

# Title
ws2.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(avg_cols))
t2 = ws2.cell(row=1, column=1, value="2008–2024 Жинлэсэн дундаж орлогын үзүүлэлт")
t2.font      = Font(name="Arial", bold=True, size=13, color="1F4E79")
t2.alignment = CENTER
ws2.row_dimensions[1].height = 28

# Header
for ci, col in enumerate(avg_cols, start=1):
    label = display_labels2.get(col, col)
    cell  = ws2.cell(row=2, column=ci, value=label)
    style_header(cell)
    ws2.column_dimensions[get_column_letter(ci)].width = col_widths2.get(col, 18)
ws2.row_dimensions[2].height = 36

# Data
prev_year = None
year_alt  = False
for ri, row in enumerate(avg_all.itertuples(index=False), start=3):
    yr = getattr(row, "year", None)
    if yr != prev_year:
        year_alt = not year_alt
        prev_year = yr
    fill = ALT_FILL if year_alt else WHITE_FILL

    for ci, col in enumerate(avg_cols, start=1):
        try:
            val = row[avg_cols.index(col)]
        except Exception:
            val = None

        cell = ws2.cell(row=ri, column=ci, value=val if (val is not None and pd.notna(val)) else None)
        cell.border = BORDER
        cell.fill   = fill

        if col == "year":
            cell.font          = BOLD_FONT
            cell.alignment     = CENTER
            cell.number_format = "0"
        elif col == "Орлогын төрөл":
            cell.font      = BODY_FONT
            cell.alignment = LEFT
        else:
            cell.font          = BODY_FONT
            cell.alignment     = RIGHT
            cell.number_format = "#,##0"

ws2.freeze_panes = "A3"

print(f"  Sheet 2: {len(avg_all):,} мөр бичлаа")

# ── Save ──────────────────────────────────────────────────────────
wb.save(OUT)
print(f"\n✓ Хадгалагдлаа: {OUT}")

Sheet 1: өрхийн түвшний мэдээлэл уншиж байна...
  2008: 11,172 өрх уншлаа
  2010: 11,195 өрх уншлаа
  2012: 12,809 өрх уншлаа
  2014: 16,172 өрх уншлаа
  2016: 16,446 өрх уншлаа
  2018: 16,451 өрх уншлаа
  2020: 16,458 өрх уншлаа
  2022: 22,992 өрх уншлаа
  2024: 15,511 өрх уншлаа

Нийт өрх: 139,206

Sheet 2: жинлэсэн дундаж уншиж байна...
  2008 SKIP: No avg2008.xlsx/.csv found in C:\Users\b22fa\Desktop\diplom unelgee\2008
  2010: avg файл уншлаа
  2012: avg файл уншлаа
  2014: avg файл уншлаа
  2016: avg файл уншлаа
  2018: avg файл уншлаа
  2020: avg файл уншлаа
  2022: avg файл уншлаа
  2024: avg файл уншлаа

Excel файл үүсгэж байна...
  Sheet 1: 139,206 мөр бичлаа
  Sheet 2: 40 мөр бичлаа

✓ Хадгалагдлаа: C:\Users\b22fa\Desktop\diplom unelgee\alldata.xlsx


In [3]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── CPI өгөгдөл ───────────────────────────────────────────────
cpi_data = {
    "2008-01":45.3,"2008-02":46.5,"2008-03":48.1,"2008-04":50.8,"2008-05":53.0,"2008-06":53.1,
    "2008-07":53.1,"2008-08":55.1,"2008-09":55.3,"2008-10":54.7,"2008-11":54.2,"2008-12":54.2,
    "2010-01":57.8,"2010-02":59.5,"2010-03":60.7,"2010-04":62.0,"2010-05":63.9,"2010-06":62.9,
    "2010-07":61.2,"2010-08":61.6,"2010-09":61.2,"2010-10":61.4,"2010-11":62.3,"2010-12":63.8,
    "2012-01":71.4,"2012-02":73.2,"2012-03":74.8,"2012-04":75.2,"2012-05":75.9,"2012-06":76.3,
    "2012-07":76.7,"2012-08":77.2,"2012-09":78.0,"2012-10":78.5,"2012-11":78.6,"2012-12":79.3,
    "2014-01":90.3,"2014-02":91.1,"2014-03":91.9,"2014-04":92.8,"2014-05":94.2,"2014-06":94.7,
    "2014-07":95.1,"2014-08":95.5,"2014-09":96.2,"2014-10":97.0,"2014-11":97.7,"2014-12":98.5,
    "2016-01":99.7,"2016-02":100.1,"2016-03":100.6,"2016-04":101.7,"2016-05":101.8,"2016-06":101.5,
    "2016-07":101.6,"2016-08":100.9,"2016-09":100.3,"2016-10":99.7,"2016-11":100.2,"2016-12":100.8,
    "2018-01":108.7,"2018-02":109.6,"2018-03":110.7,"2018-04":111.4,"2018-05":112.0,"2018-06":112.6,
    "2018-07":113.2,"2018-08":112.2,"2018-09":112.2,"2018-10":113.2,"2018-11":115.3,"2018-12":116.0,
    "2020-01":123.2,"2020-02":124.6,"2020-03":125.5,"2020-04":124.8,"2020-05":124.9,"2020-06":125.4,
    "2020-07":126.0,"2020-08":124.8,"2020-09":124.3,"2020-10":124.8,"2020-11":125.5,"2020-12":124.8,
    "2022-01":144.5,"2022-02":146.0,"2022-03":147.2,"2022-04":150.7,"2022-05":152.8,"2022-06":155.1,
    "2022-07":156.5,"2022-08":155.0142405,"2022-09":154.7666139,"2022-10":156.6238133,
    "2022-11":158.4810127,"2022-12":159.9667722,
    "2024-01":174.0814873,"2024-02":175.3196203,"2024-03":176.929193,"2024-04":178.2911392,
    "2024-05":179.5292722,"2024-06":180.0245253,"2024-07":180.1483386,"2024-08":181.6340981,
    "2024-09":181.7579114,"2024-10":182.7484177,"2024-11":185.84375,"2024-12":188.1962025,
}

# Жилийн дундаж CPI
annual_cpi = {}
for year in [2008,2010,2012,2014,2016,2018,2020,2022,2024]:
    vals = [v for k,v in cpi_data.items() if k.startswith(str(year))]
    annual_cpi[year] = np.mean(vals)

print("Жилийн дундаж CPI (2015=100):")
for y,c in annual_cpi.items():
    print(f"  {y}: {c:.4f}")

# ── Excel файл нээх ───────────────────────────────────────────
PATH = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata.xlsx"
wb = load_workbook(PATH)
ws = wb["Өрхийн мэдээлэл"]

# Styles
HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
ALT_FILL    = PatternFill("solid", fgColor="D6E4F0")
WHITE_FILL  = PatternFill("solid", fgColor="FFFFFF")
HDR_FONT    = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT   = Font(name="Arial", size=10)
CENTER      = Alignment(horizontal="center", vertical="center", wrap_text=True)
RIGHT       = Alignment(horizontal="right",  vertical="center")
thin        = Side(style="thin", color="BFBFBF")
BORDER      = Border(left=thin, right=thin, top=thin, bottom=thin)

# Хамгийн сүүлийн баганыг олох — "Нийт орлого" байгаа газар
header_row = 2
last_col = ws.max_column
new_col  = last_col + 1

# Header-ийн row 2-т "Нийт орлого" баганын дугаарыг олох
niit_col = None
for col in range(1, last_col + 1):
    val = ws.cell(row=header_row, column=col).value
    if val and "Нийт орлого" in str(val):
        niit_col = col
        break

if niit_col is None:
    raise ValueError("'Нийт орлого' багана олдсонгүй!")

new_col = niit_col + 1

# Одоо байгаа баганыг баруун тийш нэг нүдээр нэмж ("insert column" биш, зүгээр нэмнэ)
# Шинэ баганын header
title_row = 1
# Title merge-г шинэ баганыг хамруулахаар шинэчлэх
# (Merge-ийг устгаад дахин нэмнэ)
# Эхлээд merge-уудыг шалгах
merges_to_remove = []
for merge in ws.merged_cells.ranges:
    if merge.min_row == title_row and merge.max_row == title_row:
        merges_to_remove.append(str(merge))

for m in merges_to_remove:
    ws.merged_cells.remove(m)

# Title row-г шинэ багана хүртэл merge хийх
ws.merge_cells(start_row=title_row, start_column=1, end_row=title_row, end_column=new_col)
title_cell = ws.cell(row=title_row, column=1)
title_cell.font      = Font(name="Arial", bold=True, size=13, color="1F4E79")
title_cell.alignment = CENTER

# Header нэмэх
hdr_cell = ws.cell(row=header_row, column=new_col, value="Бодит орлого\n(2015 үнэ)")
hdr_cell.font      = HDR_FONT
hdr_cell.fill      = HEADER_FILL
hdr_cell.alignment = CENTER
hdr_cell.border    = BORDER
ws.column_dimensions[get_column_letter(new_col)].width = 18
ws.row_dimensions[header_row].height = 36

# "year" баганыг олох
year_col = None
for col in range(1, last_col + 1):
    val = ws.cell(row=header_row, column=col).value
    if val and "Он" in str(val):
        year_col = col
        break

print(f"\nНийт орлого багана: {niit_col}, Он багана: {year_col}, Шинэ багана: {new_col}")

# Өгөгдлийн мөрүүд
max_row = ws.max_row
added = 0
for ri in range(3, max_row + 1):
    year_val  = ws.cell(row=ri, column=year_col).value
    niit_val  = ws.cell(row=ri, column=niit_col).value

    real_val = None
    if year_val and niit_val is not None:
        try:
            yr  = int(year_val)
            cpi = annual_cpi.get(yr)
            if cpi:
                real_val = round(float(niit_val) / (cpi / 100))
        except:
            pass

    # Alternate fill — year багананы fill-тэй нийцүүлэх
    ref_fill = ws.cell(row=ri, column=niit_col).fill
    is_alt   = ref_fill.fgColor.rgb == "FFD6E4F0"

    cell = ws.cell(row=ri, column=new_col, value=real_val)
    cell.font          = BODY_FONT
    cell.fill          = ALT_FILL if is_alt else WHITE_FILL
    cell.alignment     = RIGHT
    cell.border        = BORDER
    cell.number_format = "#,##0"
    if real_val is not None:
        added += 1

print(f"Бодит орлого тооцсон мөр: {added:,}")

wb.save(PATH)
print(f"\n✓ Хадгалагдлаа: {PATH}")

Жилийн дундаж CPI (2015=100):
  2008: 51.9500
  2010: 61.5250
  2012: 76.2583
  2014: 94.5833
  2016: 100.7417
  2018: 112.2583
  2020: 124.8833
  2022: 153.1377
  2024: 180.3753

Нийт орлого багана: 12, Он багана: 1, Шинэ багана: 13
Бодит орлого тооцсон мөр: 139,206

✓ Хадгалагдлаа: C:\Users\b22fa\Desktop\diplom unelgee\alldata.xlsx


In [1]:
"""
Монгол Улсын Дундаж Давхаргын Тооцоо (2008-2024, тэгш жилүүд)
Middle Class Calculator for Mongolia - Household Survey Data

Тооцоох аргууд:
1. Медиан орлогын 75-125%  (Pressman, 2007)
2. Медиан орлогын 67-200%  (Pew Research Center, 2015)
3. Тэнцвэржүүлсэн орлогын медианы 75-200%  (OECD, 2019)
4. PPP: $2-13/өдөр   (2005 PPP) – Ravallion (2010)
5. PPP: $10-50/өдөр  (2005 PPP) – Birdsall (2012)
6. PPP: $10-100/өдөр (2005 PPP) – Kharas & Gertz (2010)
7. PPP: $2-20/өдөр   (2005 PPP) – ADB (2011)

Тэнцвэржүүлсэн орлого: Y / (тэргүүлэх*1.0 + бусад*0.5 + хүүхэд*0.3)

ЧУХАЛ ТЭМДЭГЛЭЛ:
- Медиан тооцооллыг hhweight (өрхийн жин)-ээр жинлэн тооцно.
  Учир нь өрхийн судалгаанд нэг өрх нь олон өрхийг төлөөлдөг.
  Жинлэсэн медиан = популяцийн жинхэнэ медиан.
- Орлого нь жилийн нийт орлого (annual). PPP аргад /members/PPP/365
  хийж өдрийн нэг хүнд ногдох PPP $ гаргана – ЭНЭ ЗӨРЧИЛГҮЙ.
"""

import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# PPP conversion factors  (LCU per intl $)
# ─────────────────────────────────────────────
PPP_FACTORS = {
    2008: 472.4772573,  2009: 510.2920141,  2010: 552.4958324,
    2011: 580.6376343,  2012: 638.3347089,  2013: 728.0518498,
    2014: 809.4465983,  2015: 821.2808713,  2016: 834.1651398,
    2017: 896.1046143,  2018: 926.3812866,  2019: 957.2436523,
    2020: 932.6223145,  2021: 957.8392944,  2022: 1021.233956,
    2023: 1082.35365,   2024: 1116.504182,
}

FILE_PATH  = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata.xlsx"
EVEN_YEARS = list(range(2008, 2025, 2))

# ─────────────────────────────────────────────
# Weighted median helper
# ─────────────────────────────────────────────
def weighted_median(values, weights):
    """
    Жинлэсэн медиан тооцоолно.
    values, weights – pd.Series эсвэл array-like.
    NaN утгыг хасна.
    """
    vals = np.asarray(values, dtype=float)
    wts  = np.asarray(weights, dtype=float)
    mask = ~(np.isnan(vals) | np.isnan(wts))
    vals, wts = vals[mask], wts[mask]
    if len(vals) == 0:
        return np.nan
    # Sort
    order  = np.argsort(vals)
    vals   = vals[order]
    wts    = wts[order]
    cumw   = np.cumsum(wts)
    half   = cumw[-1] / 2.0
    # First index where cumulative weight >= half
    idx = np.searchsorted(cumw, half)
    idx = min(idx, len(vals) - 1)
    return vals[idx]

# ─────────────────────────────────────────────
# Style helpers
# ─────────────────────────────────────────────
def make_border():
    s = Side(style='thin')
    return Border(left=s, right=s, top=s, bottom=s)

def set_header(cell, text, bg="1F4E79", fg="FFFFFF", size=9, bold=True):
    cell.value = text
    cell.font  = Font(bold=bold, color=fg, size=size)
    cell.fill  = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal='center', vertical='center',
                               wrap_text=True)
    cell.border = make_border()

def set_data(cell, value, bg=None, bold=False, color="000000",
             h_align='center'):
    cell.value = value
    cell.font  = Font(bold=bold, color=color, size=9)
    cell.alignment = Alignment(horizontal=h_align, vertical='center')
    cell.border = make_border()
    if bg:
        cell.fill = PatternFill("solid", fgColor=bg)

# ─────────────────────────────────────────────
# READ DATA
# ─────────────────────────────────────────────
print("=" * 65)
print("Монгол Улсын Дундаж Давхаргын Тооцоо")
print("=" * 65)

df = pd.read_excel(FILE_PATH, sheet_name=0, header=1)
print(f"Нийт мөр: {len(df)},  Баганы тоо: {len(df.columns)}")

cols = df.columns.tolist()
print("Баганууд:", cols)

# Column mapping (matches screenshot column order)
year_col          = cols[0]   # A – Он
household_col     = cols[1]   # B – Өрхийн дугаар
children_col      = cols[2]   # C – Хүүхэд (14-с доош)
adults_main_col   = cols[3]   # D – Тэргүүлэх насанд хүрсэн  (weight 1.0)
adults_other_col  = cols[4]   # E – Бусад насанд хүрсэн       (weight 0.5)
total_members_col = cols[5]   # F – Нийт гишүүд
weight_col        = cols[6]   # G – Өрхийн жин (hhweight)  ← ЖИНЛЭЛТЭД ХЭРЭГЛЭНЭ
income_col        = cols[11]  # L – Нийт орлого (жилийн)

print(f"\nЖингийн багана  : {weight_col}")
print(f"Орлогын багана  : {income_col}")

# ─────────────────────────────────────────────
# Filter even years
# ─────────────────────────────────────────────
df_even = df[df[year_col].isin(EVEN_YEARS)].copy()
print(f"Тэгш жилүүдийн өрх: {len(df_even)}")

# ─────────────────────────────────────────────
# Income per-person calculations
# ─────────────────────────────────────────────
def per_capita(row):
    """Нэг хүнд ноогдох жилийн орлого"""
    m   = row[total_members_col]
    inc = row[income_col]
    if pd.isna(m) or m <= 0 or pd.isna(inc):
        return np.nan
    return inc / m

def equivalized(row):
    """
    Тэнцвэржүүлсэн орлого:
        Y_eq = Y / (тэргүүлэх*1.0 + бусад*0.5 + хүүхэд*0.3)
    """
    lead  = row[adults_main_col]  if not pd.isna(row[adults_main_col])  else 0
    other = row[adults_other_col] if not pd.isna(row[adults_other_col]) else 0
    child = row[children_col]     if not pd.isna(row[children_col])     else 0
    denom = lead * 1.0 + other * 0.5 + child * 0.3
    inc   = row[income_col]
    if denom <= 0 or pd.isna(inc):
        return np.nan
    return inc / denom

def daily_ppp(row):
    """
    Өдрийн нэг хүнд ногдох орлого PPP ам.доллараар:
        daily_PPP = (жилийн нийт орлого / гишүүд) / PPP_factor / 365
    income_col нь жилийн орлого → /365 зөв.
    """
    yr  = row[year_col]
    ppp = PPP_FACTORS.get(yr, np.nan)
    m   = row[total_members_col]
    inc = row[income_col]
    if pd.isna(ppp) or pd.isna(m) or m <= 0 or ppp <= 0 or pd.isna(inc):
        return np.nan
    return (inc / m) / ppp / 365

df_even['per_capita_income']    = df_even.apply(per_capita,  axis=1)
df_even['equivalized_income']   = df_even.apply(equivalized, axis=1)
df_even['daily_income_PPP_USD'] = df_even.apply(daily_ppp,   axis=1)

# ─────────────────────────────────────────────
# Weighted medians per year
# ─────────────────────────────────────────────
print("\nЖинлэсэн медиан тооцоолж байна (hhweight ашиглан)...")
year_medians    = {}   # per-capita weighted median
year_eq_medians = {}   # equivalized weighted median

for yr in EVEN_YEARS:
    sub = df_even[df_even[year_col] == yr].dropna(subset=[weight_col])
    if len(sub) == 0:
        continue
    w = sub[weight_col]

    wm_pc  = weighted_median(sub['per_capita_income'],  w)
    wm_eq  = weighted_median(sub['equivalized_income'], w)

    year_medians[yr]    = wm_pc
    year_eq_medians[yr] = wm_eq

    print(f"  {yr}: жинлэсэн медиан PC={wm_pc:>12,.0f}₮   EQ={wm_eq:>12,.0f}₮")

# ─────────────────────────────────────────────
# Classification helpers  →  -1=Бага  0=Дундаж  1=Өндөр
# ─────────────────────────────────────────────
def cls_median(val, median, lo, hi):
    if pd.isna(val) or median <= 0:
        return -1
    r = val / median
    if r < lo:  return -1
    if r <= hi: return  0
    return  1

def cls_ppp(dppp, lo, hi):
    if pd.isna(dppp): return -1
    if dppp < lo:     return -1
    if dppp <= hi:    return  0
    return  1

# ─────────────────────────────────────────────
# Apply all 7 methods
# ─────────────────────────────────────────────
print("\nАрга бүрийг тооцоолж байна...")

mc_keys = ['MC1_Pressman_75_125', 'MC2_Pew_67_200', 'MC3_OECD_75_200',
           'MC4_Ravallion_2_13',  'MC5_Birdsall_10_50',
           'MC6_Kharas_10_100',   'MC7_ADB_2_20']
buf = {k: [] for k in mc_keys}

for _, row in df_even.iterrows():
    yr  = row[year_col]
    pci = row['per_capita_income']
    eqi = row['equivalized_income']
    dpp = row['daily_income_PPP_USD']
    med = year_medians.get(yr, 0)
    meq = year_eq_medians.get(yr, 0)

    buf['MC1_Pressman_75_125'].append(cls_median(pci, med, 0.75, 1.25))
    buf['MC2_Pew_67_200'     ].append(cls_median(pci, med, 0.67, 2.00))
    buf['MC3_OECD_75_200'    ].append(cls_median(eqi, meq, 0.75, 2.00))
    buf['MC4_Ravallion_2_13' ].append(cls_ppp(dpp,  2,  13))
    buf['MC5_Birdsall_10_50' ].append(cls_ppp(dpp, 10,  50))
    buf['MC6_Kharas_10_100'  ].append(cls_ppp(dpp, 10, 100))
    buf['MC7_ADB_2_20'       ].append(cls_ppp(dpp,  2,  20))

for k in mc_keys:
    df_even[k]          = buf[k]
    df_even[k + '_bin'] = (df_even[k] == 0).astype(int)

# ─────────────────────────────────────────────
# Weighted percentage helper for summaries
# Uses hhweight so each row counts proportionally
# ─────────────────────────────────────────────
def wpct(sub, col, cls_val):
    """
    Жинлэсэн хувь – cls_val (-1/0/1) ангилалд орох өрхийн
    нийт жингийн хувь.
    """
    valid = sub.dropna(subset=[weight_col])
    total_w = valid[weight_col].sum()
    if total_w == 0:
        return 0.0
    match_w = valid.loc[valid[col] == cls_val, weight_col].sum()
    return match_w / total_w * 100

def wpct_bin(sub, col):
    """Binary version: хувь нь дундаж давхаргад орох (cls==0)"""
    return wpct(sub, col, 0)

# ─────────────────────────────────────────────
# OPEN WORKBOOK
# ─────────────────────────────────────────────
print("\nExcel файлд бичиж байна...")
wb = load_workbook(FILE_PATH)
ws = wb.worksheets[0]

max_col        = ws.max_column
data_row_start = 3   # row1=title, row2=headers, row3+=data

# ══════════════════════════════════════════════════════
# SHEET 1 – add computed columns
# ══════════════════════════════════════════════════════
sheet1_cols = [
    ('per_capita_income',       'Нэг хүнд\nорлого (₮)'),
    ('equivalized_income',      'Тэнцвэржүүлсэн\nорлого (₮)'),
    ('daily_income_PPP_USD',    'Өдрийн\nPPP ($)'),
    ('MC1_Pressman_75_125_bin', 'MC1\nPressman\n75-125%'),
    ('MC2_Pew_67_200_bin',      'MC2\nPew\n67-200%'),
    ('MC3_OECD_75_200_bin',     'MC3\nOECD\n75-200%'),
    ('MC4_Ravallion_2_13_bin',  'MC4\nRavallion\n$2-13'),
    ('MC5_Birdsall_10_50_bin',  'MC5\nBirdsall\n$10-50'),
    ('MC6_Kharas_10_100_bin',   'MC6\nKharas\n$10-100'),
    ('MC7_ADB_2_20_bin',        'MC7\nADB\n$2-20'),
]

col_start = max_col + 1
for i, (_, cname) in enumerate(sheet1_cols):
    c = ws.cell(row=2, column=col_start + i)
    set_header(c, cname)
    ws.column_dimensions[get_column_letter(col_start + i)].width = 13
ws.row_dimensions[2].height = 42

# Build fast lookup  key=(year, hh_num) → Series
lookup = {}
for _, row in df_even.iterrows():
    lookup[(row[year_col], row[household_col])] = row

FILL_CALC = "D6EAF8"
FILL_YES  = "C6EFCE";  FONT_YES = ("276221", True)
FILL_NO   = "FFCCCC";  FONT_NO  = ("9C0006", False)
bin_cols  = {k for k, _ in sheet1_cols if k.endswith('_bin')}

written = 0
for er in range(data_row_start, ws.max_row + 1):
    yv = ws.cell(row=er, column=1).value
    hv = ws.cell(row=er, column=2).value
    if yv is None:
        continue
    try:
        yr_int = int(yv)
    except (ValueError, TypeError):
        continue

    rd = lookup.get((yr_int, hv))
    for i, (col_key, _) in enumerate(sheet1_cols):
        c = ws.cell(row=er, column=col_start + i)
        if rd is None:
            c.value = ""
            continue
        val = rd[col_key]
        if col_key in bin_cols:
            v = int(val) if not pd.isna(val) else 0
            c.value = v
            c.alignment = Alignment(horizontal='center', vertical='center')
            c.border = make_border()
            fv, bv = (FONT_YES, FILL_YES) if v == 1 else (FONT_NO, FILL_NO)
            c.fill = PatternFill("solid", fgColor=bv)
            c.font = Font(bold=fv[1], color=fv[0], size=9)
        else:
            c.value = ("" if pd.isna(val) else
                       round(float(val), 4) if col_key == 'daily_income_PPP_USD'
                       else round(float(val), 0))
            c.fill      = PatternFill("solid", fgColor=FILL_CALC)
            c.alignment = Alignment(horizontal='center', vertical='center')
            c.border    = make_border()
            c.font      = Font(size=9)
    written += 1

print(f"Sheet1 бичигдсэн мөр: {written}")

# ══════════════════════════════════════════════════════
# SHEET 2 – overall summary (weighted %)
# ══════════════════════════════════════════════════════
S2 = "Дундаж Давхарга Дүн"
if S2 in wb.sheetnames:
    del wb[S2]
ws2 = wb.create_sheet(S2, index=1)

ws2.merge_cells('A1:L1')
t2 = ws2['A1']
t2.value = ("Монгол Улсын Дундаж Давхаргын Тооцооны Дүн – Аргуудын харьцуулалт"
            " (hhweight-ээр жинлэсэн %)")
t2.font  = Font(bold=True, size=11, color="FFFFFF")
t2.fill  = PatternFill("solid", fgColor="1B2631")
t2.alignment = Alignment(horizontal='center', vertical='center')
t2.border = make_border()
ws2.row_dimensions[1].height = 24

s2_hdrs = ['Он', 'Нийт өрх',
           'MC1\nPressman\n75-125%', 'MC2\nPew\n67-200%',
           'MC3\nOECD\n75-200%',
           'MC4\nRavallion\n$2-13', 'MC5\nBirdsall\n$10-50',
           'MC6\nKharas\n$10-100',  'MC7\nADB\n$2-20',
           'Жинлэсэн\nмедиан PC (₮)',
           'Жинлэсэн\nмедиан EQ (₮)',
           'Жинлэсэн\nмедиан PPP ($)']

for j, h in enumerate(s2_hdrs):
    c = ws2.cell(row=2, column=j + 1)
    set_header(c, h)
    ws2.column_dimensions[get_column_letter(j + 1)].width = 14
ws2.row_dimensions[2].height = 42

for ri, yr in enumerate(EVEN_YEARS):
    sub = df_even[df_even[year_col] == yr]
    if len(sub) == 0:
        continue
    bg  = "EBF5FB" if ri % 2 == 0 else "FDFEFE"
    er  = ri + 3
    w   = sub[weight_col]

    def wp(col): return f"{wpct_bin(sub, col):.1f}%"

    row_vals = [
        yr, len(sub),
        wp('MC1_Pressman_75_125'), wp('MC2_Pew_67_200'),
        wp('MC3_OECD_75_200'),
        wp('MC4_Ravallion_2_13'),  wp('MC5_Birdsall_10_50'),
        wp('MC6_Kharas_10_100'),   wp('MC7_ADB_2_20'),
        f"{weighted_median(sub['per_capita_income'],  w):,.0f}",
        f"{weighted_median(sub['equivalized_income'], w):,.0f}",
        f"{weighted_median(sub['daily_income_PPP_USD'], w):.3f}",
    ]
    for j, val in enumerate(row_vals):
        c = ws2.cell(row=er, column=j + 1)
        set_data(c, val, bg=bg, bold=(j == 0))
    ws2.row_dimensions[er].height = 16

# ══════════════════════════════════════════════════════
# SHEET 3 – Бага / Дундаж / Өндөр detail tables (weighted)
# ══════════════════════════════════════════════════════
S3 = "Давхаргын Дүн"
if S3 in wb.sheetnames:
    del wb[S3]
ws3 = wb.create_sheet(S3, index=2)

C_TITLE  = "1B2631"
C_METHOD = "1F4E79"
C_YEAR_H = "2E86C1"
C_LOW    = "FDEDEC"
C_MID    = "EAFAF1"
C_HIGH   = "EBF5FB"
C_TOTAL  = "F4F6F7"
C_TXT_LO = "922B21"
C_TXT_MI = "1E8449"
C_TXT_HI = "1A5276"
C_WHITE  = "FFFFFF"

years_present = [yr for yr in EVEN_YEARS
                 if yr in year_medians or yr in PPP_FACTORS]
n_yc = len(years_present)

# Page title
ws3.merge_cells(start_row=1, start_column=1,
                end_row=1,   end_column=n_yc + 1)
pt = ws3.cell(row=1, column=1)
pt.value = ("Монгол Улсын Дундаж Давхаргын Тооцооны Дүн  "
            "(2008–2024, тэгш жилүүд, hhweight-ээр жинлэсэн %)")
pt.font  = Font(bold=True, size=13, color=C_WHITE)
pt.fill  = PatternFill("solid", fgColor=C_TITLE)
pt.alignment = Alignment(horizontal='center', vertical='center')
pt.border = make_border()
ws3.row_dimensions[1].height = 28

METHOD_DEFS = [
    ('MC1_Pressman_75_125',
     "Хүснэгт 1. Медиан орлогын 75–125%-аар зааглагдсан орлогын бүлгүүд (Pressman, 2007)"),
    ('MC2_Pew_67_200',
     "Хүснэгт 2. Медиан орлогын 67–200%-аар зааглагдсан орлогын бүлгүүд (Pew Research Center, 2015)"),
    ('MC3_OECD_75_200',
     "Хүснэгт 3. Тэнцвэржүүлсэн орлогын медианы 75–200%-аар зааглагдсан орлогын бүлгүүд (OECD, 2019)"),
    ('MC4_Ravallion_2_13',
     "Хүснэгт 4. Үнэмлэхүй босго $2–13/өдөр (2005 PPP) – Ravallion (2010)"),
    ('MC5_Birdsall_10_50',
     "Хүснэгт 5. Үнэмлэхүй босго $10–50/өдөр (2005 PPP) – Birdsall (2012)"),
    ('MC6_Kharas_10_100',
     "Хүснэгт 6. Үнэмлэхүй босго $10–100/өдөр (2005 PPP) – Kharas & Gertz (2010)"),
    ('MC7_ADB_2_20',
     "Хүснэгт 7. Үнэмлэхүй босго $2–20/өдөр (2005 PPP) – ADB (2011)"),
]

def w_breakdown(yr, col):
    """
    Жинлэсэн хувь: Бага / Дундаж / Өндөр.
    Жин = hhweight.  Бүх жингийн нийлбэрээр хуваана.
    """
    sub = df_even[df_even[year_col] == yr].dropna(subset=[weight_col])
    if len(sub) == 0:
        return None
    lo  = wpct(sub, col, -1)
    mid = wpct(sub, col,  0)
    hi  = wpct(sub, col,  1)
    return lo, mid, hi

ROW_DEFS = [
    ("Бага",   C_LOW,   C_TXT_LO, False),
    ("Дундаж", C_MID,   C_TXT_MI, True ),
    ("Өндөр",  C_HIGH,  C_TXT_HI, False),
    ("Нийт",   C_TOTAL, "000000", False),
]

cur = 2   # current write row (row 1 = title)

for mc_col, title_text in METHOD_DEFS:
    # ── method title bar ──────────────────────
    ws3.merge_cells(start_row=cur, start_column=1,
                    end_row=cur,   end_column=n_yc + 1)
    mc = ws3.cell(row=cur, column=1)
    mc.value = title_text
    mc.font  = Font(bold=True, size=10, color=C_WHITE)
    mc.fill  = PatternFill("solid", fgColor=C_METHOD)
    mc.alignment = Alignment(horizontal='left', vertical='center',
                             wrap_text=True)
    mc.border = make_border()
    ws3.row_dimensions[cur].height = 22
    cur += 1

    # ── year header row ───────────────────────
    lh = ws3.cell(row=cur, column=1)
    lh.fill   = PatternFill("solid", fgColor=C_YEAR_H)
    lh.border = make_border()
    lh.value  = ""
    for ci, yr in enumerate(years_present):
        hc = ws3.cell(row=cur, column=ci + 2)
        hc.value = str(yr)
        hc.font  = Font(bold=True, color=C_WHITE, size=9)
        hc.fill  = PatternFill("solid", fgColor=C_YEAR_H)
        hc.alignment = Alignment(horizontal='center', vertical='center')
        hc.border = make_border()
    ws3.row_dimensions[cur].height = 18
    cur += 1

    # ── data rows ─────────────────────────────
    for label, bg, txt, bold in ROW_DEFS:
        lc = ws3.cell(row=cur, column=1)
        lc.value = label
        lc.font  = Font(bold=bold, color=txt, size=9)
        lc.fill  = PatternFill("solid", fgColor=bg)
        lc.alignment = Alignment(horizontal='left', vertical='center',
                                 indent=1)
        lc.border = make_border()

        for ci, yr in enumerate(years_present):
            dc = ws3.cell(row=cur, column=ci + 2)
            bd = w_breakdown(yr, mc_col)
            if bd is None:
                dc.value = "–"
            elif label == "Бага":
                dc.value = f"{bd[0]:.1f}%"
            elif label == "Дундаж":
                dc.value = f"{bd[1]:.1f}%"
            elif label == "Өндөр":
                dc.value = f"{bd[2]:.1f}%"
            else:
                dc.value = "100%"
            dc.font      = Font(bold=bold, color=txt, size=9)
            dc.fill      = PatternFill("solid", fgColor=bg)
            dc.alignment = Alignment(horizontal='center', vertical='center')
            dc.border    = make_border()
        ws3.row_dimensions[cur].height = 16
        cur += 1

    cur += 1   # blank gap between tables

# Column widths Sheet 3
ws3.column_dimensions['A'].width = 11
for ci in range(n_yc):
    ws3.column_dimensions[get_column_letter(ci + 2)].width = 9
ws3.freeze_panes = "B3"

# ─────────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────────
output_path = FILE_PATH.replace(".xlsx", "_middle_class.xlsx")
wb.save(output_path)
print(f"\n✅  Хадгалагдлаа: {output_path}")

# ─────────────────────────────────────────────
# CONSOLE SUMMARY  (weighted %)
# ─────────────────────────────────────────────
print("\n" + "=" * 72)
print(f"{'Он':>6}  {'MC1':>7} {'MC2':>7} {'MC3':>7} {'MC4':>7} "
      f"{'MC5':>7} {'MC6':>7} {'MC7':>7}")
print("-" * 72)
for yr in EVEN_YEARS:
    sub = df_even[df_even[year_col] == yr]
    if len(sub) == 0:
        continue
    print(f"{yr:>6}  "
          f"{wpct_bin(sub,'MC1_Pressman_75_125'):>6.1f}% "
          f"{wpct_bin(sub,'MC2_Pew_67_200'):>6.1f}% "
          f"{wpct_bin(sub,'MC3_OECD_75_200'):>6.1f}% "
          f"{wpct_bin(sub,'MC4_Ravallion_2_13'):>6.1f}% "
          f"{wpct_bin(sub,'MC5_Birdsall_10_50'):>6.1f}% "
          f"{wpct_bin(sub,'MC6_Kharas_10_100'):>6.1f}% "
          f"{wpct_bin(sub,'MC7_ADB_2_20'):>6.1f}%")
print("=" * 72)
print("Дундаж давхаргын hhweight-ээр жинлэсэн хувь (%)")
print("\n✅  Done!")

Монгол Улсын Дундаж Давхаргын Тооцоо
Нийт мөр: 139206,  Баганы тоо: 14
Баганууд: ['Он', 'Өрхийн дугаар', 'Хүүхэд (14-с доош)', 'Тэргүүлэх насанд хүрсэн', 'Бусад насанд хүрсэн', 'Нийт гишүүд', 'Өрхийн жин (hhweight)', 'Цалин хөлс', 'Тэтгэвэр тэтгэмж бусад', 'Үйлдвэрлэл үйлчилгээ', 'Бэлэг тусламж өөрийн аж ахуй', 'Нийт орлого', 'Нэрлэсэн орлого\n(2015 үнэ)', 'Бодит орлого\n(2015 үнэ, ₮)']

Жингийн багана  : Өрхийн жин (hhweight)
Орлогын багана  : Нийт орлого
Тэгш жилүүдийн өрх: 139206

Жинлэсэн медиан тооцоолж байна (hhweight ашиглан)...
  2008: жинлэсэн медиан PC=     582,000₮   EQ=     950,208₮
  2010: жинлэсэн медиан PC=     929,000₮   EQ=   1,493,333₮
  2012: жинлэсэн медиан PC=   1,854,300₮   EQ=   2,970,889₮
  2014: жинлэсэн медиан PC=   2,631,667₮   EQ=   4,148,333₮
  2016: жинлэсэн медиан PC=   2,591,000₮   EQ=   3,963,333₮
  2018: жинлэсэн медиан PC=   3,010,000₮   EQ=   4,666,667₮
  2020: жинлэсэн медиан PC=   4,055,000₮   EQ=   6,316,700₮
  2022: жинлэсэн медиан PC=   5,156,00

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import warnings, io
warnings.filterwarnings("ignore")

FILE_PATH  = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata.xlsx"
EVEN_YEARS = list(range(2008, 2025, 2))
BW_METHOD  = 0.15

YEAR_COLORS = {
    2008: "#E8A838", 2010: "#E05C5C", 2012: "#9B59B6",
    2014: "#27AE60", 2016: "#2980B9", 2018: "#1ABC9C",
    2020: "#F39C12", 2022: "#C0392B", 2024: "#2C3E50",
}

def gini_weighted_fast(values, weights):
    vals = np.asarray(values, dtype=float)
    wts  = np.asarray(weights, dtype=float)
    mask = (~np.isnan(vals)) & (~np.isnan(wts)) & (vals > 0) & (wts > 0)
    vals, wts = vals[mask], wts[mask]
    if len(vals) < 2:
        return np.nan
    order  = np.argsort(vals)
    vals, wts = vals[order], wts[order]
    wts_n  = wts / wts.sum()
    cumw   = np.cumsum(wts_n)
    cumwy  = np.cumsum(wts_n * vals)
    lorenz_y = cumwy / cumwy[-1]
    auc = np.trapezoid(lorenz_y, cumw)
    return 1 - 2 * auc

def weighted_median(values, weights):
    vals = np.asarray(values, dtype=float)
    wts  = np.asarray(weights, dtype=float)
    mask = (~np.isnan(vals)) & (~np.isnan(wts))
    vals, wts = vals[mask], wts[mask]
    order = np.argsort(vals)
    vals, wts = vals[order], wts[order]
    cumw = np.cumsum(wts)
    idx  = np.searchsorted(cumw, cumw[-1] / 2.0)
    return vals[min(idx, len(vals) - 1)]

def weighted_mean(values, weights):
    vals = np.asarray(values, dtype=float)
    wts  = np.asarray(weights, dtype=float)
    mask = (~np.isnan(vals)) & (~np.isnan(wts))
    vals, wts = vals[mask], wts[mask]
    return np.average(vals, weights=wts) if wts.sum() > 0 else np.nan

# ── READ ──────────────────────────────────────────────
print("Файл уншиж байна...")
df = pd.read_excel(FILE_PATH, sheet_name=0, header=1)
cols = df.columns.tolist()
print("Баганууд:", cols)

year_col     = cols[0]
weight_col   = cols[6]
real_inc_col = cols[13]

print(f"Он багана    : {year_col}")
print(f"Жин багана   : {weight_col}")
print(f"Орлого багана: {real_inc_col}")

df_even = df[df[year_col].isin(EVEN_YEARS)].copy()
df_even = df_even.dropna(subset=[real_inc_col, weight_col])
df_even = df_even[df_even[real_inc_col] > 0]

# ── 12-Д ХУВААЖ САРЫН БОЛГООД 1000-Д ХУВААЖ МЯН.ТӨГ БОЛГОНО ────────────────────────
df_even['inc'] = df_even[real_inc_col] / 12 / 1000

X_MAX = 4000.0
print(f"X_MAX = {X_MAX:.0f}")

# ── STATS ─────────────────────────────────────────────
print("\nТайлан статистик (сарын, мян.төг):")
stats = []
for yr in EVEN_YEARS:
    sub = df_even[df_even[year_col] == yr].dropna(subset=['inc', weight_col])
    if len(sub) < 10:
        continue
    vals = sub['inc'].values
    wts  = sub[weight_col].values
    mn   = weighted_mean(vals, wts)
    med  = weighted_median(vals, wts)
    gin  = gini_weighted_fast(vals, wts)
    stats.append({'year': yr, 'mean': mn, 'median': med, 'gini': gin, 'n': len(sub)})
    print(f"  {yr}: дундаж={mn:>10.3f}  медиан={med:>10.3f}  Жини={gin:.3f}  n={len(sub)}")

stats_df = pd.DataFrame(stats)

# ── FIGURE ────────────────────────────────────────────
fig = plt.figure(figsize=(16, 8))
fig.patch.set_facecolor('white')
ax = fig.add_axes([0.07, 0.18, 0.57, 0.72])

x_grid    = np.linspace(0, X_MAX, 3000)
all_peaks = []

for yr in EVEN_YEARS:
    sub = df_even[df_even[year_col] == yr].dropna(subset=['inc', weight_col])
    if len(sub) < 10:
        continue
    vals = sub['inc'].values
    wts  = sub[weight_col].values
    p995 = np.percentile(vals, 99.5)
    mask = vals <= p995
    vals_kde, wts_kde = vals[mask], wts[mask]

    try:
        kde     = gaussian_kde(vals_kde, weights=wts_kde / wts_kde.sum(),
                               bw_method=BW_METHOD)
        density = kde(x_grid)
    except Exception as e:
        print(f"  {yr} KDE алдаа: {e}")
        continue

    color = YEAR_COLORS.get(yr, "#555555")
    ax.fill_between(x_grid, density, alpha=0.22, color=color)
    ax.plot(x_grid, density, color=color, linewidth=2.0)

    peak_idx = np.argmax(density)
    all_peaks.append(density[peak_idx])
    px, py = x_grid[peak_idx], density[peak_idx]
    ax.annotate(str(yr), xy=(px, py),
                xytext=(px + X_MAX * 0.06, py),
                fontsize=10, color=color, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=color, lw=0.8))

# ── X AXIS TICKS ──────────────────────────────────────
step = 500
xticks = list(range(0, int(X_MAX) + 1, step))
ax.set_xticks(xticks)
ax.set_xticklabels([f"{x:,.0f}" for x in xticks], fontsize=8, rotation=45, ha='right')
ax.tick_params(axis='x', which='major', pad=4)

# ── GRIDLINE БАЙХГҮЙ ──────────────────────────────────
ax.grid(False)

ax.set_xlim(0, X_MAX)
ax.set_ylim(bottom=0, top=max(all_peaks) * 1.18 if all_peaks else None)
ax.set_xlabel(
    "Нэг хүнд ноогдох өрхийн сарын бодит орлого\n(мян.төг, 2015 оны үнээр зэрэгцүүлсэн)",
    fontsize=10, labelpad=8)
ax.set_ylabel("Тархалтын нягт", fontsize=10, labelpad=8)  # ← энэ мөрийг нэм
ax.set_title(
    "Зураг. Нэг хүнд ноогдох өрхийн сарын орлогын тархалтын нягтын муруйнууд\n"
    "2008–2024 оны тэгш жилүүд, 2015 оны үнээр зэрэгцүүлсэн",
    fontsize=11, fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

# ── RIGHT TABLE (САРЫН УТГААР, МЯН.ТӨГ) ────────────────────────
ax_t = fig.add_axes([0.68, 0.22, 0.30, 0.60])
ax_t.axis('off')

table_data = [["Он", "Дундаж\n(сар)", "Медиан\n(сар)", "Жини"]]
for _, row in stats_df.iterrows():
    table_data.append([
        str(int(row['year'])),
        f"{row['mean']:,.1f}",
        f"{row['median']:,.1f}",
        f"{row['gini']:.3f}"
    ])

tbl = ax_t.table(cellText=table_data[1:], colLabels=table_data[0],
                 cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)

for j in range(4):
    cell = tbl[0, j]
    cell.set_facecolor("#2C3E50")
    cell.set_text_props(color='white', fontweight='bold')
    cell.set_edgecolor('#AAAAAA')

for i, row_s in enumerate(stats_df.itertuples()):
    yr    = int(row_s.year)
    color = YEAR_COLORS.get(yr, "#555555")
    for j in range(4):
        cell = tbl[i + 1, j]
        cell.set_edgecolor('#CCCCCC')
        if j == 0:
            cell.set_facecolor(color)
            cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_facecolor("#F8F9F9" if i % 2 == 0 else "white")

for j, w in enumerate([0.16, 0.28, 0.28, 0.22]):
    for i in range(len(stats_df) + 1):
        tbl[i, j].set_width(w)

ax_t.set_title("Сарын дундаж, медиан, Жини\n(жинлэсэн, 2015 үнэ, мян.төг)",
               fontsize=9, fontweight='bold', pad=6)

# ── EXCEL-Д ХАДГАЛАХ ──────────────────────────────────
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage

buf = io.BytesIO()
fig.savefig(buf, format='png', dpi=150, bbox_inches='tight', facecolor='white')
buf.seek(0)
plt.close()

xl_path = FILE_PATH.replace(".xlsx", "_middle_class.xlsx")
try:
    wb = load_workbook(xl_path)
except FileNotFoundError:
    wb = load_workbook(FILE_PATH)

SHEET_NAME = "Тархалт"
if SHEET_NAME in wb.sheetnames:
    del wb[SHEET_NAME]
ws = wb.create_sheet(SHEET_NAME)

def brd():
    s = Side(style='thin')
    return Border(left=s, right=s, top=s, bottom=s)

ws.merge_cells('A1:E1')
tc = ws['A1']
tc.value = "Нэг хүнд ноогдох өрхийн сарын орлогын тархалт – Дундаж, Медиан, Жини (2008–2024, 2015 үнэ, мян.төг)"
tc.font  = Font(bold=True, size=11, color="FFFFFF")
tc.fill  = PatternFill("solid", fgColor="1B2631")
tc.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
tc.border = brd()
ws.row_dimensions[1].height = 36

for j, h in enumerate(["Он", "Өрхийн тоо", "Дундаж (сар, мян.₮)", "Медиан (сар, мян.₮)", "Жини"]):
    c = ws.cell(row=2, column=j + 1)
    c.value = h
    c.font  = Font(bold=True, color="FFFFFF", size=10)
    c.fill  = PatternFill("solid", fgColor="1F4E79")
    c.alignment = Alignment(horizontal='center', vertical='center')
    c.border = brd()
ws.row_dimensions[2].height = 20

for i, row_s in enumerate(stats_df.itertuples()):
    r  = i + 3
    bg = "EBF5FB" if i % 2 == 0 else "FDFEFE"
    for j, v in enumerate([int(row_s.year), int(row_s.n),
                            f"{row_s.mean:,.1f}", f"{row_s.median:,.1f}",
                            f"{row_s.gini:.3f}"]):
        c = ws.cell(row=r, column=j + 1)
        c.value = v
        c.font  = Font(bold=(j == 0), size=10)
        c.fill  = PatternFill("solid", fgColor=bg)
        c.alignment = Alignment(horizontal='center', vertical='center')
        c.border = brd()
    ws.row_dimensions[r].height = 18

note_row = len(stats_df) + 4
ws.merge_cells(f'A{note_row}:E{note_row}')
nc = ws[f'A{note_row}']
nc.value = ("Тайлбар: Нягтын муруйг жинлэсэн KDE аргаар тооцов. "
            "Жин болгон hhweight ашигласан. Жилийн бодит орлогыг 12-д хувааж, 1000-д хувааж мян.төг болгов. "
            "Бодит орлогыг 2015 оны үнээр зэрэгцүүлсэн. "
            "Дундаж, медианыг жинлэсэн аргаар, Жини коэффициентийг Лоренцын муруйд тулгуурлан тооцов.")
nc.font      = Font(italic=True, size=9, color="444444")
nc.alignment = Alignment(wrap_text=True, horizontal='left', vertical='top')
ws.row_dimensions[note_row].height = 40

for j, w in enumerate([10, 14, 20, 20, 12]):
    ws.column_dimensions[get_column_letter(j + 1)].width = w

try:
    img = XLImage(buf)
    img.anchor = 'G1'
    img.width  = 820
    img.height = 480
    ws.add_image(img)
    print("✅  График Excel-д нэмэгдлээ")
except Exception as e:
    print(f"Зураг нэмэхэд алдаа: {e}")

wb.save(xl_path)
print(f"✅  Excel хадгалагдлаа: {xl_path}")
print("✅  Бүгд дууслаа!")

Файл уншиж байна...
Баганууд: ['Он', 'Өрхийн дугаар', 'Хүүхэд (14-с доош)', 'Тэргүүлэх насанд хүрсэн', 'Бусад насанд хүрсэн', 'Нийт гишүүд', 'Өрхийн жин (hhweight)', 'Цалин хөлс', 'Тэтгэвэр тэтгэмж бусад', 'Үйлдвэрлэл үйлчилгээ', 'Бэлэг тусламж өөрийн аж ахуй', 'Нийт орлого', 'Нэрлэсэн орлого\n(2015 үнэ)', 'Бодит орлого\n(2015 үнэ, ₮)']
Он багана    : Он
Жин багана   : Өрхийн жин (hhweight)
Орлого багана: Бодит орлого
(2015 үнэ, ₮)
X_MAX = 4000

Тайлан статистик (сарын, мян.төг):
  2008: дундаж=   604.084  медиан=   383.702  Жини=0.533  n=10772
  2010: дундаж=   735.319  медиан=   451.984  Жини=0.547  n=10622
  2012: дундаж=  1051.294  медиан=   672.850  Жини=0.522  n=12751
  2014: дундаж=   957.095  медиан=   740.969  Жини=0.414  n=15990
  2016: дундаж=   823.777  медиан=   655.141  Жини=0.393  n=16305
  2018: дундаж=   910.922  медиан=   712.939  Жини=0.406  n=16153
  2020: дундаж=  1115.570  медиан=   861.137  Жини=0.407  n=16239
  2022: дундаж=  1105.856  медиан=   843.271  Жини=0.

In [ ]:
"""
2024 оны дундаж давхаргын тархалтын 4-панел график
Б. $10–$50   (Birdsall, 2012)
Г. $2–$20    (ADB, 2011)
Е. 67–200%   (Pew, 2015)
Ж. 75–200% тэнцвэржүүлсэн (OECD, 2019)
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings("ignore")

FILE_PATH   = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata_middle_class.xlsx"
TARGET_YEAR = 2024
OUT_PATH    = r"C:\Users\b22fa\Desktop\diplom unelgee\fig1_2024_4methods.png"

PPP_FACTORS = {
    2008:472.48, 2010:552.50, 2012:638.33, 2014:809.45,
    2016:834.17, 2018:926.38, 2020:932.62, 2022:1021.23, 2024:1116.50,
}

# ── ӨГӨГДӨЛ УНШИХ ─────────────────────────────────────────────
df_raw = pd.read_excel(FILE_PATH, sheet_name=0, header=1)
cols   = df_raw.columns.tolist()
year_col   = cols[0]
weight_col = cols[6]

def find_col(df, keys):
    for k in keys:
        if k in df.columns: return k
    hits = [c for c in df.columns if any(k.lower() in c.lower() for k in keys)]
    return hits[0] if hits else None

pc_col  = find_col(df_raw, ["per_capita_income"])
eq_col  = find_col(df_raw, ["equivalized_income"])
ppp_col = find_col(df_raw, ["daily_income_PPP_USD","daily_income"])

df = df_raw[df_raw[year_col] == TARGET_YEAR].copy().dropna(subset=[weight_col])
print(f"{TARGET_YEAR} он: {len(df):,} өрх")
if len(df) == 0:
    raise ValueError(f"{TARGET_YEAR} оны өгөгдөл олдсонгүй")

w = df[weight_col].values.astype(float)

# ── ОРЛОГЫН УТГУУД ─────────────────────────────────────────────
if pc_col and pc_col in df.columns:
    pc_vals = df[pc_col].values.astype(float)
else:
    pc_vals = (df[cols[11]] / df[cols[5]].replace(0, np.nan)).values.astype(float)

if eq_col and eq_col in df.columns:
    eq_vals = df[eq_col].values.astype(float)
else:
    denom = (df[cols[3]].fillna(0)*1.0 + df[cols[4]].fillna(0)*0.5 + df[cols[2]].fillna(0)*0.3)
    eq_vals = (df[cols[11]] / denom.replace(0, np.nan)).values.astype(float)

if ppp_col and ppp_col in df.columns:
    ppp_vals = df[ppp_col].values.astype(float)
else:
    ppp_f = PPP_FACTORS.get(TARGET_YEAR, 1116.5)
    ppp_vals = ((df[cols[11]] / df[cols[5]].replace(0, np.nan)) / ppp_f / 365).values.astype(float)

# ── ЖИНЛЭСЭН МЕДИАН ───────────────────────────────────────────
def wmedian(vals, wts):
    mask = ~(np.isnan(vals)|np.isnan(wts))
    v, w2 = vals[mask], wts[mask]
    if len(v) == 0: return np.nan
    idx = np.argsort(v); v, w2 = v[idx], w2[idx]
    cumw = np.cumsum(w2)
    i = np.searchsorted(cumw, cumw[-1]/2)
    return v[min(i, len(v)-1)]

med_pc = wmedian(pc_vals, w)
med_eq = wmedian(eq_vals, w)
print(f"Медиан PC: {med_pc:,.0f} ₮   Медиан EQ: {med_eq:,.0f} ₮")

def wpct(mask_bool, wts):
    v = ~np.isnan(wts)
    return wts[v & mask_bool].sum() / wts[v].sum() * 100

# ── 4 АРГА ────────────────────────────────────────────────────
METHODS = [
    dict(
        label="Б. Нэг хүнд ноогдох өдрийн орлого 10$–50$\n(2005 PPP, Birdsall, 2012)",
        vals=ppp_vals, lo=10.0, hi=50.0,
        bw=0.18, is_ppp=True,
        lo_lbl="$10", hi_lbl="$50",
    ),
    dict(
        label="Г. Нэг хүнд ноогдох өдрийн орлого 2$–20$\n(2005 PPP, ADB, 2011)",
        vals=ppp_vals, lo=2.0, hi=20.0,
        bw=0.18, is_ppp=True,
        lo_lbl="$2", hi_lbl="$20",
    ),
    dict(
        label="Е. Нэг хүнд ноогдох өрхийн орлогын\n67%–200% (Pew Research Center, 2015)",
        vals=pc_vals, lo=0.67*med_pc, hi=2.00*med_pc,
        bw=0.15, is_ppp=False,
        lo_lbl="67%", hi_lbl="200%", med=med_pc,
    ),
    dict(
        label="Ж. Тэнцвэржүүлсэн орлогын\n75%–200% (OECD, 2019)",
        vals=eq_vals, lo=0.75*med_eq, hi=2.00*med_eq,
        bw=0.15, is_ppp=False,
        lo_lbl="75%", hi_lbl="200%", med=med_eq,
    ),
]

# xmax автоматаар
for m in METHODS:
    v_pos = m["vals"]; v_pos = v_pos[v_pos > 0]
    p99 = float(np.nanpercentile(v_pos, 99.0)) if len(v_pos) else 100
    m["xmax"] = min(p99*1.1, m["hi"]*3.0, 250.0) if m["is_ppp"] else min(p99, m["hi"]*2.2)

# ── KDE ───────────────────────────────────────────────────────
def build_kde(vals, wts, xmax, bw=0.15, n=3000):
    mask = (~np.isnan(vals)) & (~np.isnan(wts)) & (vals > 0) & (vals < xmax*1.6)
    v, w2 = vals[mask], wts[mask]
    if len(v) < 10: return None, None
    kde = gaussian_kde(v, weights=w2/w2.sum(), bw_method=bw)
    x   = np.linspace(0, xmax, n)
    return x, kde(x)

# ── ЗУРАГ ─────────────────────────────────────────────────────
ORANGE = "#F5A623"
GRAY_L = "#CCCCCC"
GRAY_H = "#BBBBBB"
LINE_C = "#555555"

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor("white")
fig.suptitle(
    f"Зураг 1. {TARGET_YEAR} оны орлогын тархалтын нягтын муруй ба дундаж давхаргыг эзлэх хувь,\n"
    "орлогын тодорхойлолт тус бүрээр",
    fontsize=13, fontweight="bold", y=0.99, ha="center"
)

def draw_panel(ax, m):
    x, y = build_kde(m["vals"], w, m["xmax"], m["bw"])
    if x is None:
        ax.text(0.5, 0.5, "Өгөгдөл хангалтгүй",
                transform=ax.transAxes, ha="center")
        ax.set_title(m["label"], fontsize=10, fontweight="bold", pad=10)
        return  

    lo, hi, xmax = m["lo"], m["hi"], m["xmax"]

    ax.fill_between(x, y, where=(x < lo),          color=GRAY_L, alpha=0.55)
    ax.fill_between(x, y, where=(x>=lo)&(x<=hi),   color=ORANGE, alpha=0.85)
    ax.fill_between(x, y, where=(x > hi),           color=GRAY_H, alpha=0.38)
    ax.plot(x, y, color=LINE_C, lw=1.5, alpha=0.65)
    ax.axvline(lo, color="#888888", lw=0.9, ls="--", alpha=0.7)
    ax.axvline(hi, color="#888888", lw=0.9, ls="--", alpha=0.7)

    ymax = float(y.max())
    ax.set_ylim(0, ymax * 1.42)

    # Жинлэсэн хувь
    vals  = m["vals"]
    valid = ~np.isnan(vals)
    lo_p  = wpct(valid & (vals <  lo), w)
    mid_p = wpct(valid & (vals >= lo) & (vals <= hi), w)
    hi_p  = wpct(valid & (vals >  hi), w)

    # Хувийн тэмдэглэл — зургийн байрлалтай яг адил
    lo_x  = lo * 0.40
    mid_x = (lo + hi) / 2
    hi_x  = min(hi * 1.65, xmax * 0.87)

    for xp, pct, yf in [
        (lo_x,  lo_p,  0.55),
        (mid_x, mid_p, 0.34),
        (hi_x,  hi_p,  0.13),
    ]:
        ax.text(xp, ymax*yf, f"{pct:.1f}%",
                ha="center", fontsize=12, fontweight="bold", color="#222222",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", alpha=0.55, ec="none"))

    # X тэнхлэгийн босго тэмдэглэл
    ax.text(lo, -ymax*0.11, m["lo_lbl"], ha="center", va="top",
            fontsize=9, color="#444444", fontweight="bold", linespacing=1.3)
    ax.text(hi, -ymax*0.11, m["hi_lbl"], ha="center", va="top",
            fontsize=9, color="#444444", fontweight="bold")

    # Тэнхлэг
    ax.set_xlim(0, xmax)
    xlabel = ("Нэг хүнд ноогдох өдрийн орлого (2005 PPP ам.$)"
              if m["is_ppp"] else "Нэг хүнд ноогдох орлого (₮/жил)")
    ax.set_xlabel(xlabel, fontsize=9.5, labelpad=16)
    ax.set_ylabel("Тархалтын нягт", fontsize=9.5, labelpad=6)
    ax.yaxis.set_ticklabels([])
    ax.tick_params(axis="y", length=0)
    ax.spines["left"].set_color("#DDDDDD")
    ax.spines["bottom"].set_color("#CCCCCC")
    ax.grid(False)
    ax.set_title(m["label"], fontsize=11, fontweight="bold", pad=12)
    ax.set_facecolor("white")

    print(f"  {m['label'].split(chr(10))[0]}: Бага={lo_p:.1f}%  Дундаж={mid_p:.1f}%  Өндөр={hi_p:.1f}%")

print("\nПанел бүрийн хувь:")
for ax, m in zip(axes.flat, METHODS):
    draw_panel(ax, m)

plt.tight_layout(rect=[0, 0.0, 1, 0.965], h_pad=4.0, w_pad=3.0)
plt.savefig(OUT_PATH, dpi=200, bbox_inches="tight", facecolor="white")
plt.close()
print(f"\n✅  Зураг хадгалагдлаа: {OUT_PATH}")

2024 он: 15,511 өрх
Медиан PC: 7,558,412 ₮   Медиан EQ: 11,707,667 ₮

Панел бүрийн хувь:
  Б. Нэг хүнд ноогдох өдрийн орлого 10$–50$: Бага=21.1%  Дундаж=71.4%  Өндөр=7.5%
  Г. Нэг хүнд ноогдох өдрийн орлого 2$–20$: Бага=4.3%  Дундаж=50.9%  Өндөр=44.9%
  Е. Нэг хүнд ноогдох өрхийн орлогын: Бага=28.7%  Дундаж=57.6%  Өндөр=13.6%
  Ж. Тэнцвэржүүлсэн орлогын: Бага=33.4%  Дундаж=53.1%  Өндөр=13.6%

✅  Зураг хадгалагдлаа: C:\Users\b22fa\Desktop\diplom unelgee\fig1_2024_4methods.png


In [10]:
"""
Дундаж давхаргын тархалт — 6 арга, 2 багана × 3 мөр layout
────────────────────────────────────────────────────────────
Аргууд:
  [1] Ravallion (2010):        $2–$13   өдрийн PPP
  [2] Birdsall (2012):        $10–$50  өдрийн PPP
  [3] ADB (2011):              $2–$20   өдрийн PPP
  [4] Pressman (2007):         75%–125% медиан PC
  [5] Pew Research Center (2015): 67%–200% медиан PC
  [6] OECD (2019):             75%–200% медиан EQ
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings("ignore")
matplotlib.rcParams["text.usetex"] = False
matplotlib.rcParams["mathtext.default"] = "regular"

# ── ТОХИРГОО ──────────────────────────────────────────────────
FILE_PATH   = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata_middle_class.xlsx"
TARGET_YEAR = 2024
OUT_PATH    = r"C:\Users\b22fa\Desktop\diplom unelgee\fig_2x3_methods.png"

# PPP conversion factors (₮ → 1 ам.$, жилийн дундаж)
PPP_FACTORS = {
    2008: 472.48,  2010: 552.50,  2012: 638.33,  2014: 809.45,
    2016: 834.17,  2018: 926.38,  2020: 932.62,  2022: 1021.23,
    2024: 1116.50,
}

# ── ӨГӨГДӨЛ УНШИХ ─────────────────────────────────────────────
df_raw = pd.read_excel(FILE_PATH, sheet_name=0, header=1)
cols   = df_raw.columns.tolist()

year_col   = cols[0]   # жил
weight_col = cols[6]   # жин (өрхийн)

def find_col(df, keys):
    for k in keys:
        if k in df.columns:
            return k
    hits = [c for c in df.columns if any(k.lower() in c.lower() for k in keys)]
    return hits[0] if hits else None

pc_col  = find_col(df_raw, ["per_capita_income"])
eq_col  = find_col(df_raw, ["equivalized_income"])
ppp_col = find_col(df_raw, ["daily_income_PPP_USD", "daily_income"])

# Тухайн жилийн өгөгдөл
df = df_raw[df_raw[year_col] == TARGET_YEAR].copy().dropna(subset=[weight_col])
print(f"{TARGET_YEAR} он: {len(df):,} өрх")
if len(df) == 0:
    raise ValueError(f"{TARGET_YEAR} оны өгөгдөл олдсонгүй!")

w = df[weight_col].values.astype(float)

# ── ОРЛОГЫН УТГУУД ТООЦОХ ─────────────────────────────────────

# 1) Нэг хүнд ноогдох орлого (PC) — ₮/жил
if pc_col and pc_col in df.columns:
    pc_vals = df[pc_col].values.astype(float)
else:
    pc_vals = (df[cols[11]] / df[cols[5]].replace(0, np.nan)).values.astype(float)

# 2) Тэнцвэржүүлсэн орлого (EQ) — OECD equivalized scale
if eq_col and eq_col in df.columns:
    eq_vals = df[eq_col].values.astype(float)
else:
    denom = (
        df[cols[3]].fillna(0) * 1.0
        + df[cols[4]].fillna(0) * 0.5
        + df[cols[2]].fillna(0) * 0.3
    )
    eq_vals = (df[cols[11]] / denom.replace(0, np.nan)).values.astype(float)

# 3) Өдрийн PPP орлого (ам.$)
if ppp_col and ppp_col in df.columns:
    ppp_vals = df[ppp_col].values.astype(float)
else:
    ppp_f    = PPP_FACTORS.get(TARGET_YEAR, 1116.50)
    ppp_vals = (
        (df[cols[11]] / df[cols[5]].replace(0, np.nan))
        / ppp_f / 365
    ).values.astype(float)

# ── ЖИНЛЭСЭН МЕДИАН ───────────────────────────────────────────
def wmedian(vals, wts):
    mask = ~(np.isnan(vals) | np.isnan(wts))
    v, w2 = vals[mask], wts[mask]
    if len(v) == 0:
        return np.nan
    idx    = np.argsort(v)
    v, w2  = v[idx], w2[idx]
    cumw   = np.cumsum(w2)
    i      = np.searchsorted(cumw, cumw[-1] / 2)
    return v[min(i, len(v) - 1)]

med_pc = wmedian(pc_vals, w)
med_eq = wmedian(eq_vals, w)
print(f"  Медиан PC  : {med_pc:>12,.0f} ₮/жил")
print(f"  Медиан EQ  : {med_eq:>12,.0f} ₮/жил")

# ── ЖИНЛЭСЭН ХУВЬ ─────────────────────────────────────────────
def wpct(mask_bool, wts):
    valid = ~np.isnan(wts)
    denom = wts[valid].sum()
    if denom == 0:
        return 0.0
    return wts[valid & mask_bool].sum() / denom * 100.0

# ── 6 АРГЫН ТОДОРХОЙЛОЛТ ──────────────────────────────────────
METHODS = [
    dict(
        title="А. Нэг хүнд ноогдох өдрийн орлого 2\u202f–\u202f13 ам.доллар",
        vals=ppp_vals, lo=2.0,  hi=13.0,
        bw=0.20, is_ppp=True,
        lo_lbl="$2", hi_lbl="$13",
    ),
    dict(
        title="Б. Нэг хүнд ноогдох өдрийн орлого 10\u202f–\u202f50 ам.доллар",
        vals=ppp_vals, lo=10.0, hi=50.0,
        bw=0.20, is_ppp=True,
        lo_lbl="$10", hi_lbl="$50",
    ),
    dict(
        title="В. Нэг хүнд ноогдох өдрийн орлого 2\u202f–\u202f20 ам.доллар",
        vals=ppp_vals, lo=2.0,  hi=20.0,
        bw=0.20, is_ppp=True,
        lo_lbl="$2", hi_lbl="$20",
    ),
    dict(
        title="Г. Нэг хүнд ноогдох өрхийн медиан орлогын 75%–125%",
        vals=pc_vals, lo=0.75 * med_pc, hi=1.25 * med_pc,
        bw=0.16, is_ppp=False,
        lo_lbl="75%", hi_lbl="125%",
    ),
    dict(
        title="Д. Нэг хүнд ноогдох өрхийн медиан орлогын 67%–200%",
        vals=pc_vals, lo=0.67 * med_pc, hi=2.00 * med_pc,
        bw=0.16, is_ppp=False,
        lo_lbl="67%", hi_lbl="200%",
    ),
    dict(
        title="Е. Нэг хүнд ноогдох өрхийн тэнцвэржүүлсэн\nмедиан орлогын 75%–200%",
        vals=eq_vals, lo=0.75 * med_eq, hi=2.00 * med_eq,
        bw=0.16, is_ppp=False,
        lo_lbl="75%", hi_lbl="200%",
    ),
]

# xmax тохируулах
for m in METHODS:
    v_pos = m["vals"]
    v_pos = v_pos[(v_pos > 0) & ~np.isnan(v_pos)]
    p99   = float(np.nanpercentile(v_pos, 99.0)) if len(v_pos) else 100
    if m["is_ppp"]:
        m["xmax"] = min(p99 * 1.15, m["hi"] * 3.5, 260.0)
    else:
        m["xmax"] = min(p99 * 1.05, m["hi"] * 2.4)

# ── KDE ТООЦОХ ────────────────────────────────────────────────
def build_kde(vals, wts, xmax, bw=0.16, n=4000):
    mask = (
        ~np.isnan(vals) & ~np.isnan(wts)
        & (vals > 0) & (vals < xmax * 1.8)
    )
    v, w2 = vals[mask], wts[mask]
    if len(v) < 10:
        return None, None
    kde = gaussian_kde(v, weights=w2 / w2.sum(), bw_method=bw)
    x   = np.linspace(0, xmax, n)
    y   = kde(x)
    return x, y

# ── ЗУРАХ ТУСГАЙ ФУНКЦ ────────────────────────────────────────
ORANGE  = "#F5A623"
GRAY_LO = "#C8C8C8"
GRAY_HI = "#B0B0B0"
LINE_C  = "#404040"

def peak_in_range(x, y, x_lo, x_hi):
    """Тухайн завсарт KDE-ийн оргилын (x, y) олох"""
    mask = (x >= x_lo) & (x <= x_hi)
    if not mask.any():
        mid = (x_lo + x_hi) / 2
        idx = int(np.argmin(np.abs(x - mid)))
        return float(x[idx]), float(y[idx])
    idx = int(np.argmax(y[mask]))
    return float(x[mask][idx]), float(y[mask][idx])


def draw_panel(ax, m, w):
    x, y = build_kde(m["vals"], w, m["xmax"], m["bw"])
    if x is None:
        ax.text(0.5, 0.5, "Өгөгдөл хангалтгүй",
                transform=ax.transAxes, ha="center", va="center", fontsize=14)
        ax.set_title(m["title"], fontsize=20, fontweight="bold")
        return

    lo, hi, xmax = m["lo"], m["hi"], m["xmax"]
    ymax  = float(y.max())
    ylim  = ymax * 1.45      # ← өмнө 1.60 байсан, одоо бага зай

    # ── Дүүргэлт ────────────────────────────────────────────────
    ax.fill_between(x, y, where=(x <  lo),             color=GRAY_LO, alpha=0.60, linewidth=0)
    ax.fill_between(x, y, where=(x >= lo) & (x <= hi), color=ORANGE,  alpha=0.88, linewidth=0)
    ax.fill_between(x, y, where=(x >  hi),             color=GRAY_HI, alpha=0.40, linewidth=0)

    # ── KDE муруй ────────────────────────────────────────────────
    ax.plot(x, y, color=LINE_C, lw=1.8, alpha=0.72, zorder=5)

    # ── Босоо зааглах шугам ──────────────────────────────────────
    for xv in [lo, hi]:
        ax.axvline(xv, color="#777777", lw=1.0, ls="--", alpha=0.75, zorder=4)

    # ── Хязгаарын тэмдэглэл — x тэнхлэгийн дор ────────────────
    for xv, lbl in [(lo, m["lo_lbl"]), (hi, m["hi_lbl"])]:
        ax.text(
            xv, -ylim * 0.10,
            lbl,
            ha="center", va="top",
            fontsize=16, fontweight="bold", color="#222222",
            transform=ax.transData,
            clip_on=False,
            zorder=11,
        )

    # ── Хувиуд тооцох ───────────────────────────────────────────
    vals  = m["vals"]
    valid = ~np.isnan(vals)
    lo_p  = wpct(valid & (vals <  lo),              w)
    mid_p = wpct(valid & (vals >= lo) & (vals <= hi), w)
    hi_p  = wpct(valid & (vals >  hi),              w)

    eps = xmax * 0.002
    regions = [
        ("left",   eps,      lo - eps, lo_p),
        ("mid",    lo,       hi,       mid_p),
        ("right",  hi + eps, xmax,     hi_p),
    ]

    for side, r_lo, r_hi, pct in regions:
        if r_lo >= r_hi or r_hi > xmax * 1.01:
            continue
        pk_x, pk_y = peak_in_range(x, y, r_lo, r_hi)
        if pk_y < ymax * 0.02:
            continue

        if side == "left":
            # Зүүн нарийн бүс: lo зааглах шугамаас зүүн тийш тогтмол
            cx = lo * 0.45
            if cx <= 0:
                cx = lo * 0.5
            idx_cx = int(np.argmin(np.abs(x - cx)))
            cy = float(y[idx_cx])
            label_y = max(cy * 0.50, ymax * 0.12)
        else:
            # Дунд ба баруун бүс: жинлэсэн масс-ын төв
            mask_r = (x >= r_lo) & (x <= min(r_hi, xmax))
            if mask_r.sum() > 1:
                w_r  = y[mask_r]
                x_r  = x[mask_r]
                cx   = float(np.average(x_r, weights=w_r))
                idx_cx = int(np.argmin(np.abs(x - cx)))
                cy   = float(y[idx_cx])
            else:
                cx, cy = pk_x, pk_y
            label_y = cy * 0.45

        ax.text(
            cx, label_y,
            f"{pct:.1f}%",
            ha="center", va="center",
            fontsize=19, fontweight="bold", color="#1a1a1a",
            bbox=dict(boxstyle="round,pad=0.18",
                      facecolor="white", alpha=0.70, edgecolor="none"),
            zorder=10,
            clip_on=False,
        )

    # ── Тэнхлэг тохируулга ──────────────────────────────────────
    ax.set_xlim(0, xmax)
    ax.set_ylim(0, ylim)

    xlabel = "Өдрийн орлого" if m["is_ppp"] else "Нэг хүнд ноогдох орлого"
    ax.set_xlabel(xlabel, fontsize=17, labelpad=34)

    # ── ӨӨРЧЛӨЛТ: Y тэнхлэгийн label-ийг бүрэн хас ────────────
    ax.set_ylabel("")   # хоосон болгох

    ax.yaxis.set_ticklabels([])
    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", labelsize=15, colors="#555555", pad=10)

    if not m["is_ppp"]:
        ax.xaxis.set_major_formatter(
            mticker.FuncFormatter(
                lambda v, _: f"{v/1e6:.0f}сая" if v >= 1e6 else f"{v/1e3:.0f}м"
            )
        )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#DDDDDD")
    ax.spines["bottom"].set_color("#CCCCCC")
    ax.grid(False)
    ax.set_facecolor("white")

    # ── Гарчиг ──────────────────────────────────────────────────
    ax.set_title(m["title"], fontsize=20, fontweight="heavy", pad=14, loc="center",
                 fontfamily="DejaVu Sans")

    print(
        f"  {m['title'].split(chr(10))[0]:48s}"
        f"  Доод={lo_p:5.1f}%  Дундаж={mid_p:5.1f}%  Дээд={hi_p:5.1f}%"
    )

# ── ЗУРАГ ҮҮСГЭХ (2 багана × 3 мөр) ──────────────────────────
fig, axes = plt.subplots(
    nrows=3, ncols=2,
    figsize=(20, 24),
    facecolor="white",
)
fig.patch.set_facecolor("white")

fig.suptitle(
    f"Зураг 1. {TARGET_YEAR} оны дундаж давхаргын орлогын тархалтын нягтын муруй\n"
    "болон дундаж давхаргын эзлэх хувь — аргачлал тус бүрээр",
    fontsize=26, fontweight="bold",
    y=0.995, ha="center", va="top",
)

print("\nПанел тус бүрийн тооцоо:")
for ax, m in zip(axes.flat, METHODS):
    draw_panel(ax, m, w)

plt.tight_layout(rect=[0, 0.0, 1, 0.975], h_pad=7.0, w_pad=5.0)
plt.savefig(OUT_PATH, dpi=200, bbox_inches="tight", facecolor="white")
plt.close()

print(f"\n✅  Зураг хадгалагдлаа → {OUT_PATH}")

2024 он: 15,511 өрх
  Медиан PC  :    7,558,412 ₮/жил
  Медиан EQ  :   11,707,667 ₮/жил

Панел тус бүрийн тооцоо:
  А. Нэг хүнд ноогдох өдрийн орлого 2 – 13 ам.доллар  Доод=  4.3%  Дундаж= 26.3%  Дээд= 69.4%
  Б. Нэг хүнд ноогдох өдрийн орлого 10 – 50 ам.доллар  Доод= 21.1%  Дундаж= 71.4%  Дээд=  7.5%
  В. Нэг хүнд ноогдох өдрийн орлого 2 – 20 ам.доллар  Доод=  4.3%  Дундаж= 50.9%  Дээд= 44.9%
  Г. Нэг хүнд ноогдох өрхийн медиан орлогын 75%–125%  Доод= 33.6%  Дундаж= 30.9%  Дээд= 35.5%
  Д. Нэг хүнд ноогдох өрхийн медиан орлогын 67%–200%  Доод= 28.7%  Дундаж= 57.6%  Дээд= 13.6%
  Е. Нэг хүнд ноогдох өрхийн тэнцвэржүүлсэн         Доод= 33.4%  Дундаж= 53.1%  Дээд= 13.6%

✅  Зураг хадгалагдлаа → C:\Users\b22fa\Desktop\diplom unelgee\fig_2x3_methods.png


In [12]:
"""
Дундаж давхаргын өөрчлөлт — line graph → Word (.docx)
"""

import io
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import warnings
warnings.filterwarnings("ignore")
matplotlib.rcParams["text.usetex"] = False
matplotlib.rcParams["mathtext.default"] = "regular"

from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

OUT_PATH = r"C:\Users\b22fa\Desktop\diplom unelgee\middle_class_line_graph.docx"

# ── ӨГӨГДӨЛ ───────────────────────────────────────────────────
years = [2008, 2010, 2012, 2014, 2016, 2018, 2020, 2022, 2024]

SERIES = [
    dict(label="2–13 ам.$/өдөр",               data=[61.1,67.5,63.4,64.1,68.7,65.6,52.7,42.7,26.3], color="#3266AD", ls="-"),
    dict(label="10–50 ам.$/өдөр",              data=[10.6,15.9,36.0,41.5,39.1,41.9,57.3,65.2,71.4], color="#E2692A", ls="-"),
    dict(label="2–20 ам.$/өдөр",               data=[64.8,73.6,77.3,80.3,84.4,81.8,76.3,71.5,50.9], color="#C7392A", ls="-"),
    dict(label="Медиан орлогын 75–125%",        data=[19.9,22.5,24.3,27.9,28.2,27.4,30.2,32.6,30.9], color="#3D9E75", ls="--"),
    dict(label="Медиан орлогын 67–200%",        data=[41.7,46.4,48.2,53.3,55.4,53.8,56.7,59.2,57.6], color="#7B4DB5", ls="--"),
    dict(label="Тэнцвэржүүлсэн медианы 75–200%",data=[39.7,42.8,45.4,50.5,54.5,52.2,53.6,56.0,53.1], color="#888780", ls="--"),
]

# ── LINE GRAPH ЗУРАХ → PNG buffer ─────────────────────────────
fig, ax = plt.subplots(figsize=(13, 7), facecolor="white")
ax.set_facecolor("white")

for s in SERIES:
    ax.plot(
        years, s["data"],
        color=s["color"], linestyle=s["ls"],
        linewidth=2.4, marker="o", markersize=5.5,
        markerfacecolor="white", markeredgewidth=2.0,
        markeredgecolor=s["color"],
        label=s["label"], zorder=4,
    )
    # Эхний ба эцсийн утга
    for xi, yi in [(years[0], s["data"][0]), (years[-1], s["data"][-1])]:
        ax.annotate(
            f"{yi:.1f}%", xy=(xi, yi), xytext=(0, 9),
            textcoords="offset points",
            ha="center", va="bottom",
            fontsize=9.5, color=s["color"], fontweight="bold",
        )

ax.set_xlim(2007, 2025)
ax.set_ylim(0, 95)
ax.set_xticks(years)
ax.xaxis.set_tick_params(labelsize=12, colors="#555555")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.yaxis.set_tick_params(labelsize=12, colors="#555555")
ax.grid(axis="y", color="#EEEEEE", linewidth=0.8, zorder=1)
ax.grid(axis="x", color="#F5F5F5", linewidth=0.6, zorder=1)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#DDDDDD")
ax.spines["bottom"].set_color("#CCCCCC")
ax.set_xlabel("Он", fontsize=13, labelpad=8, color="#444444")
ax.set_ylabel("Дундаж давхаргын эзлэх хувь (%)", fontsize=13, labelpad=8, color="#444444")

# Legend 2 баганаар
from matplotlib.lines import Line2D
solid_lbl = Line2D([0],[0], color="#888", lw=0, label="Үнэмлэхүй босго (PPP):")
dash_lbl  = Line2D([0],[0], color="#888", lw=0, label="Харьцангуй медиан:")
handles = [solid_lbl]
for s in SERIES[:3]:
    handles.append(Line2D([0],[0], color=s["color"], lw=2.2, linestyle="-",
                          marker="o", markersize=5, markerfacecolor="white",
                          markeredgewidth=1.8, markeredgecolor=s["color"], label=s["label"]))
handles.append(dash_lbl)
for s in SERIES[3:]:
    handles.append(Line2D([0],[0], color=s["color"], lw=2.2, linestyle="--",
                          marker="o", markersize=5, markerfacecolor="white",
                          markeredgewidth=1.8, markeredgecolor=s["color"], label=s["label"]))

ax.legend(handles=handles, fontsize=10.5, loc="upper center",
          bbox_to_anchor=(0.5, -0.13), ncol=2, frameon=False,
          columnspacing=1.4, handlelength=2.2, handletextpad=0.6)

ax.set_title(
    "Зураг 2. Дундаж давхаргын эзлэх хувийн өөрчлөлт — аргачлал тус бүрээр (2008–2024)",
    fontsize=14, fontweight="heavy", pad=16, color="#1a1a1a",
)

plt.tight_layout(rect=[0, 0.08, 1, 1])

# PNG-г memory buffer-т хадгалах
img_buf = io.BytesIO()
plt.savefig(img_buf, dpi=200, bbox_inches="tight", facecolor="white", format="png")
img_buf.seek(0)
plt.close()
print("Chart rendered.")

# ── WORD ДОКУМЕНТ ҮҮСГЭХ ──────────────────────────────────────
doc = Document()

# Хуудасны тохиргоо — A4 landscape
section = doc.sections[0]
section.page_width  = int(16838 * 914.4 / 914400 * 914400 / 914.4)   # keep default A4
section.page_height = int(11906 * 914.4 / 914400 * 914400 / 914.4)
# Landscape: swap width/height
section.page_width  = 16838 * 635 // 450  # ~23.7 cm in EMU-compatible twips
section.page_height = 11906 * 635 // 450

# Хялбар арга: python-docx Inches ашиглана
from docx.shared import Inches, Cm
s = doc.sections[0]
s.page_width  = Inches(11.69)   # A4 landscape width
s.page_height = Inches(8.27)    # A4 landscape height
s.left_margin   = Cm(2)
s.right_margin  = Cm(2)
s.top_margin    = Cm(2)
s.bottom_margin = Cm(2)

# Гарчиг
title = doc.add_heading("Дундаж давхаргын эзлэх хувийн өөрчлөлт (2008–2024)", level=1)
title.alignment = WD_ALIGN_PARAGRAPH.CENTER
title.runs[0].font.size = Pt(16)
title.runs[0].font.color.rgb = RGBColor(0x1A, 0x1A, 0x1A)

doc.add_paragraph()  # хоосон мөр

# Зураг оруулах — өргөнийг хуудасны агуулгын хэмжээнд тохируулна
# A4 landscape, 2cm margin хоёр талд → агуулгын өргөн = 11.69 - 1.57 = ~10.12 inch
doc.add_picture(img_buf, width=Inches(10.0))

# Зургийн тайлбар
cap = doc.add_paragraph()
cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
run = cap.add_run(
    "Зураг 2. Дундаж давхаргын эзлэх хувийн өөрчлөлт — аргачлал тус бүрээр (2008–2024)\n"
    "Хатуу шугам: үнэмлэхүй PPP босго (3 арга) | Тасархай шугам: харьцангуй медиан (3 арга)"
)
run.font.size   = Pt(10)
run.font.italic = True
run.font.color.rgb = RGBColor(0x55, 0x55, 0x55)

# Хадгалах
doc.save(OUT_PATH)
print(f"Word файл хадгалагдлаа: {OUT_PATH}")

Chart rendered.
Word файл хадгалагдлаа: C:\Users\b22fa\Desktop\diplom unelgee\middle_class_line_graph.docx


In [13]:
"""
Монгол Улсын Дундаж Давхаргын Тооцоо – OECD (2019) арга
=========================================================
Тэнцвэржүүлсэн орлогын томьёо:
    Y_eq = Y / (N_adults + 0.5 × N_children)

Давхаргын ангилал (медианы харьцаагаар):
    Бага    : Y_eq_i / median  < 0.75
    Дундаж  : 0.75 ≤ Y_eq_i / median ≤ 2.00
    Өндөр   : Y_eq_i / median  > 2.00

Гаралт (нэмэгдэх баганууд):
    • Тэнцвэржүүлсэн орлого (₮)
    • Медиантай харьцаа
    • Бага    (0/1)
    • Дундаж  (0/1)
    • Өндөр   (0/1)
"""

import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# ТОХИРГОО  –  зам болон жилүүд
# ─────────────────────────────────────────────
FILE_PATH  = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata.xlsx"
EVEN_YEARS = list(range(2008, 2025, 2))   # 2008, 2010, …, 2024

# OECD завсар
LO, HI = 0.75, 2.00

# ─────────────────────────────────────────────
# Туслах функцүүд
# ─────────────────────────────────────────────
def weighted_median(values, weights):
    """Жинлэсэн медиан (NaN-г хасна)."""
    v = np.asarray(values, float)
    w = np.asarray(weights, float)
    mask = ~(np.isnan(v) | np.isnan(w))
    v, w = v[mask], w[mask]
    if len(v) == 0:
        return np.nan
    order = np.argsort(v)
    v, w  = v[order], w[order]
    cumw  = np.cumsum(w)
    half  = cumw[-1] / 2.0
    idx   = min(np.searchsorted(cumw, half), len(v) - 1)
    return v[idx]

def border_thin():
    s = Side(style='thin')
    return Border(left=s, right=s, top=s, bottom=s)

def hdr(cell, text, bg="1F4E79", fg="FFFFFF", sz=9):
    cell.value     = text
    cell.font      = Font(bold=True, color=fg, size=sz)
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal='center', vertical='center',
                               wrap_text=True)
    cell.border    = border_thin()

def dat(cell, value, bg=None, bold=False, color="000000", align='center'):
    cell.value     = value
    cell.font      = Font(bold=bold, color=color, size=9)
    cell.alignment = Alignment(horizontal=align, vertical='center')
    cell.border    = border_thin()
    if bg:
        cell.fill  = PatternFill("solid", fgColor=bg)

# ─────────────────────────────────────────────
# ӨГӨГДӨЛ УНШИХ
# ─────────────────────────────────────────────
print("="*60)
print("Монгол Улсын Дундаж Давхаргын Тооцоо  (OECD 75–200%)")
print("="*60)

df = pd.read_excel(FILE_PATH, sheet_name=0, header=1)
print(f"Нийт мөр: {len(df)},  баганы тоо: {len(df.columns)}")

cols = df.columns.tolist()
print("Баганууд:", cols)

# Баганы харгалзуулалт (дараалал тогтмол гэж үзнэ)
year_col          = cols[0]   # A – Он
household_col     = cols[1]   # B – Өрхийн дугаар
children_col      = cols[2]   # C – Хүүхэд (14-с доош)
adults_main_col   = cols[3]   # D – Тэргүүлэх насанд хүрсэн
adults_other_col  = cols[4]   # E – Бусад насанд хүрсэн
total_members_col = cols[5]   # F – Нийт гишүүд
weight_col        = cols[6]   # G – Өрхийн жин (hhweight)
income_col        = cols[11]  # L – Нийт орлого (жилийн)

print(f"\nЖингийн багана : {weight_col}")
print(f"Орлогын багана : {income_col}")

# ─────────────────────────────────────────────
# ТЭГШ ЖИЛҮҮД ШҮҮНЭ
# ─────────────────────────────────────────────
df_e = df[df[year_col].isin(EVEN_YEARS)].copy()
print(f"Тэгш жилийн өрх: {len(df_e)}")

# ─────────────────────────────────────────────
# ТЭНЦВЭРЖҮҮЛСЭН ОРЛОГО
# Y_eq = Y / (N_adults + 0.5 × N_children)
# N_adults  = тэргүүлэх + бусад насанд хүрсэн
# N_children = хүүхэд (14-с доош)
# ─────────────────────────────────────────────
def calc_eq(row):
    n_adults  = (0 if pd.isna(row[adults_main_col])  else row[adults_main_col]) \
              + (0 if pd.isna(row[adults_other_col]) else row[adults_other_col])
    n_child   =  0 if pd.isna(row[children_col])    else row[children_col]
    denom     = n_adults + 0.5 * n_child
    inc       = row[income_col]
    if denom <= 0 or pd.isna(inc):
        return np.nan
    return inc / denom

df_e['eq_income'] = df_e.apply(calc_eq, axis=1)

# ─────────────────────────────────────────────
# ЖИНЛЭСЭН МЕДИАН – ОНЫ БҮРТ
# ─────────────────────────────────────────────
print("\nЖилийн жинлэсэн медиан тооцоолж байна...")
medians = {}
for yr in EVEN_YEARS:
    sub = df_e[df_e[year_col] == yr].dropna(subset=[weight_col])
    if len(sub) == 0:
        continue
    wm = weighted_median(sub['eq_income'], sub[weight_col])
    medians[yr] = wm
    print(f"  {yr}: медиан = {wm:>12,.0f} ₮")

# ─────────────────────────────────────────────
# АНГИЛАЛ  (Бага / Дундаж / Өндөр)
# ─────────────────────────────────────────────
def classify(row):
    yr  = row[year_col]
    eq  = row['eq_income']
    med = medians.get(yr, np.nan)
    if pd.isna(eq) or pd.isna(med) or med <= 0:
        return np.nan, 0, 0, 0   # тодорхойгүй → бага гэж үзнэ
    ratio = eq / med
    if ratio < LO:
        return ratio, 1, 0, 0    # Бага
    elif ratio <= HI:
        return ratio, 0, 1, 0    # Дундаж
    else:
        return ratio, 0, 0, 1    # Өндөр

results = df_e.apply(classify, axis=1, result_type='expand')
results.columns = ['ratio', 'Бага', 'Дундаж', 'Өндөр']
df_e = df_e.join(results)

# ─────────────────────────────────────────────
# ЖИНЛЭСЭН ХУВЬ – НЭГДСЭН ДҮН
# ─────────────────────────────────────────────
def wpct(sub, col):
    valid   = sub.dropna(subset=[weight_col])
    total_w = valid[weight_col].sum()
    if total_w == 0:
        return 0.0
    return valid.loc[valid[col] == 1, weight_col].sum() / total_w * 100

print("\n" + "="*52)
print(f"{'Он':>6}  {'Бага':>8} {'Дундаж':>8} {'Өндөр':>8}")
print("-"*52)
for yr in EVEN_YEARS:
    sub = df_e[df_e[year_col] == yr]
    if len(sub) == 0:
        continue
    lo  = wpct(sub, 'Бага')
    mid = wpct(sub, 'Дундаж')
    hi  = wpct(sub, 'Өндөр')
    print(f"{yr:>6}  {lo:>7.1f}%  {mid:>7.1f}%  {hi:>7.1f}%")
print("="*52)

# ─────────────────────────────────────────────
# EXCEL ФАЙЛД БИЧИХ
# ─────────────────────────────────────────────
print("\nExcel-д бичиж байна...")
wb = load_workbook(FILE_PATH)

# ══════════════════════════════════════════════
# SHEET 1 – одоо байгаа өгөгдөлд баганууд нэмнэ
# ══════════════════════════════════════════════
ws1 = wb.worksheets[0]
max_c = ws1.max_column

NEW_COLS = [
    ('eq_income', 'Тэнцвэржүүлсэн\nорлого (₮)',  "D6EAF8", None,    None   ),
    ('ratio',     'Медиантай\nharьцаа',            "D6EAF8", None,    None   ),
    ('Бага',      'Бага\n(0/1)',                   None,     "FDEDEC","922B21"),
    ('Дундаж',    'Дундаж\n(0/1)',                 None,     "EAFAF1","1E8449"),
    ('Өндөр',     'Өндөр\n(0/1)',                  None,     "EBF5FB","1A5276"),
]

col_start = max_c + 1
for i, (_, lbl, *_) in enumerate(NEW_COLS):
    hdr(ws1.cell(row=2, column=col_start + i), lbl)
    ws1.column_dimensions[get_column_letter(col_start + i)].width = 13
ws1.row_dimensions[2].height = 42

# Хурдан хайлт: (он, өрхийн дугаар) → мөр
lookup = {(row[year_col], row[household_col]): row
          for _, row in df_e.iterrows()}

written = 0
for er in range(3, ws1.max_row + 1):
    yv = ws1.cell(row=er, column=1).value
    hv = ws1.cell(row=er, column=2).value
    if yv is None:
        continue
    try:
        yr_int = int(yv)
    except (ValueError, TypeError):
        continue

    rd = lookup.get((yr_int, hv))
    for i, (key, _, bg_calc, bg_bin, txt_bin) in enumerate(NEW_COLS):
        c = ws1.cell(row=er, column=col_start + i)
        c.border = border_thin()
        c.alignment = Alignment(horizontal='center', vertical='center')
        c.font = Font(size=9)

        if rd is None:
            c.value = ""
            continue

        val = rd[key]

        if bg_bin is not None:                  # binary column (0/1)
            v = int(val) if not pd.isna(val) else 0
            c.value = v
            c.font  = Font(bold=(v == 1), color=txt_bin, size=9)
            c.fill  = PatternFill("solid",
                       fgColor=("C6EFCE" if v == 1 else bg_bin))
        else:                                   # numeric column
            if pd.isna(val):
                c.value = ""
            elif key == 'ratio':
                c.value = round(float(val), 4)
            else:
                c.value = round(float(val), 0)
            c.fill = PatternFill("solid", fgColor=bg_calc)
    written += 1

print(f"Sheet1-д нэмэгдсэн мөр: {written}")

# ══════════════════════════════════════════════
# SHEET 2 – жилийн нэгдсэн хүснэгт
# ══════════════════════════════════════════════
S2 = "OECD 75-200% Дүн"
if S2 in wb.sheetnames:
    del wb[S2]
ws2 = wb.create_sheet(S2, index=1)

# Гарчиг
ws2.merge_cells("A1:G1")
t = ws2["A1"]
t.value = ("Монгол Улсын Дундаж Давхаргын Тооцоо  "
           "–  OECD (2019): Тэнцвэржүүлсэн орлогын медианы 75–200%  "
           "(hhweight-ээр жинлэсэн %)")
t.font      = Font(bold=True, size=11, color="FFFFFF")
t.fill      = PatternFill("solid", fgColor="1B2631")
t.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
t.border    = border_thin()
ws2.row_dimensions[1].height = 28

hdrs2 = ['Он', 'Нийт өрх',
         'Жинлэсэн\nмедиан (₮)',
         'Бага\n(<75%)',
         'Дундаж\n(75–200%)',
         'Өндөр\n(>200%)',
         'Нийт']
for j, h in enumerate(hdrs2):
    hdr(ws2.cell(row=2, column=j + 1), h)
    ws2.column_dimensions[get_column_letter(j + 1)].width = 16
ws2.row_dimensions[2].height = 42

for ri, yr in enumerate(EVEN_YEARS):
    sub = df_e[df_e[year_col] == yr]
    if len(sub) == 0:
        continue
    bg = "EBF5FB" if ri % 2 == 0 else "FDFEFE"
    er = ri + 3
    lo  = wpct(sub, 'Бага')
    mid = wpct(sub, 'Дундаж')
    hi  = wpct(sub, 'Өндөр')
    med = medians.get(yr, np.nan)

    row_vals = [
        yr, len(sub),
        f"{med:,.0f}" if not np.isnan(med) else "–",
        f"{lo:.1f}%",
        f"{mid:.1f}%",
        f"{hi:.1f}%",
        "100%",
    ]
    for j, v in enumerate(row_vals):
        c = ws2.cell(row=er, column=j + 1)
        dat(c, v, bg=bg, bold=(j == 0))
        # Дундаж баганыг ногоон өнгөөр
        if j == 4:
            c.font = Font(bold=True, color="1E8449", size=9)
            c.fill = PatternFill("solid", fgColor="EAFAF1")
    ws2.row_dimensions[er].height = 18

# ══════════════════════════════════════════════
# SHEET 3 – Бага / Дундаж / Өндөр дэлгэрэнгүй
# ══════════════════════════════════════════════
S3 = "Давхаргын Дэлгэрэнгүй"
if S3 in wb.sheetnames:
    del wb[S3]
ws3 = wb.create_sheet(S3, index=2)

years_p = [yr for yr in EVEN_YEARS if yr in medians]
n_yc    = len(years_p)

# Гарчиг
ws3.merge_cells(start_row=1, start_column=1,
                end_row=1,   end_column=n_yc + 1)
pt = ws3.cell(row=1, column=1)
pt.value = ("OECD (2019): Тэнцвэржүүлсэн орлогын медианы 75–200%  "
            "–  2008–2024 тэгш жилүүд (hhweight-ээр жинлэсэн %)")
pt.font      = Font(bold=True, size=12, color="FFFFFF")
pt.fill      = PatternFill("solid", fgColor="1B2631")
pt.alignment = Alignment(horizontal='center', vertical='center')
pt.border    = border_thin()
ws3.row_dimensions[1].height = 28

# Он толгой
ws3.cell(row=2, column=1).border = border_thin()
ws3.cell(row=2, column=1).fill   = PatternFill("solid", fgColor="2E86C1")
for ci, yr in enumerate(years_p):
    c = ws3.cell(row=2, column=ci + 2)
    c.value     = str(yr)
    c.font      = Font(bold=True, color="FFFFFF", size=9)
    c.fill      = PatternFill("solid", fgColor="2E86C1")
    c.alignment = Alignment(horizontal='center', vertical='center')
    c.border    = border_thin()
ws3.row_dimensions[2].height = 18

ROWS = [
    ("Бага (<75%)",    "FDEDEC", "922B21", False),
    ("Дундаж (75–200%)", "EAFAF1", "1E8449", True ),
    ("Өндөр (>200%)",  "EBF5FB", "1A5276", False),
    ("Нийт",           "F4F6F7", "000000", False),
]
BIN_MAP = {"Бага (<75%)": 'Бага', "Дундаж (75–200%)": 'Дундаж',
           "Өндөр (>200%)": 'Өндөр'}

for ri, (lbl, bg, tc, bold) in enumerate(ROWS):
    er = ri + 3
    lc = ws3.cell(row=er, column=1)
    lc.value     = lbl
    lc.font      = Font(bold=bold, color=tc, size=9)
    lc.fill      = PatternFill("solid", fgColor=bg)
    lc.alignment = Alignment(horizontal='left', vertical='center', indent=1)
    lc.border    = border_thin()

    for ci, yr in enumerate(years_p):
        dc = ws3.cell(row=er, column=ci + 2)
        sub = df_e[df_e[year_col] == yr]
        if lbl == "Нийт":
            dc.value = "100%"
        else:
            pct = wpct(sub, BIN_MAP[lbl])
            dc.value = f"{pct:.1f}%"
        dc.font      = Font(bold=bold, color=tc, size=9)
        dc.fill      = PatternFill("solid", fgColor=bg)
        dc.alignment = Alignment(horizontal='center', vertical='center')
        dc.border    = border_thin()
    ws3.row_dimensions[er].height = 18

ws3.column_dimensions['A'].width = 18
for ci in range(n_yc):
    ws3.column_dimensions[get_column_letter(ci + 2)].width = 9
ws3.freeze_panes = "B3"

# ─────────────────────────────────────────────
# ХАДГАЛАХ
# ─────────────────────────────────────────────
out_path = FILE_PATH.replace(".xlsx", "_OECD_middle_class.xlsx")
wb.save(out_path)
print(f"\n✅  Хадгалагдлаа: {out_path}")
print("   → Sheet1: Анхны өгөгдөл + 5 нэмэгдсэн багана")
print("   → Sheet2: Жилийн нэгдсэн хувиар дүн")
print("   → Sheet3: Бага / Дундаж / Өндөр дэлгэрэнгүй")

Монгол Улсын Дундаж Давхаргын Тооцоо  (OECD 75–200%)
Нийт мөр: 139206,  баганы тоо: 14
Баганууд: ['Он', 'Өрхийн дугаар', 'Хүүхэд (14-с доош)', 'Тэргүүлэх насанд хүрсэн', 'Бусад насанд хүрсэн', 'Нийт гишүүд', 'Өрхийн жин (hhweight)', 'Цалин хөлс', 'Тэтгэвэр тэтгэмж бусад', 'Үйлдвэрлэл үйлчилгээ', 'Бэлэг тусламж өөрийн аж ахуй', 'Нийт орлого', 'Нэрлэсэн орлого\n(2015 үнэ)', 'Бодит орлого\n(2015 үнэ, ₮)']

Жингийн багана : Өрхийн жин (hhweight)
Орлогын багана : Нийт орлого
Тэгш жилийн өрх: 139206

Жилийн жинлэсэн медиан тооцоолж байна...
  2008: медиан =      650,701 ₮
  2010: медиан =    1,042,000 ₮
  2012: медиан =    2,072,667 ₮
  2014: медиан =    2,935,533 ₮
  2016: медиан =    2,910,000 ₮
  2018: медиан =    3,400,000 ₮
  2020: медиан =    4,571,056 ₮
  2022: медиан =    5,796,000 ₮
  2024: медиан =    8,456,667 ₮

    Он      Бага   Дундаж    Өндөр
----------------------------------------------------
  2008     38.7%     39.4%     21.9%
  2010     38.3%     43.0%     18.6%
  2012  

In [14]:
"""
Боловсрол × Орлогын бүлэг хүснэгт  (2008 ба 2024)
====================================================
Алхамууд:
1. alldata.xlsx → OECD 75-200% аргаар 2008, 2024-ийн өрхийн
   Бага/Дундаж/Өндөр ангилал тооцно
2. Individual файлуудаас тэргүүн (ind_id=1)-ий боловсролыг авна
3. Өрхийн ID-гаар нийлүүлнэ
4. Боловсролын 4 бүлгээр орлогын ангиллын % хүснэгт гаргана
5. Шинэ Excel sheet-д хадгална

Боловсролын бүлгүүд (4-р зурагнаас):
  Бага                  : 2008→[2],      2024→[2]
  Дунд                  : 2008→[4],      2024→[4]
  Мэргэжлийн анхан дунд: 2008→[5],      2024→[5,6]
  Дээд                  : 2008→[7],      2024→[8,9,10]
"""

import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# ЗАМУУД
# ─────────────────────────────────────────────
ALLDATA_PATH  = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata.xlsx"
INDIV_2024    = r"C:\Users\b22fa\Desktop\diplom unelgee\2024\02_indiv (1).xlsx"
INDIV_2008    = r"C:\Users\b22fa\Desktop\diplom unelgee\2008\Indivdual (1).xlsx"
OUTPUT_PATH   = ALLDATA_PATH.replace(".xlsx", "_educ_income.xlsx")

EVEN_YEARS = list(range(2008, 2025, 2))
LO, HI     = 0.75, 2.00

# Боловсролын бүлгийн харгалзуулалт
EDUC_MAP = {
    2008: {2: "Бага", 4: "Дунд", 5: "Мэргэжлийн анхан дунд", 7: "Дээд"},
    2024: {2: "Бага", 4: "Дунд", 5: "Мэргэжлийн анхан дунд",
           6: "Мэргэжлийн анхан дунд", 8: "Дээд", 9: "Дээд", 10: "Дээд"},
}
EDUC_ORDER = ["Бага", "Дунд", "Мэргэжлийн анхан дунд", "Дээд"]

# ─────────────────────────────────────────────
# ТУСЛАХ ФУНКЦҮҮД
# ─────────────────────────────────────────────
def weighted_median(values, weights):
    v = np.asarray(values, float); w = np.asarray(weights, float)
    mask = ~(np.isnan(v) | np.isnan(w))
    v, w = v[mask], w[mask]
    if len(v) == 0: return np.nan
    order = np.argsort(v); v, w = v[order], w[order]
    cumw = np.cumsum(w); half = cumw[-1] / 2.0
    return v[min(np.searchsorted(cumw, half), len(v)-1)]

def border_thin():
    s = Side(style='thin')
    return Border(left=s, right=s, top=s, bottom=s)

def hdr(ws, r, c, text, bg="1F4E79", fg="FFFFFF", sz=9, merge_end_col=None):
    cell = ws.cell(row=r, column=c)
    if merge_end_col:
        ws.merge_cells(start_row=r, start_column=c,
                       end_row=r, end_column=merge_end_col)
    cell.value     = text
    cell.font      = Font(bold=True, color=fg, size=sz)
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal='center', vertical='center',
                               wrap_text=True)
    cell.border    = border_thin()
    return cell

def dat(ws, r, c, value, bg=None, bold=False, color="000000",
        align='center', sz=9):
    cell = ws.cell(row=r, column=c)
    cell.value     = value
    cell.font      = Font(bold=bold, color=color, size=sz)
    cell.alignment = Alignment(horizontal=align, vertical='center')
    cell.border    = border_thin()
    if bg: cell.fill = PatternFill("solid", fgColor=bg)
    return cell

# ─────────────────────────────────────────────
# 1. ALLDATA УНШИЖ OECD АНГИЛАЛ ТООЦНО
# ─────────────────────────────────────────────
print("="*60)
print("alldata.xlsx уншиж байна...")
df = pd.read_excel(ALLDATA_PATH, sheet_name=0, header=1)
cols = df.columns.tolist()
print("Баганууд:", cols[:12])

year_col          = cols[0]
household_col     = cols[1]
children_col      = cols[2]
adults_main_col   = cols[3]
adults_other_col  = cols[4]
total_members_col = cols[5]
weight_col        = cols[6]
income_col        = cols[11]

# Зөвхөн 2008, 2024
df_sel = df[df[year_col].isin([2008, 2024])].copy()
print(f"2008 өрх: {len(df_sel[df_sel[year_col]==2008])}, "
      f"2024 өрх: {len(df_sel[df_sel[year_col]==2024])}")

# Тэнцвэржүүлсэн орлого
def calc_eq(row):
    n_a = (0 if pd.isna(row[adults_main_col])  else row[adults_main_col]) \
        + (0 if pd.isna(row[adults_other_col]) else row[adults_other_col])
    n_c =  0 if pd.isna(row[children_col])    else row[children_col]
    denom = n_a + 0.5 * n_c
    inc   = row[income_col]
    if denom <= 0 or pd.isna(inc): return np.nan
    return inc / denom

df_sel['eq_income'] = df_sel.apply(calc_eq, axis=1)

# Жинлэсэн медиан
medians = {}
for yr in [2008, 2024]:
    sub = df_sel[df_sel[year_col]==yr].dropna(subset=[weight_col])
    wm  = weighted_median(sub['eq_income'], sub[weight_col])
    medians[yr] = wm
    print(f"  {yr} медиан: {wm:,.0f} ₮")

# Ангилал
def classify(row):
    yr = row[year_col]; eq = row['eq_income']
    med = medians.get(yr, np.nan)
    if pd.isna(eq) or pd.isna(med) or med <= 0: return "Бага"
    r = eq / med
    if r < LO:   return "Бага"
    if r <= HI:  return "Дундаж"
    return "Өндөр"

df_sel['income_class'] = df_sel.apply(classify, axis=1)

print("\nОрлогын ангилал (жинлэсэн %):")
for yr in [2008, 2024]:
    sub = df_sel[df_sel[year_col]==yr].dropna(subset=[weight_col])
    tw  = sub[weight_col].sum()
    for cl in ["Бага","Дундаж","Өндөр"]:
        w = sub.loc[sub['income_class']==cl, weight_col].sum()
        print(f"  {yr} {cl}: {w/tw*100:.1f}%")

# Өрхийн ID тогтоох – 'identif' байвал ашиглана, байхгүй бол cols[1]
hh_id_col = 'identif' if 'identif' in df_sel.columns else household_col
print(f"\nӨрхийн ID багана: {hh_id_col}")

# ─────────────────────────────────────────────
# 2. INDIVIDUAL ФАЙЛУУДААС ТЭРГҮҮНИЙ БОЛОВСРОЛ
# ─────────────────────────────────────────────

def load_head_educ(path, id_col_guess, educ_col, year):
    """ind_id=1 буюу тэргүүний боловсролыг авна."""
    print(f"\n  {year} individual файл: {path}")
    dfi = pd.read_excel(path, header=0)
    print(f"  Баганууд: {dfi.columns.tolist()[:15]}")

    # ind_id багана олох
    ind_col = None
    for c in dfi.columns:
        if str(c).lower() in ['ind_id','indid','ind id','memberid','member_id']:
            ind_col = c; break
    if ind_col is None:
        # 2-р багана ихэвчлэн ind_id
        ind_col = dfi.columns[1]
        print(f"  ind_id багана олдсонгүй → {ind_col} ашиглана")
    else:
        print(f"  ind_id багана: {ind_col}")

    # identif / өрхийн ID багана олох
    hh_col = None
    for c in dfi.columns:
        if str(c).lower() in ['identif','hhid','hh_id','hh id','household_id']:
            hh_col = c; break
    if hh_col is None:
        hh_col = dfi.columns[0]
        print(f"  identif багана олдсонгүй → {hh_col} ашиглана")
    else:
        print(f"  identif багана: {hh_col}")

    # Боловсролын багана
    educ_c = None
    for c in dfi.columns:
        if str(c).lower() == educ_col.lower():
            educ_c = c; break
    if educ_c is None:
        print(f"  АНХААРУУЛГА: {educ_col} багана олдсонгүй!")
        print(f"  Байгаа баганууд: {dfi.columns.tolist()}")
        return None
    print(f"  Боловсролын багана: {educ_c}")

    # Тэргүүн (ind_id == 1) шүүнэ
    heads = dfi[dfi[ind_col] == 1][[hh_col, educ_c]].copy()
    heads.columns = ['identif_indiv', 'educ_raw']
    heads['educ_raw'] = pd.to_numeric(heads['educ_raw'], errors='coerce')
    print(f"  Тэргүүний тоо: {len(heads)}")
    return heads

heads_2024 = load_head_educ(INDIV_2024, 'identif', 'q0210', 2024)
heads_2008 = load_head_educ(INDIV_2008, 'identif', 'q0204', 2008)

# ─────────────────────────────────────────────
# 3. НИЙЛҮҮЛЭХ
# ─────────────────────────────────────────────
def merge_and_classify(df_hh, heads, year):
    """Өрхийн өгөгдлийг individual-тай нийлүүлж боловсролын бүлэг нэмнэ."""
    sub = df_hh[df_hh[year_col] == year].copy()

    # identif нэрийг нэгтгэх
    sub = sub.rename(columns={hh_id_col: 'identif_hh'})
    sub['identif_hh'] = pd.to_numeric(sub['identif_hh'], errors='coerce')
    heads['identif_indiv'] = pd.to_numeric(heads['identif_indiv'], errors='coerce')

    merged = sub.merge(heads, left_on='identif_hh', right_on='identif_indiv',
                       how='left')
    print(f"\n{year} нийлүүлэлт: {len(merged)} өрх, "
          f"боловсрол олдсон: {merged['educ_raw'].notna().sum()}")

    # Боловсролын бүлэг
    emap = EDUC_MAP[year]
    merged['educ_group'] = merged['educ_raw'].map(emap)
    print(f"  Боловсролын бүлгийн тархалт:\n"
          f"{merged['educ_group'].value_counts().to_string()}")
    return merged

if heads_2024 is not None:
    mrg_2024 = merge_and_classify(df_sel, heads_2024, 2024)
else:
    mrg_2024 = None

if heads_2008 is not None:
    mrg_2008 = merge_and_classify(df_sel, heads_2008, 2008)
else:
    mrg_2008 = None

# ─────────────────────────────────────────────
# 4. ХУВЬ ТООЦООЛОХ (hhweight-ээр жинлэсэн)
# ─────────────────────────────────────────────
INC_CLASSES = ["Бага", "Дундаж", "Өндөр"]

def calc_pct_table(merged, year):
    """
    Боловсролын бүлэг × орлогын ангилал  →  жинлэсэн % хүснэгт
    """
    result = {}
    for eg in EDUC_ORDER:
        sub = merged[merged['educ_group'] == eg].dropna(subset=[weight_col])
        tw  = sub[weight_col].sum()
        row = {}
        for cl in INC_CLASSES:
            if tw == 0:
                row[cl] = 0.0
            else:
                w = sub.loc[sub['income_class']==cl, weight_col].sum()
                row[cl] = w / tw * 100
        row['Нийт'] = sum(row[c] for c in INC_CLASSES)
        row['n']    = len(sub)
        result[eg]  = row
    return result

tbl_2008 = calc_pct_table(mrg_2008, 2008) if mrg_2008 is not None else {}
tbl_2024 = calc_pct_table(mrg_2024, 2024) if mrg_2024 is not None else {}

print("\n" + "="*70)
print(f"{'Боловсрол':<28} {'2008':^30} {'2024':^30}")
print(f"{'':28} {'Бага':>8} {'Дундаж':>8} {'Өндөр':>8}  "
      f"{'Бага':>8} {'Дундаж':>8} {'Өндөр':>8}")
print("-"*70)
for eg in EDUC_ORDER:
    r8 = tbl_2008.get(eg, {})
    r24= tbl_2024.get(eg, {})
    print(f"{eg:<28} "
          f"{r8.get('Бага',0):>7.1f}% "
          f"{r8.get('Дундаж',0):>7.1f}% "
          f"{r8.get('Өндөр',0):>7.1f}%  "
          f"{r24.get('Бага',0):>7.1f}% "
          f"{r24.get('Дундаж',0):>7.1f}% "
          f"{r24.get('Өндөр',0):>7.1f}%")
print("="*70)

# ─────────────────────────────────────────────
# 5. EXCEL-Д ХАДГАЛАХ
# ─────────────────────────────────────────────
print("\nExcel-д бичиж байна...")
wb = load_workbook(ALLDATA_PATH)

S_NAME = "Боловсрол × Орлого"
if S_NAME in wb.sheetnames:
    del wb[S_NAME]
ws = wb.create_sheet(S_NAME)

# ── Гарчиг ───────────────────────────────────
hdr(ws, 1, 1,
    "Хүснэгт. Өрхийн тэргүүний боловсрол, орлогын бүлгээр (OECD 75–200%, hhweight-ээр жинлэсэн %)",
    bg="1B2631", fg="FFFFFF", sz=11, merge_end_col=9)
ws.row_dimensions[1].height = 28

# ── Дэд гарчиг: он ───────────────────────────
hdr(ws, 2, 1, "Боловсролын бүлэг", bg="2E86C1", sz=10)
hdr(ws, 2, 2, "2008 он", bg="1F4E79", sz=10, merge_end_col=4)
hdr(ws, 2, 6, "2024 он", bg="154360", sz=10, merge_end_col=8)
# dummy merged cells
for c in [3,4,7,8]:
    ws.cell(row=2, column=c).fill = PatternFill("solid",
        fgColor="1F4E79" if c<6 else "154360")
    ws.cell(row=2, column=c).border = border_thin()
hdr(ws, 2, 9, "Нийт", bg="4D5656", sz=9)
ws.row_dimensions[2].height = 20

# ── Дэд гарчиг: ангилал ─────────────────────
sub_hdrs = ["Бага", "Дундаж", "Өндөр", "Нийт"]
BG_SUB   = ["FADBD8", "D5F5E3", "D6EAF8", "F2F3F4"]
TC_SUB   = ["922B21", "1E8449", "1A5276", "000000"]

for i, (sh, bg, tc) in enumerate(zip(sub_hdrs, BG_SUB, TC_SUB)):
    # 2008
    c = ws.cell(row=3, column=2 + i)
    c.value = sh; c.font = Font(bold=True, color=tc, size=9)
    c.fill  = PatternFill("solid", fgColor=bg)
    c.alignment = Alignment(horizontal='center', vertical='center')
    c.border = border_thin()
    # 2024
    c2 = ws.cell(row=3, column=6 + i)
    c2.value = sh; c2.font = Font(bold=True, color=tc, size=9)
    c2.fill  = PatternFill("solid", fgColor=bg)
    c2.alignment = Alignment(horizontal='center', vertical='center')
    c2.border = border_thin()

ws.cell(row=3, column=1).border = border_thin()
ws.cell(row=3, column=1).fill   = PatternFill("solid", fgColor="EBF5FB")
ws.cell(row=3, column=9).border = border_thin()
ws.row_dimensions[3].height = 18

# ── Өгөгдлийн мөрүүд ─────────────────────────
EDUC_BG = ["FEF9E7", "EAFAF1", "EBF5FB", "F9EBEA"]
EDUC_TC = ["7D6608", "1E8449", "1A5276", "922B21"]

for ri, eg in enumerate(EDUC_ORDER):
    er = ri + 4
    bg = EDUC_BG[ri]; tc = EDUC_TC[ri]

    # Боловсролын нэр
    lc = ws.cell(row=er, column=1)
    lc.value = eg
    lc.font  = Font(bold=True, color=tc, size=9)
    lc.fill  = PatternFill("solid", fgColor=bg)
    lc.alignment = Alignment(horizontal='left', vertical='center', indent=1)
    lc.border = border_thin()

    r8  = tbl_2008.get(eg, {})
    r24 = tbl_2024.get(eg, {})

    for ci, cl in enumerate(["Бага","Дундаж","Өндөр","Нийт"]):
        v8  = r8.get(cl,  0)
        v24 = r24.get(cl, 0)
        txt = f"{v8:.1f}%"  if v8  != 100 else "100%"
        t24 = f"{v24:.1f}%" if v24 != 100 else "100%"

        c8  = ws.cell(row=er, column=2 + ci)
        c8.value = txt
        c8.font  = Font(bold=(cl=="Дундаж"), color=EDUC_TC[ri]
                        if cl!="Нийт" else "000000", size=9)
        c8.fill  = PatternFill("solid", fgColor=bg)
        c8.alignment = Alignment(horizontal='center', vertical='center')
        c8.border = border_thin()

        c24 = ws.cell(row=er, column=6 + ci)
        c24.value = t24
        c24.font  = Font(bold=(cl=="Дундаж"), color=EDUC_TC[ri]
                         if cl!="Нийт" else "000000", size=9)
        c24.fill  = PatternFill("solid", fgColor=bg)
        c24.alignment = Alignment(horizontal='center', vertical='center')
        c24.border = border_thin()

    # Нийт багана (баруун)
    ntc = ws.cell(row=er, column=9)
    ntc.value = "100%"
    ntc.font  = Font(bold=True, size=9)
    ntc.fill  = PatternFill("solid", fgColor=bg)
    ntc.alignment = Alignment(horizontal='center', vertical='center')
    ntc.border = border_thin()
    ws.row_dimensions[er].height = 18

# ── Таних тэмдэглэл ───────────────────────────
note_row = len(EDUC_ORDER) + 5
ws.merge_cells(start_row=note_row, start_column=1,
               end_row=note_row, end_column=9)
nc = ws.cell(row=note_row, column=1)
nc.value = ("Тайлбар: OECD (2019) аргаар тэнцвэржүүлсэн орлогын медианы "
            "75–200% = Дундаж давхарга.  "
            "Боловсролын бүлэг: 2008 (2=Бага,4=Дунд,5=Мэрж.дунд,7=Дээд)  |  "
            "2024 (2=Бага,4=Дунд,5-6=Мэрж.дунд,8-10=Дээд)")
nc.font  = Font(italic=True, size=8, color="555555")
nc.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
nc.border = border_thin()
ws.row_dimensions[note_row].height = 30

# ── Баганын өргөн ─────────────────────────────
ws.column_dimensions['A'].width = 26
for ci in range(8):
    ws.column_dimensions[get_column_letter(ci + 2)].width = 11

wb.save(OUTPUT_PATH)
print(f"\n✅  Хадгалагдлаа: {OUTPUT_PATH}")
print(f"   → Sheet: '{S_NAME}'")

alldata.xlsx уншиж байна...
Баганууд: ['Он', 'Өрхийн дугаар', 'Хүүхэд (14-с доош)', 'Тэргүүлэх насанд хүрсэн', 'Бусад насанд хүрсэн', 'Нийт гишүүд', 'Өрхийн жин (hhweight)', 'Цалин хөлс', 'Тэтгэвэр тэтгэмж бусад', 'Үйлдвэрлэл үйлчилгээ', 'Бэлэг тусламж өөрийн аж ахуй', 'Нийт орлого']
2008 өрх: 11172, 2024 өрх: 15511
  2008 медиан: 650,701 ₮
  2024 медиан: 8,456,667 ₮

Орлогын ангилал (жинлэсэн %):
  2008 Бага: 38.7%
  2008 Дундаж: 39.4%
  2008 Өндөр: 21.9%
  2024 Бага: 31.9%
  2024 Дундаж: 54.7%
  2024 Өндөр: 13.4%

Өрхийн ID багана: Өрхийн дугаар

  2024 individual файл: C:\Users\b22fa\Desktop\diplom unelgee\2024\02_indiv (1).xlsx
  Баганууд: ['identif', 'ind_id', 'q0102', 'q0103', 'q0105y', 'q0105m', 'q0106', 'q0107', 'q0108', 'q0109', 'q0110', 'q0111', 'q0112a', 'q0112b', 'q0113']
  ind_id багана: ind_id
  identif багана: identif
  Боловсролын багана: q0210
  Тэргүүний тоо: 15513

  2008 individual файл: C:\Users\b22fa\Desktop\diplom unelgee\2008\Indivdual (1).xlsx
  Баганууд: ['ide

In [17]:
"""
Хадгаламж ба Зээлийн хүснэгт  (2008 ба 2024)  – засварласан хувилбар
"""

import pandas as pd
import numpy as np

OECD_PATH  = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata_OECD_middle_class.xlsx"
HHOLD_2024 = r"C:\Users\b22fa\Desktop\diplom unelgee\2024\01_hhold.xlsx"
HHOLD_2008 = r"C:\Users\b22fa\Desktop\diplom unelgee\2008\Household (1).xlsx"

INC_ORDER = ["Бага", "Дундаж", "Өндөр"]

# ─────────────────────────────────────────────
# Хэсэгчилсэн тохирох багана хайх
# ─────────────────────────────────────────────
def find_col(df, keywords):
    """Баганын нэрэнд keyword агуулагдах эсэхийг шалгана (case-insensitive)."""
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in str(col).lower():
                return col
    return None

# ─────────────────────────────────────────────
# 1. OECD файл унших
# ─────────────────────────────────────────────
df = pd.read_excel(OECD_PATH, sheet_name=0, header=1)

year_col   = find_col(df, ['он','year'])
hh_col     = find_col(df, ['өрхийн дугаар','identif','hhid'])
weight_col = find_col(df, ['hhweight','жин'])
baga_col   = find_col(df, ['бага'])
dund_col   = find_col(df, ['дундаж'])
ondor_col  = find_col(df, ['өндөр'])

print(f"year={year_col}, hh={hh_col}, weight={weight_col}")
print(f"Бага={baga_col}, Дундаж={dund_col}, Өндөр={ondor_col}")

# Орлогын ангилал
def get_cls(row):
    try:
        if row[dund_col]  == 1: return "Дундаж"
        if row[baga_col]  == 1: return "Бага"
        if row[ondor_col] == 1: return "Өндөр"
    except: pass
    return np.nan

df['inc_class'] = df.apply(get_cls, axis=1)
df['hh_id']     = pd.to_numeric(df[hh_col],     errors='coerce')
df['wt']        = pd.to_numeric(df[weight_col], errors='coerce')
df['yr']        = pd.to_numeric(df[year_col],   errors='coerce')

# Шалгах
for yr in [2008, 2024]:
    sub = df[df['yr'] == yr]
    print(f"\n{yr}: өрх={len(sub)}, inc_class тархалт:")
    print(sub['inc_class'].value_counts().to_string())

# ─────────────────────────────────────────────
# 2. Household файлуудаас хадгаламж/зээл авах
# ─────────────────────────────────────────────
def load_hhold(path, sav_q, loan_q):
    hh = pd.read_excel(path, header=0)
    id_col   = find_col(hh, ['identif','hhid']) or hh.columns[0]
    sav_col  = find_col(hh, [sav_q])
    loan_col = find_col(hh, [loan_q])
    print(f"\n  id={id_col}, sav={sav_col}, loan={loan_col}")

    out = pd.DataFrame()
    out['hh_id'] = pd.to_numeric(hh[id_col], errors='coerce')
    sraw = pd.to_numeric(hh[sav_col],  errors='coerce') if sav_col  else np.nan
    lraw = pd.to_numeric(hh[loan_col], errors='coerce') if loan_col else np.nan
    out['savings'] = np.where(sraw == 1, 1, np.where(sraw == 2, 0, np.nan))
    out['loan']    = np.where(lraw == 1, 1, np.where(lraw == 2, 0, np.nan))
    return out

print("\n▶ 2024 household:")
hh24 = load_hhold(HHOLD_2024, 'q0801', 'q0806')

print("\n▶ 2008 household:")
hh08 = load_hhold(HHOLD_2008, 'q1201', 'q1202')

# ─────────────────────────────────────────────
# 3. Нийлүүлэх
# ─────────────────────────────────────────────
def merge_yr(year, hh_prep):
    oecd = df[df['yr'] == year][['hh_id','wt','inc_class']].copy()
    m    = oecd.merge(hh_prep, on='hh_id', how='left')
    print(f"\n{year}: OECD={len(oecd)}, нийлэгдсэн savings={m['savings'].notna().sum()}")
    return m

m24 = merge_yr(2024, hh24)
m08 = merge_yr(2008, hh08)

# ─────────────────────────────────────────────
# 4. Жинлэсэн % тооцоо
# ─────────────────────────────────────────────
def wpct(merged, col):
    res = {}
    for inc in INC_ORDER:
        sub = merged[(merged['inc_class'] == inc) & merged[col].notna()]
        tw  = sub['wt'].sum()
        res[inc] = (sub.loc[sub[col]==1,'wt'].sum() / tw * 100) if tw > 0 else np.nan
    return res

sav08 = wpct(m08, 'savings')
sav24 = wpct(m24, 'savings')
lon08 = wpct(m08, 'loan')
lon24 = wpct(m24, 'loan')

# ─────────────────────────────────────────────
# 5. Хүснэгт хэвлэх
# ─────────────────────────────────────────────
def print_tbl(title, d08, d24):
    print(f"\n{'='*48}")
    print(f"  {title}")
    print(f"{'='*48}")
    print(f"  {'':14} {'2008':>10} {'2024':>10}")
    print(f"  {'-'*36}")
    for inc in INC_ORDER:
        v8  = d08.get(inc, np.nan)
        v24 = d24.get(inc, np.nan)
        s8  = f"{v8:.1f}%"  if not (isinstance(v8,  float) and np.isnan(v8))  else "–"
        s24 = f"{v24:.1f}%" if not (isinstance(v24, float) and np.isnan(v24)) else "–"
        pfx = "**" if inc == "Дундаж" else "  "
        print(f"  {pfx}{inc:<14} {s8:>10} {s24:>10}")
    print(f"{'='*48}")

print("\n\n" + "#"*48)
print("  ЭЦС ДҮН")
print("#"*48)

print_tbl("Хүснэгт 8. Хадгаламжтай өрхийн эзлэх хувь", sav08, sav24)
print_tbl("Хүснэгт 9. Зээлтэй өрхийн эзлэх хувь",      lon08, lon24)
print("\n✅ Дууслаа.")

year=Он, hh=Өрхийн дугаар, weight=Өрхийн жин (hhweight)
Бага=Бага
(0/1), Дундаж=Дундаж
(0/1), Өндөр=Өндөр
(0/1)

2008: өрх=11172, inc_class тархалт:
inc_class
Бага      4502
Дундаж    4360
Өндөр     2310

2024: өрх=15511, inc_class тархалт:
inc_class
Дундаж    7978
Бага      6032
Өндөр     1501

▶ 2024 household:

  id=identif, sav=q0801, loan=q0806

▶ 2008 household:

  id=identif, sav=q1201, loan=q1202

2024: OECD=15511, нийлэгдсэн savings=15511

2008: OECD=11172, нийлэгдсэн savings=11172


################################################
  ЭЦС ДҮН
################################################

  Хүснэгт 8. Хадгаламжтай өрхийн эзлэх хувь
                       2008       2024
  ------------------------------------
    Бага                15.7%      21.8%
  **Дундаж              21.4%      37.2%
    Өндөр               41.2%      56.0%

  Хүснэгт 9. Зээлтэй өрхийн эзлэх хувь
                       2008       2024
  ------------------------------------
    Бага                24.1% 

In [18]:
"""
Хадгаламж ба Зээлийн хүснэгт  (2008 ба 2024)  – засварласан хувилбар
"""

import pandas as pd
import numpy as np

OECD_PATH  = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata_OECD_middle_class.xlsx"
HHOLD_2024 = r"C:\Users\b22fa\Desktop\diplom unelgee\2024\01_hhold.xlsx"
HHOLD_2008 = r"C:\Users\b22fa\Desktop\diplom unelgee\2008\Household (1).xlsx"

INC_ORDER = ["Бага", "Дундаж", "Өндөр"]

# ─────────────────────────────────────────────
# Хэсэгчилсэн тохирох багана хайх
# ─────────────────────────────────────────────
def find_col(df, keywords):
    """Баганын нэрэнд keyword агуулагдах эсэхийг шалгана (case-insensitive)."""
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in str(col).lower():
                return col
    return None

# ─────────────────────────────────────────────
# 1. OECD файл унших
# ─────────────────────────────────────────────
df = pd.read_excel(OECD_PATH, sheet_name=0, header=1)

year_col   = find_col(df, ['он','year'])
hh_col     = find_col(df, ['өрхийн дугаар','identif','hhid'])
weight_col = find_col(df, ['hhweight','жин'])
baga_col   = find_col(df, ['бага'])
dund_col   = find_col(df, ['дундаж'])
ondor_col  = find_col(df, ['өндөр'])

print(f"year={year_col}, hh={hh_col}, weight={weight_col}")
print(f"Бага={baga_col}, Дундаж={dund_col}, Өндөр={ondor_col}")

# Орлогын ангилал
def get_cls(row):
    try:
        if row[dund_col]  == 1: return "Дундаж"
        if row[baga_col]  == 1: return "Бага"
        if row[ondor_col] == 1: return "Өндөр"
    except: pass
    return np.nan

df['inc_class'] = df.apply(get_cls, axis=1)
df['hh_id']     = pd.to_numeric(df[hh_col],     errors='coerce')
df['wt']        = pd.to_numeric(df[weight_col], errors='coerce')
df['yr']        = pd.to_numeric(df[year_col],   errors='coerce')

# Шалгах
for yr in [2008, 2024]:
    sub = df[df['yr'] == yr]
    print(f"\n{yr}: өрх={len(sub)}, inc_class тархалт:")
    print(sub['inc_class'].value_counts().to_string())

# ─────────────────────────────────────────────
# 2. Household файлуудаас хадгаламж/зээл авах
# ─────────────────────────────────────────────
def load_hhold(path, sav_q, loan_q):
    hh = pd.read_excel(path, header=0)
    id_col   = find_col(hh, ['identif','hhid']) or hh.columns[0]
    sav_col  = find_col(hh, [sav_q])
    loan_col = find_col(hh, [loan_q])
    print(f"\n  id={id_col}, sav={sav_col}, loan={loan_col}")

    out = pd.DataFrame()
    out['hh_id'] = pd.to_numeric(hh[id_col], errors='coerce')
    sraw = pd.to_numeric(hh[sav_col],  errors='coerce') if sav_col  else np.nan
    lraw = pd.to_numeric(hh[loan_col], errors='coerce') if loan_col else np.nan
    out['savings'] = np.where(sraw == 1, 1, np.where(sraw == 2, 0, np.nan))
    out['loan']    = np.where(lraw == 1, 1, np.where(lraw == 2, 0, np.nan))
    return out

print("\n▶ 2024 household:")
hh24 = load_hhold(HHOLD_2024, 'q0801', 'q0803')

print("\n▶ 2008 household:")
hh08 = load_hhold(HHOLD_2008, 'q1201', 'q1202')

# ─────────────────────────────────────────────
# 3. Нийлүүлэх
# ─────────────────────────────────────────────
def merge_yr(year, hh_prep):
    oecd = df[df['yr'] == year][['hh_id','wt','inc_class']].copy()
    m    = oecd.merge(hh_prep, on='hh_id', how='left')
    print(f"\n{year}: OECD={len(oecd)}, нийлэгдсэн savings={m['savings'].notna().sum()}")
    return m

m24 = merge_yr(2024, hh24)
m08 = merge_yr(2008, hh08)

# ─────────────────────────────────────────────
# 4. Жинлэсэн % тооцоо
# ─────────────────────────────────────────────
def wpct(merged, col):
    res = {}
    for inc in INC_ORDER:
        sub = merged[(merged['inc_class'] == inc) & merged[col].notna()]
        tw  = sub['wt'].sum()
        res[inc] = (sub.loc[sub[col]==1,'wt'].sum() / tw * 100) if tw > 0 else np.nan
    return res

sav08 = wpct(m08, 'savings')
sav24 = wpct(m24, 'savings')
lon08 = wpct(m08, 'loan')
lon24 = wpct(m24, 'loan')

# ─────────────────────────────────────────────
# 5. Хүснэгт хэвлэх
# ─────────────────────────────────────────────
def print_tbl(title, d08, d24):
    print(f"\n{'='*48}")
    print(f"  {title}")
    print(f"{'='*48}")
    print(f"  {'':14} {'2008':>10} {'2024':>10}")
    print(f"  {'-'*36}")
    for inc in INC_ORDER:
        v8  = d08.get(inc, np.nan)
        v24 = d24.get(inc, np.nan)
        s8  = f"{v8:.1f}%"  if not (isinstance(v8,  float) and np.isnan(v8))  else "–"
        s24 = f"{v24:.1f}%" if not (isinstance(v24, float) and np.isnan(v24)) else "–"
        pfx = "**" if inc == "Дундаж" else "  "
        print(f"  {pfx}{inc:<14} {s8:>10} {s24:>10}")
    print(f"{'='*48}")

print("\n\n" + "#"*48)
print("  ЭЦС ДҮН")
print("#"*48)

print_tbl("Хүснэгт 8. Хадгаламжтай өрхийн эзлэх хувь", sav08, sav24)
print_tbl("Хүснэгт 9. Зээлтэй өрхийн эзлэх хувь",      lon08, lon24)
print("\n✅ Дууслаа.")

year=Он, hh=Өрхийн дугаар, weight=Өрхийн жин (hhweight)
Бага=Бага
(0/1), Дундаж=Дундаж
(0/1), Өндөр=Өндөр
(0/1)

2008: өрх=11172, inc_class тархалт:
inc_class
Бага      4502
Дундаж    4360
Өндөр     2310

2024: өрх=15511, inc_class тархалт:
inc_class
Дундаж    7978
Бага      6032
Өндөр     1501

▶ 2024 household:

  id=identif, sav=q0801, loan=q0803

▶ 2008 household:

  id=identif, sav=q1201, loan=q1202

2024: OECD=15511, нийлэгдсэн savings=15511

2008: OECD=11172, нийлэгдсэн savings=11172


################################################
  ЭЦС ДҮН
################################################

  Хүснэгт 8. Хадгаламжтай өрхийн эзлэх хувь
                       2008       2024
  ------------------------------------
    Бага                15.7%      21.8%
  **Дундаж              21.4%      37.2%
    Өндөр               41.2%      56.0%

  Хүснэгт 9. Зээлтэй өрхийн эзлэх хувь
                       2008       2024
  ------------------------------------
    Бага                24.1% 

In [22]:
"""
Салбар × Орлогын бүлэг  (2024)
================================
q0432 = өрхийн тэргүүний ажилладаг салбар
Эхлээд q0432-ийн бүх утгыг харуулна, дараа хүснэгт гаргана.
"""

import pandas as pd
import numpy as np

OECD_PATH  = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata_OECD_middle_class.xlsx"
INDIV_2024 = r"C:\Users\b22fa\Desktop\diplom unelgee\2024\02_indiv (1).xlsx"

INC_ORDER = ["Бага", "Дундаж", "Өндөр"]

def find_col(df, keywords):
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in str(col).lower():
                return col
    return None

# ─────────────────────────────────────────────
# 1. OECD файл – орлогын ангилал авах
# ─────────────────────────────────────────────
df = pd.read_excel(OECD_PATH, sheet_name=0, header=1)

year_col   = find_col(df, ['он','year'])
hh_col     = find_col(df, ['өрхийн дугаар','identif','hhid'])
weight_col = find_col(df, ['hhweight','жин'])
baga_col   = find_col(df, ['бага'])
dund_col   = find_col(df, ['дундаж'])
ondor_col  = find_col(df, ['өндөр'])

def get_cls(row):
    try:
        if row[dund_col]  == 1: return "Дундаж"
        if row[baga_col]  == 1: return "Бага"
        if row[ondor_col] == 1: return "Өндөр"
    except: pass
    return np.nan

df['inc_class'] = df.apply(get_cls, axis=1)
df['hh_id'] = pd.to_numeric(df[hh_col],     errors='coerce')
df['wt']    = pd.to_numeric(df[weight_col], errors='coerce')
df['yr']    = pd.to_numeric(df[year_col],   errors='coerce')

oecd24 = df[df['yr'] == 2024][['hh_id','wt','inc_class']].copy()
print(f"2024 OECD өрх: {len(oecd24)}")
print(f"Орлогын ангилал:\n{oecd24['inc_class'].value_counts().to_string()}")

# ─────────────────────────────────────────────
# 2. Individual файл – тэргүүний q0432 авах
# ─────────────────────────────────────────────
print("\n" + "="*60)
print("Individual файл уншиж байна...")
di = pd.read_excel(INDIV_2024, header=0)

id_col  = find_col(di, ['identif','hhid']) or di.columns[0]
ind_col = find_col(di, ['ind_id','indid']) or di.columns[1]
q_col   = find_col(di, ['q0432'])

print(f"identif={id_col}, ind_id={ind_col}, q0432={q_col}")

if q_col is None:
    print("❌ q0432 багана олдсонгүй!")
    print("Байгаа баганууд:", di.columns.tolist())
    exit()



# ─────────────────────────────────────────────
# 3. Нийлүүлэх
# ─────────────────────────────────────────────
merged = oecd24.merge(heads[['hh_id','sector']], on='hh_id', how='left')
print(f"\nНийлэгдсэн: {merged['sector'].notna().sum()} / {len(merged)}")

# ─────────────────────────────────────────────
# 4. Салбар бүрт жинлэсэн %
# ─────────────────────────────────────────────
sectors = sorted(merged['sector'].dropna().unique().astype(int).tolist())
print(f"\nНийт салбарын код: {sectors}")

rows = []
for sec in sectors:
    sub = merged[merged['sector'] == sec].dropna(subset=['wt'])
    tw  = sub['wt'].sum()
    row = {'Салбар код': int(sec), 'n': len(sub)}
    for inc in INC_ORDER:
        if tw == 0:
            row[inc] = np.nan
        else:
            w = sub.loc[sub['inc_class'] == inc, 'wt'].sum()
            row[inc] = w / tw * 100
    row['Нийт'] = sum(row[inc] for inc in INC_ORDER
                      if not (isinstance(row[inc], float) and np.isnan(row[inc])))
    rows.append(row)

result = pd.DataFrame(rows)

# ─────────────────────────────────────────────
# 5. Хүснэгт хэвлэх
# ─────────────────────────────────────────────
print("\n\n" + "="*70)
print("  2024 он – Салбараар орлогын бүлгийн тархалт")
print("  (OECD 75–200%, hhweight-ээр жинлэсэн %)")
print("="*70)
print(f"  {'Код':>4}  {'n':>6}  {'Бага':>9} {'Дундаж':>9} {'Өндөр':>9} {'Нийт':>7}")
print("  " + "-"*56)

for _, r in result.iterrows():
    b  = f"{r['Бага']:.1f}%"   if not np.isnan(r['Бага'])   else "–"
    d  = f"{r['Дундаж']:.1f}%" if not np.isnan(r['Дундаж']) else "–"
    o  = f"{r['Өндөр']:.1f}%"  if not np.isnan(r['Өндөр'])  else "–"
    nt = f"{r['Нийт']:.1f}%"
    mrk = "**" if r['Дундаж'] == result['Дундаж'].max() else "  "
    print(f"  {mrk}{int(r['Салбар код']):>4}  {int(r['n']):>6}  "
          f"{b:>9} {d:>9} {o:>9} {nt:>7}")

print("="*70)
print("\nТайлбар: Код тус бүрийн утга (нэршил) судалгааны кодчлолоос харна уу.")

# ─────────────────────────────────────────────
# 6. EXCEL-Д ХАДГАЛАХ
# ─────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

OUTPUT = r"C:\Users\b22fa\Desktop\diplom unelgee\sector_income_2024.xlsx"

def brd():
    s = Side(style='thin')
    return Border(left=s, right=s, top=s, bottom=s)

def hdr(ws, r, c, text, bg="1F4E79", fg="FFFFFF"):
    cell = ws.cell(row=r, column=c)
    cell.value     = text
    cell.font      = Font(bold=True, color=fg, size=9)
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border    = brd()

def dat(ws, r, c, value, bg=None, bold=False, color="000000"):
    cell = ws.cell(row=r, column=c)
    cell.value     = value
    cell.font      = Font(bold=bold, color=color, size=9)
    cell.alignment = Alignment(horizontal='center', vertical='center')
    cell.border    = brd()
    if bg:
        cell.fill = PatternFill("solid", fgColor=bg)

wb = Workbook()
ws = wb.active
ws.title = "Салбар × Орлого"

# Гарчиг
ws.merge_cells("A1:E1")
t = ws["A1"]
t.value     = "2024 он – Салбараар орлогын бүлгийн тархалт (OECD 75–200%, hhweight-ээр жинлэсэн %)"
t.font      = Font(bold=True, size=11, color="FFFFFF")
t.fill      = PatternFill("solid", fgColor="1B2631")
t.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
t.border    = brd()
ws.row_dimensions[1].height = 28

# Толгой мөр
for ci, (txt, bg) in enumerate([
        ("Салбар код", "1F4E79"),
        ("Тоо (n)",    "1F4E79"),
        ("Бага",       "922B21"),
        ("Дундаж",     "1E8449"),
        ("Өндөр",      "1A5276")]):
    hdr(ws, 2, ci+1, txt, bg=bg)
ws.row_dimensions[2].height = 20

for ci, w in enumerate([14, 10, 12, 12, 12]):
    ws.column_dimensions[get_column_letter(ci+1)].width = w

BG_LO = "FDEDEC"; TC_LO = "922B21"
BG_MD = "EAFAF1"; TC_MD = "1E8449"
BG_HI = "EBF5FB"; TC_HI = "1A5276"
STRIPE = ["F2F3F4", "FDFEFE"]

for ri, (_, r) in enumerate(result.iterrows()):
    er = ri + 3
    bg = STRIPE[ri % 2]
    dat(ws, er, 1, int(r['Салбар код']), bg=bg, bold=True)
    dat(ws, er, 2, int(r['n']),          bg=bg)
    for ci, (inc, ibg, itc) in enumerate([
            ("Бага",   BG_LO, TC_LO),
            ("Дундаж", BG_MD, TC_MD),
            ("Өндөр",  BG_HI, TC_HI)]):
        v   = r[inc]
        txt = f"{v:.1f}%" if not (isinstance(v, float) and np.isnan(v)) else "–"
        dat(ws, er, ci+3, txt, bg=ibg, bold=(inc=="Дундаж"), color=itc)
    ws.row_dimensions[er].height = 16

# Тайлбар
nr = len(result) + 3
ws.merge_cells(start_row=nr, start_column=1, end_row=nr, end_column=5)
nc = ws.cell(row=nr, column=1)
nc.value     = ("Тайлбар: Салбар кодын нэршлийг судалгааны кодчлолоос харна уу.  "
                "Дундаж = OECD (2019) 75–200% аргаар тодорхойлогдсон дундаж давхарга.")
nc.font      = Font(italic=True, size=8, color="555555")
nc.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
nc.border    = brd()
ws.row_dimensions[nr].height = 24

wb.save(OUTPUT)
print(f"\n✅ Excel хадгалагдлаа: {OUTPUT}")
print("✅ Дууслаа.")

2024 OECD өрх: 15511
Орлогын ангилал:
inc_class
Дундаж    7978
Бага      6032
Өндөр     1501

Individual файл уншиж байна...
identif=identif, ind_id=ind_id, q0432=q0432

Нийлэгдсэн: 666 / 15511

Нийт салбарын код: [2, 5, 7, 8, 9, 10, 13, 14, 15, 16, 19, 20, 22, 23, 25, 31, 32, 33, 35, 36, 38, 39, 41, 42, 43, 45, 46, 47, 49, 51, 52, 53, 55, 56, 60, 61, 63, 64, 66, 69, 74, 75, 78, 79, 80, 81, 84, 85, 86, 90, 93, 94, 96, 97, 99, 111, 112, 113, 114, 115, 116]


  2024 он – Салбараар орлогын бүлгийн тархалт
  (OECD 75–200%, hhweight-ээр жинлэсэн %)
   Код       n       Бага    Дундаж     Өндөр    Нийт
  --------------------------------------------------------
       2       8      60.4%     39.6%      0.0%  100.0%
       5      27      15.7%     43.3%     41.0%  100.0%
       7      13       1.9%     32.7%     65.4%  100.0%
       8       3      32.6%     67.4%      0.0%  100.0%
       9      16      24.9%     73.0%      2.1%  100.0%
      10      16      69.1%     28.9%      2.1%  100.0%
 

In [23]:
"""
Хүснэгт. Хот, хөдөөгийн дундаж давхаргын эзлэх хувь (2008 ба 2024)
– Хүснэгт 21-тэй ижил бүтэц
"""

import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# Файлын замууд
# ─────────────────────────────────────────────
OECD_PATH      = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata_OECD_middle_class.xlsx"
BASIC_2024     = r"C:\Users\b22fa\Desktop\diplom unelgee\2024\basicvars.xlsx"
BASIC_2008     = r"C:\Users\b22fa\Desktop\diplom unelgee\2008\basicvars (1).xlsx"

INC_ORDER = ["Бага", "Дундаж", "Өндөр"]

# ─────────────────────────────────────────────
# Хэсэгчилсэн тохирох багана хайх
# ─────────────────────────────────────────────
def find_col(df, keywords):
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in str(col).lower():
                return col
    return None

# ─────────────────────────────────────────────
# 1. OECD файл унших – орлогын ангилал + жин
# ─────────────────────────────────────────────
df = pd.read_excel(OECD_PATH, sheet_name=0, header=1)

year_col   = find_col(df, ['он', 'year'])
hh_col     = find_col(df, ['өрхийн дугаар', 'identif', 'hhid'])
weight_col = find_col(df, ['hhweight', 'жин'])
baga_col   = find_col(df, ['бага'])
dund_col   = find_col(df, ['дундаж'])
ondor_col  = find_col(df, ['өндөр'])

print(f"year={year_col}, hh={hh_col}, weight={weight_col}")
print(f"Бага={baga_col}, Дундаж={dund_col}, Өндөр={ondor_col}")

def get_cls(row):
    try:
        if row[dund_col]  == 1: return "Дундаж"
        if row[baga_col]  == 1: return "Бага"
        if row[ondor_col] == 1: return "Өндөр"
    except: pass
    return np.nan

df['inc_class'] = df.apply(get_cls, axis=1)
df['hh_id']     = pd.to_numeric(df[hh_col],     errors='coerce')
df['wt']        = pd.to_numeric(df[weight_col], errors='coerce')
df['yr']        = pd.to_numeric(df[year_col],   errors='coerce')

for yr in [2008, 2024]:
    sub = df[df['yr'] == yr]
    print(f"\n{yr}: өрх={len(sub)}, inc_class:")
    print(sub['inc_class'].value_counts().to_string())

# ─────────────────────────────────────────────
# 2. basicvars файлаас urban багана авах
# ─────────────────────────────────────────────
def load_basicvars(path):
    bv = pd.read_excel(path, header=0)
    print(f"\n  Баганууд: {list(bv.columns[:15])}")
    
    id_col     = find_col(bv, ['identif', 'hhid']) or bv.columns[0]
    urban_col  = find_col(bv, ['urban', 'хот'])
    
    print(f"  id={id_col}, urban={urban_col}")
    
    out = pd.DataFrame()
    out['hh_id'] = pd.to_numeric(bv[id_col], errors='coerce')
    
    if urban_col:
        raw = pd.to_numeric(bv[urban_col], errors='coerce')
        # 1 = Хот, 2 = Хөдөө
        out['location'] = raw.map({1: 'Хот', 2: 'Хөдөө'})
    else:
        out['location'] = np.nan
        print("  ⚠️  urban багана олдсонгүй!")
    
    return out

print("\n▶ 2024 basicvars:")
bv24 = load_basicvars(BASIC_2024)

print("\n▶ 2008 basicvars:")
bv08 = load_basicvars(BASIC_2008)

# ─────────────────────────────────────────────
# 3. OECD + basicvars нийлүүлэх
# ─────────────────────────────────────────────
def merge_yr(year, bv):
    oecd = df[df['yr'] == year][['hh_id', 'wt', 'inc_class']].copy()
    m    = oecd.merge(bv, on='hh_id', how='left')
    print(f"\n{year}: OECD={len(oecd)}, location заагдсан={m['location'].notna().sum()}")
    print(m['location'].value_counts().to_string())
    return m

m24 = merge_yr(2024, bv24)
m08 = merge_yr(2008, bv08)

# ─────────────────────────────────────────────
# 4. Жинлэсэн % тооцоо – орлогын анги × байршил
# ─────────────────────────────────────────────
def wpct_by_location(merged):
    """
    Хот болон Хөдөөгөөр тус бүрт орлогын ангиллын жинлэсэн хувийг тооцно.
    Нийт = тухайн байршлын нийт жинтэй харьцуулна.
    """
    result = {}
    for loc in ['Хот', 'Хөдөө']:
        loc_sub = merged[merged['location'] == loc]
        total_w = loc_sub['wt'].sum()
        loc_result = {}
        for inc in INC_ORDER:
            inc_sub = loc_sub[loc_sub['inc_class'] == inc]
            w = inc_sub['wt'].sum()
            loc_result[inc] = (w / total_w * 100) if total_w > 0 else np.nan
        result[loc] = loc_result
    return result

res08 = wpct_by_location(m08)
res24 = wpct_by_location(m24)

# ─────────────────────────────────────────────
# 5. Хүснэгт хэвлэх (Хүснэгт 21-тэй ижил бүтэц)
# ─────────────────────────────────────────────
print("\n\n" + "="*62)
print("  Хүснэгт. Хот, хөдөөгийн дундаж давхаргын эзлэх хувь")
print("="*62)
print(f"  {'':16} {'2008':^22} {'2024':^22}")
print(f"  {'':16} {'Хот':>10} {'Хөдөө':>10}  {'Хот':>10} {'Хөдөө':>10}")
print(f"  {'-'*58}")

for inc in INC_ORDER:
    v08_hot   = res08['Хот'].get(inc, np.nan)
    v08_hodoo = res08['Хөдөө'].get(inc, np.nan)
    v24_hot   = res24['Хот'].get(inc, np.nan)
    v24_hodoo = res24['Хөдөө'].get(inc, np.nan)

    def fmt(v):
        return f"{v:.1f}" if not (isinstance(v, float) and np.isnan(v)) else "–"

    pfx  = "**" if inc == "Дундаж" else "  "
    bold = " *" if inc == "Дундаж" else ""
    label = f"{pfx}{inc}{bold}"
    
    print(f"  {label:<18} {fmt(v08_hot):>10} {fmt(v08_hodoo):>10}  {fmt(v24_hot):>10} {fmt(v24_hodoo):>10}")

# Нийт мөр
print(f"  {'-'*58}")
print(f"  {'Нийт':<18} {'100':>10} {'100':>10}  {'100':>10} {'100':>10}")
print("="*62)
print("\n  Эх сурвалж: УСХ, судлаачийн тооцоолол")
print("\n✅ Дууслаа.")

year=Он, hh=Өрхийн дугаар, weight=Өрхийн жин (hhweight)
Бага=Бага
(0/1), Дундаж=Дундаж
(0/1), Өндөр=Өндөр
(0/1)

2008: өрх=11172, inc_class:
inc_class
Бага      4502
Дундаж    4360
Өндөр     2310

2024: өрх=15511, inc_class:
inc_class
Дундаж    7978
Бага      6032
Өндөр     1501

▶ 2024 basicvars:

  Баганууд: ['identif', 'cluster', 'newaimag', 'newsoum', 'bag', 'urban', 'region', 'location', 'strata', 'month', 'quarter', 'interviewer', 'supervisor', 'hhsize', 'hhweight']
  id=identif, urban=urban

▶ 2008 basicvars:

  Баганууд: ['identif', 'cluster', 'aimag', 'strata', 'strata4', 'region', 'urban', 'month', 'quarter', 'year', 'hhweight', 'hhsize', 'supervisor']
  id=identif, urban=urban

2024: OECD=15511, location заагдсан=15511
location
Хөдөө    7793
Хот      7718

2008: OECD=11172, location заагдсан=11172
location
Хот      6192
Хөдөө    4980


  Хүснэгт. Хот, хөдөөгийн дундаж давхаргын эзлэх хувь
                            2008                   2024         
                      

In [24]:
"""
Хүснэгт. Байршлаар орлогын ангиллын эзлэх хувь (2024)
– Хүснэгт 21-тэй ижил бүтэц, 4 байршлаар

2024 location ангилал (basicvars):
  1 = Ulaanbaatar  → Улаанбаатар
  2 = Aimagcenter  → Аймаг
  3 = Soumcenter   → Сум
  4 = Countryside  → Хөдөө
"""

import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
# Файлын замууд
# ─────────────────────────────────────────────
OECD_PATH  = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata_OECD_middle_class.xlsx"
BASIC_2024 = r"C:\Users\b22fa\Desktop\diplom unelgee\2024\basicvars.xlsx"

INC_ORDER = ["Бага", "Дундаж", "Өндөр"]
LOC_ORDER = ["Улаанбаатар", "Аймаг", "Сум", "Хөдөө"]

LOC_MAP = {
    1: "Улаанбаатар",
    2: "Аймаг",
    3: "Сум",
    4: "Хөдөө"
}

# ─────────────────────────────────────────────
# Хэсэгчилсэн тохирох багана хайх
# ─────────────────────────────────────────────
def find_col(df, keywords):
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in str(col).lower():
                return col
    return None

# ─────────────────────────────────────────────
# 1. OECD файл унших – орлогын ангилал + жин
# ─────────────────────────────────────────────
df = pd.read_excel(OECD_PATH, sheet_name=0, header=1)

year_col   = find_col(df, ['он', 'year'])
hh_col     = find_col(df, ['өрхийн дугаар', 'identif', 'hhid'])
weight_col = find_col(df, ['hhweight', 'жин'])
baga_col   = find_col(df, ['бага'])
dund_col   = find_col(df, ['дундаж'])
ondor_col  = find_col(df, ['өндөр'])

print(f"year={year_col}, hh={hh_col}, weight={weight_col}")
print(f"Бага={baga_col}, Дундаж={dund_col}, Өндөр={ondor_col}")

def get_cls(row):
    try:
        if row[dund_col]  == 1: return "Дундаж"
        if row[baga_col]  == 1: return "Бага"
        if row[ondor_col] == 1: return "Өндөр"
    except: pass
    return np.nan

df['inc_class'] = df.apply(get_cls, axis=1)
df['hh_id']     = pd.to_numeric(df[hh_col],     errors='coerce')
df['wt']        = pd.to_numeric(df[weight_col], errors='coerce')
df['yr']        = pd.to_numeric(df[year_col],   errors='coerce')

# 2024 он шүүх
oecd24 = df[df['yr'] == 2024][['hh_id', 'wt', 'inc_class']].copy()
print(f"\n2024: өрх={len(oecd24)}")
print(oecd24['inc_class'].value_counts().to_string())

# ─────────────────────────────────────────────
# 2. basicvars 2024 – location багана авах
# ─────────────────────────────────────────────
bv = pd.read_excel(BASIC_2024, header=0)
print(f"\nbasicvars баганууд: {list(bv.columns[:15])}")

id_col    = find_col(bv, ['identif', 'hhid']) or bv.columns[0]
loc_col   = find_col(bv, ['location', 'urban', 'байршил'])

print(f"id={id_col}, location={loc_col}")

bv_out = pd.DataFrame()
bv_out['hh_id']    = pd.to_numeric(bv[id_col], errors='coerce')
bv_out['location'] = pd.to_numeric(bv[loc_col], errors='coerce').map(LOC_MAP)

print("\nlocation тархалт:")
print(bv_out['location'].value_counts().to_string())

# ─────────────────────────────────────────────
# 3. Нийлүүлэх
# ─────────────────────────────────────────────
merged = oecd24.merge(bv_out, on='hh_id', how='left')
print(f"\nНийлэгдсэн: {len(merged)}, location заагдсан: {merged['location'].notna().sum()}")

# ─────────────────────────────────────────────
# 4. Жинлэсэн % тооцоо – орлогын анги × байршил
# ─────────────────────────────────────────────
def wpct_by_location(merged):
    result = {}
    for loc in LOC_ORDER:
        loc_sub = merged[merged['location'] == loc]
        total_w = loc_sub['wt'].sum()
        loc_result = {}
        for inc in INC_ORDER:
            inc_sub = loc_sub[loc_sub['inc_class'] == inc]
            w = inc_sub['wt'].sum()
            loc_result[inc] = (w / total_w * 100) if total_w > 0 else np.nan
        result[loc] = loc_result
    return result

res = wpct_by_location(merged)

# ─────────────────────────────────────────────
# 5. Хүснэгт хэвлэх
# ─────────────────────────────────────────────
def fmt(v):
    return f"{v:.1f}" if not (isinstance(v, float) and np.isnan(v)) else "–"

col_w = 12

print("\n\n" + "="*70)
print("  Хүснэгт. Байршлаар орлогын ангиллын эзлэх хувь (2024)")
print("="*70)

header = f"  {'':16}"
for loc in LOC_ORDER:
    header += f"{loc:>{col_w}}"
print(header)
print(f"  {'-'*66}")

for inc in INC_ORDER:
    pfx   = "**" if inc == "Дундаж" else "  "
    label = f"{pfx}{inc}"
    row   = f"  {label:<18}"
    for loc in LOC_ORDER:
        row += f"{fmt(res[loc].get(inc, np.nan)):>{col_w}}"
    print(row)

print(f"  {'-'*66}")
total_row = f"  {'Нийт':<18}"
for loc in LOC_ORDER:
    total_row += f"{'100':>{col_w}}"
print(total_row)

print("="*70)
print("\n  Эх сурвалж: УСХ, судлаачийн тооцоолол")
print("\n✅ Дууслаа.")

year=Он, hh=Өрхийн дугаар, weight=Өрхийн жин (hhweight)
Бага=Бага
(0/1), Дундаж=Дундаж
(0/1), Өндөр=Өндөр
(0/1)

2024: өрх=15511
inc_class
Дундаж    7978
Бага      6032
Өндөр     1501

basicvars баганууд: ['identif', 'cluster', 'newaimag', 'newsoum', 'bag', 'urban', 'region', 'location', 'strata', 'month', 'quarter', 'interviewer', 'supervisor', 'hhsize', 'hhweight']
id=identif, location=location

location тархалт:
location
Аймаг          5142
Сум            4311
Хөдөө          3483
Улаанбаатар    2577

Нийлэгдсэн: 15511, location заагдсан: 15511


  Хүснэгт. Байршлаар орлогын ангиллын эзлэх хувь (2024)
                   Улаанбаатар       Аймаг         Сум       Хөдөө
  ------------------------------------------------------------------
    Бага                    15.1        28.2        39.4        70.1
  **Дундаж                  64.9        59.9        53.6        26.7
    Өндөр                   20.0        12.0         7.0         3.2
  --------------------------------------------

In [26]:
"""
Хүснэгт. УБ дүүргүүдээр орлогын ангиллын эзлэх хувь (2024)
– q0115b-ээс дүүргийн код авах
– Excel файл үүсгэх
"""

import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
OECD_PATH  = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata_OECD_middle_class.xlsx"
INDIV_2024 = r"C:\Users\b22fa\Desktop\diplom unelgee\2024\02_indiv (1).xlsx"
OUTPUT     = r"C:\Users\b22fa\Desktop\diplom unelgee\husnegt_duureg_2024.xlsx"

INC_ORDER = ["Бага", "Дундаж", "Өндөр"]

DISTRICT_MAP = {
    1107: "Баянгол",
    1110: "Баянзүрх",
    1113: "Налайх",
    1116: "Сонгинохайрхан",
    1119: "Сүхбаатар",
    1122: "Хан-Уул",
    1125: "Чингэлтэй"
}
DIST_ORDER = list(DISTRICT_MAP.values())

def find_col(df, keywords):
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in str(col).lower():
                return col
    return None

# ─── 1. OECD ────────────────────────────────
df = pd.read_excel(OECD_PATH, sheet_name=0, header=1)

year_col   = find_col(df, ['он', 'year'])
hh_col     = find_col(df, ['өрхийн дугаар', 'identif', 'hhid'])
weight_col = find_col(df, ['hhweight', 'жин'])
baga_col   = find_col(df, ['бага'])
dund_col   = find_col(df, ['дундаж'])
ondor_col  = find_col(df, ['өндөр'])

def get_cls(row):
    try:
        if row[dund_col]  == 1: return "Дундаж"
        if row[baga_col]  == 1: return "Бага"
        if row[ondor_col] == 1: return "Өндөр"
    except: pass
    return np.nan

df['inc_class'] = df.apply(get_cls, axis=1)
df['hh_id']     = pd.to_numeric(df[hh_col],     errors='coerce')
df['wt']        = pd.to_numeric(df[weight_col], errors='coerce')
df['yr']        = pd.to_numeric(df[year_col],   errors='coerce')

oecd24 = df[df['yr'] == 2024][['hh_id', 'wt', 'inc_class']].copy()
print(f"2024: өрх={len(oecd24)}")

# ─── 2. indiv – q0115b ──────────────────────
indiv = pd.read_excel(INDIV_2024, header=0)

indiv['hh_id']     = pd.to_numeric(indiv['identif'], errors='coerce')
indiv['dist_code'] = pd.to_numeric(indiv['q0115b'],  errors='coerce')

print("\nq0115b тархалт:")
print(indiv['dist_code'].value_counts().sort_index().to_string())

# Зөвхөн дүүргийн кодууд шүүх
indiv_f = indiv[indiv['dist_code'].isin(DISTRICT_MAP.keys())]

# Өрх тус бүрд нэг дүүрэг
hh_dist = (
    indiv_f.groupby('hh_id')['dist_code']
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan)
    .reset_index()
)
hh_dist['district'] = hh_dist['dist_code'].map(DISTRICT_MAP)

print(f"\nДүүргээр өрхийн тоо:")
print(hh_dist['district'].value_counts().to_string())

# ─── 3. Нийлүүлэх ───────────────────────────
merged = oecd24.merge(hh_dist[['hh_id', 'district']], on='hh_id', how='left')
print(f"\nНийлэгдсэн: {len(merged)}, дүүрэг заагдсан: {merged['district'].notna().sum()}")

# ─── 4. Жинлэсэн % ──────────────────────────
def wpct_by_district(merged):
    result = {}
    for dist in DIST_ORDER:
        sub   = merged[merged['district'] == dist]
        total = sub['wt'].sum()
        d = {}
        for inc in INC_ORDER:
            w = sub[sub['inc_class'] == inc]['wt'].sum()
            d[inc] = (w / total * 100) if total > 0 else np.nan
        result[dist] = d
    return result

res = wpct_by_district(merged)

# ─── 5. DataFrame үүсгэх ────────────────────
rows = []
for inc in INC_ORDER:
    row = {"Орлогын ангилал": inc}
    for dist in DIST_ORDER:
        row[dist] = round(res[dist].get(inc, np.nan), 1)
    rows.append(row)

total_row = {"Орлогын ангилал": "Нийт"}
for dist in DIST_ORDER:
    total_row[dist] = 100.0
rows.append(total_row)

result_df = pd.DataFrame(rows)

# ─── 6. Excel хэвлэх ────────────────────────
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    result_df.to_excel(writer, sheet_name='Дүүрэг', index=False)

    wb = writer.book
    ws = writer.sheets['Дүүрэг']

    thin        = Side(style='thin')
    border      = Border(top=thin, bottom=thin, left=thin, right=thin)
    header_fill = PatternFill("solid", fgColor="1F4E79")
    dund_fill   = PatternFill("solid", fgColor="D6E4F0")
    niit_fill   = PatternFill("solid", fgColor="F2F2F2")

    # Толгой мөр
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.fill      = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = border

    # Өгөгдлийн мөрүүд
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        for cell in row:
            cell.border    = border
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.font      = Font(name="Arial", size=10)

        label = row[0].value
        if label == "Дундаж":
            for cell in row:
                cell.fill = dund_fill
                cell.font = Font(bold=True, name="Arial", size=10)
        elif label == "Нийт":
            for cell in row:
                cell.fill = niit_fill
                cell.font = Font(bold=True, name="Arial", size=10)

        row[0].alignment = Alignment(horizontal="left", vertical="center")

    # Багана өргөн
    ws.column_dimensions['A'].width = 20
    for i in range(2, len(DIST_ORDER) + 2):
        ws.column_dimensions[get_column_letter(i)].width = 16

    # Гарчиг мөр оруулах
    ws.insert_rows(1)
    ws.merge_cells(start_row=1, start_column=1,
                   end_row=1,   end_column=len(DIST_ORDER) + 1)
    title_cell = ws.cell(row=1, column=1,
        value="Хүснэгт. УБ дүүргүүдээр орлогын ангиллын эзлэх хувь (2024)")
    title_cell.font      = Font(bold=True, name="Arial", size=11)
    title_cell.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 22

    # Эх сурвалж
    src_row = ws.max_row + 1
    ws.cell(row=src_row, column=1,
            value="Эх сурвалж: УСХ, судлаачийн тооцоолол").font = Font(
                italic=True, name="Arial", size=9)

print(f"\n✅ Excel хадгалагдлаа: {OUTPUT}")


2024: өрх=15511

q0115b тархалт:
dist_code
1107.0    12
1110.0    68
1113.0     1
1116.0    26
1119.0    13
1122.0    18
1125.0     7
2101.0     4
2104.0     1
2137.0     1
2201.0     1
2222.0     3
2228.0     1
4137.0     1
4152.0     1
4301.0     3
4322.0     1
4407.0     1
4431.0     1
4501.0     8
4504.0     1
4601.0    11
4607.0     1
4616.0     1
4622.0     1
4625.0     1
4631.0     9
4634.0     1
4640.0     1
4643.0     6
4801.0     1
4804.0     1
4819.0     1
4828.0     1
4834.0     3
4840.0     1
6201.0     1
6222.0     1
6322.0     1
6340.0     3
6401.0     1
6422.0     1
6501.0     1
6522.0     1
6537.0     1
6540.0     2
6701.0     8
6716.0     3
8101.0     3
8113.0     1
8116.0     1
8140.0     2
8146.0     1
8164.0     1
8201.0     2
8204.0     3
8234.0     1
8301.0     9
8316.0     8
8322.0    10
8337.0     3
8401.0     4
8404.0     1
8407.0     2
8434.0     1
8501.0     3
8507.0     1
8516.0     2
8540.0     1
8546.0     1

Дүүргээр өрхийн тоо:
district
Баянзүрх        

In [31]:
"""
Хүснэгт 5. Өрхийн ам бүлийн тоо ба хүн ам зүйн ачаалал, орлогын бүлгээр
2008 ба 2024 он

Тооцоолол:
  - Нийт ам бүл  = өрх (identif) тус бүрийн ind_id-ийн тоо
  - Хүүхэд       = q0105y < 18 насны гишүүдийн тоо
  - Ачаалал      = хүүхэд / нийт ам бүл
"""

import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
OECD_PATH  = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata_OECD_middle_class.xlsx"
INDIV_2024 = r"C:\Users\b22fa\Desktop\diplom unelgee\2024\02_indiv (1).xlsx"
INDIV_2008 = r"C:\Users\b22fa\Desktop\diplom unelgee\2008\Indivdual (1).xlsx"
OUTPUT     = r"C:\Users\b22fa\Desktop\diplom unelgee\husnegt5_ambul_achaалал.xlsx"

INC_ORDER = ["Бага", "Дундаж", "Өндөр"]

def find_col(df, keywords):
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in str(col).lower():
                return col
    return None

# ─── 1. OECD файл унших ─────────────────────
df = pd.read_excel(OECD_PATH, sheet_name=0, header=1)

year_col   = find_col(df, ['он', 'year'])
hh_col     = find_col(df, ['өрхийн дугаар', 'identif', 'hhid'])
weight_col = find_col(df, ['hhweight', 'жин'])
baga_col   = find_col(df, ['бага'])
dund_col   = find_col(df, ['дундаж'])
ondor_col  = find_col(df, ['өндөр'])

def get_cls(row):
    try:
        if row[dund_col]  == 1: return "Дундаж"
        if row[baga_col]  == 1: return "Бага"
        if row[ondor_col] == 1: return "Өндөр"
    except: pass
    return np.nan

df['inc_class'] = df.apply(get_cls, axis=1)
df['hh_id']     = pd.to_numeric(df[hh_col],     errors='coerce')
df['wt']        = pd.to_numeric(df[weight_col], errors='coerce')
df['yr']        = pd.to_numeric(df[year_col],   errors='coerce')

# ─── 2. indiv файлаас ам бүл, хүүхэд тооцох ─
def calc_hh_stats(indiv_path):
    indiv = pd.read_excel(indiv_path, header=0)
    print(f"\n  Баганууд: {list(indiv.columns[:10])}")

    id_col  = find_col(indiv, ['identif']) or indiv.columns[0]
    ind_col = find_col(indiv, ['ind_id'])
    age_col = find_col(indiv, ['q0105y'])

    print(f"  identif={id_col}, ind_id={ind_col}, age={age_col}")

    indiv['hh_id'] = pd.to_numeric(indiv[id_col],  errors='coerce')
    indiv['age']   = pd.to_numeric(indiv[age_col], errors='coerce')

    # Нийт ам бүл = өрх бүрт гишүүдийн тоо
    members = indiv.groupby('hh_id').size().reset_index(name='members')

    # Хүүхэд = нас < 18
    children = (
        indiv[indiv['age'] < 18]
        .groupby('hh_id')
        .size()
        .reset_index(name='children')
    )

    # Насанд хүрэгч = нас >= 18
    adults = (
        indiv[indiv['age'] >= 18]
        .groupby('hh_id')
        .size()
        .reset_index(name='adults')
    )

    hh_stats = members.merge(children, on='hh_id', how='left')
    hh_stats = hh_stats.merge(adults,  on='hh_id', how='left')
    hh_stats['children'] = hh_stats['children'].fillna(0)
    hh_stats['adults']   = hh_stats['adults'].fillna(0)
    # Ачаалал = хүүхэд / насанд хүрэгч
    hh_stats['achaалал'] = np.where(
        hh_stats['adults'] > 0,
        hh_stats['children'] / hh_stats['adults'],
        np.nan
    )

    print(f"  Нийт өрх: {len(hh_stats)}")
    print(f"  Ам бүлийн дундаж: {hh_stats['members'].mean():.2f}")
    print(f"  Хүүхдийн дундаж: {hh_stats['children'].mean():.2f}")

    return hh_stats

print("\n▶ 2024 indiv:")
hh24 = calc_hh_stats(INDIV_2024)

print("\n▶ 2008 indiv:")
hh08 = calc_hh_stats(INDIV_2008)

# ─── 3. OECD + hh_stats нийлүүлэх ──────────
def merge_and_calc(year, hh_stats):
    oecd = df[df['yr'] == year][['hh_id', 'wt', 'inc_class']].copy()
    m    = oecd.merge(hh_stats, on='hh_id', how='left')
    print(f"\n{year}: OECD={len(oecd)}, нийлэгдсэн members={m['members'].notna().sum()}")

    def weighted_mean(sub, col):
        sub = sub[sub[col].notna() & sub['wt'].notna()]
        tw  = sub['wt'].sum()
        return (sub[col] * sub['wt']).sum() / tw if tw > 0 else np.nan

    result = {}
    for inc in INC_ORDER:
        inc_sub = m[m['inc_class'] == inc]
        am  = weighted_mean(inc_sub, 'members')
        ach = weighted_mean(inc_sub, 'achaалал')
        result[inc] = {
            'ambul':    round(am,  1) if pd.notna(am)  else np.nan,
            'achaалал': round(ach, 2) if pd.notna(ach) else np.nan,
        }
        print(f"  {inc}: n={len(inc_sub)}, ам бүл={result[inc]['ambul']}, ачаалал={result[inc]['achaалал']}")
    return result

stats08 = merge_and_calc(2008, hh08)
stats24 = merge_and_calc(2024, hh24)

# ─── 4. DataFrame үүсгэх ────────────────────
rows = []
rows.append({"Орлогын ангилал": "2008 он", "Ам бүл": "", "Ачаалал": ""})
for inc in INC_ORDER:
    rows.append({
        "Орлогын ангилал": inc,
        "Ам бүл":  stats08[inc]['ambul']   if pd.notna(stats08[inc]['ambul'])   else "",
        "Ачаалал": stats08[inc]['achaалал'] if pd.notna(stats08[inc]['achaалал']) else "",
    })

rows.append({"Орлогын ангилал": "2024 он", "Ам бүл": "", "Ачаалал": ""})
for inc in INC_ORDER:
    rows.append({
        "Орлогын ангилал": inc,
        "Ам бүл":  stats24[inc]['ambul']   if pd.notna(stats24[inc]['ambul'])   else "",
        "Ачаалал": stats24[inc]['achaалал'] if pd.notna(stats24[inc]['achaалал']) else "",
    })

result_df = pd.DataFrame(rows)
print("\n=== ХҮСНЭГТ ===")
print(result_df.to_string(index=False))

# ─── 5. Excel хэвлэх ────────────────────────
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    result_df.to_excel(writer, sheet_name='Хүснэгт5', index=False)

    wb = writer.book
    ws = writer.sheets['Хүснэгт5']

    thin        = Side(style='thin')
    border      = Border(top=thin, bottom=thin, left=thin, right=thin)
    header_fill = PatternFill("solid", fgColor="1F4E79")
    year_fill   = PatternFill("solid", fgColor="BDD7EE")
    dund_fill   = PatternFill("solid", fgColor="D6E4F0")

    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.fill      = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border    = border

    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        label = str(row[0].value) if row[0].value is not None else ""
        for cell in row:
            cell.border    = border
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.font      = Font(name="Arial", size=10)

        if "он" in label:
            for cell in row:
                cell.fill = year_fill
                cell.font = Font(bold=True, name="Arial", size=10)
            ws.merge_cells(
                start_row=row[0].row, start_column=1,
                end_row=row[0].row,   end_column=3
            )
            row[0].alignment = Alignment(horizontal="center", vertical="center")
        elif label == "Дундаж":
            for cell in row:
                cell.fill = dund_fill
                cell.font = Font(bold=True, name="Arial", size=10)
            row[0].alignment = Alignment(horizontal="left", vertical="center")
        else:
            row[0].alignment = Alignment(horizontal="left", vertical="center")

    ws.column_dimensions['A'].width = 20
    ws.column_dimensions['B'].width = 12
    ws.column_dimensions['C'].width = 12

    ws.insert_rows(1)
    ws.merge_cells('A1:C1')
    title_cell = ws.cell(row=1, column=1,
        value="Хүснэгт 5. Өрхийн ам бүлийн тоо ба хүн ам зүйн ачаалал, орлогын бүлгээр")
    title_cell.font      = Font(bold=True, name="Arial", size=11)
    title_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 30

    src_row = ws.max_row + 1
    ws.cell(row=src_row, column=1,
            value="Эх сурвалж: УСХ, судлаачийн тооцоолол").font = Font(
                italic=True, name="Arial", size=9)

print(f"\n✅ Excel хадгалагдлаа: {OUTPUT}")


▶ 2024 indiv:

  Баганууд: ['identif', 'ind_id', 'q0102', 'q0103', 'q0105y', 'q0105m', 'q0106', 'q0107', 'q0108', 'q0109']
  identif=identif, ind_id=ind_id, age=q0105y
  Нийт өрх: 15513
  Ам бүлийн дундаж: 3.47
  Хүүхдийн дундаж: 1.28

▶ 2008 indiv:

  Баганууд: ['identif', 'ind_id', 'q0102', 'q0103', 'q0104', 'q0105y', 'q0105m', 'q0106', 'q0107', 'q0108']
  identif=identif, ind_id=ind_id, age=q0105y
  Нийт өрх: 11172
  Ам бүлийн дундаж: 3.98
  Хүүхдийн дундаж: 1.40

2008: OECD=11172, нийлэгдсэн members=11172
  Бага: n=4502, ам бүл=4.4, ачаалал=0.79
  Дундаж: n=4360, ам бүл=3.9, ачаалал=0.56
  Өндөр: n=2310, ам бүл=3.4, ачаалал=0.49

2024: OECD=15511, нийлэгдсэн members=15511
  Бага: n=6032, ам бүл=4.2, ачаалал=0.77
  Дундаж: n=7978, ам бүл=3.4, ачаалал=0.59
  Өндөр: n=1501, ам бүл=2.9, ачаалал=0.5

=== ХҮСНЭГТ ===
Орлогын ангилал Ам бүл Ачаалал
        2008 он               
           Бага    4.4    0.79
         Дундаж    3.9    0.56
          Өндөр    3.4    0.49
        2024 он  

In [33]:
"""
Хүснэгт 17. Хотын өрхийн сууцны төрөл
2008 ба 2024 он

2008 (q1302):
  1        = Гэр
  2        = Нийтийн орон сууц
  3        = Сууцны тусдаа байшин
  4, 5, 6  = Бусад

2024 (q0904):
  1        = Гэр
  2        = Нийтийн орон сууц
  3, 4     = Сууцны тусдаа байшин
  5, 6, 7  = Бусад
"""

import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
OECD_PATH   = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata_OECD_middle_class.xlsx"
HHOLD_2008  = r"C:\Users\b22fa\Desktop\diplom unelgee\2008\Household (1).xlsx"
INDIV_2024  = r"C:\Users\b22fa\Desktop\diplom unelgee\2024\01_hhold.xlsx"
OUTPUT      = r"C:\Users\b22fa\Desktop\diplom unelgee\husnegt17_suuts.xlsx"

INC_ORDER  = ["Бага", "Дундаж", "Өндөр"]
CAT_ORDER  = ["Гэр", "Нийтийн орон сууц", "Сууцны тусдаа байшин", "Бусад"]

# Mapping
MAP_2008 = {1: "Гэр", 2: "Нийтийн орон сууц", 3: "Сууцны тусдаа байшин",
            4: "Бусад", 5: "Бусад", 6: "Бусад"}
MAP_2024 = {1: "Гэр", 2: "Нийтийн орон сууц", 3: "Сууцны тусдаа байшин",
            4: "Сууцны тусдаа байшин", 5: "Бусад", 6: "Бусад", 7: "Бусад"}

def find_col(df, keywords):
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in str(col).lower():
                return col
    return None

# ─── 1. OECD файл унших ─────────────────────
df = pd.read_excel(OECD_PATH, sheet_name=0, header=1)

year_col   = find_col(df, ['он', 'year'])
hh_col     = find_col(df, ['өрхийн дугаар', 'identif', 'hhid'])
weight_col = find_col(df, ['hhweight', 'жин'])
baga_col   = find_col(df, ['бага'])
dund_col   = find_col(df, ['дундаж'])
ondor_col  = find_col(df, ['өндөр'])

def get_cls(row):
    try:
        if row[dund_col]  == 1: return "Дундаж"
        if row[baga_col]  == 1: return "Бага"
        if row[ondor_col] == 1: return "Өндөр"
    except: pass
    return np.nan

df['inc_class'] = df.apply(get_cls, axis=1)
df['hh_id']     = pd.to_numeric(df[hh_col],     errors='coerce')
df['wt']        = pd.to_numeric(df[weight_col], errors='coerce')
df['yr']        = pd.to_numeric(df[year_col],   errors='coerce')

# ─── 2. Сууцны төрөл файлуудаас авах ────────
def load_housing(path, question_col, mapping):
    hh = pd.read_excel(path, header=0)
    print(f"\n  Баганууд: {list(hh.columns[:15])}")
    id_col = find_col(hh, ['identif', 'hhid']) or hh.columns[0]
    q_col  = find_col(hh, [question_col])
    print(f"  id={id_col}, housing={q_col}")

    out = pd.DataFrame()
    out['hh_id']   = pd.to_numeric(hh[id_col], errors='coerce')
    out['housing'] = pd.to_numeric(hh[q_col],  errors='coerce').map(mapping)

    print(f"  housing тархалт:")
    print(out['housing'].value_counts().to_string())
    return out

print("\n▶ 2008 household:")
hh08 = load_housing(HHOLD_2008, 'q1302', MAP_2008)

print("\n▶ 2024 indiv:")
# 2024 indiv файлд өрх бүрт нэг утга авах (mode)
raw24 = pd.read_excel(INDIV_2024, header=0)
id_col24 = find_col(raw24, ['identif']) or raw24.columns[0]
q_col24  = find_col(raw24, ['q0904'])
print(f"  id={id_col24}, housing={q_col24}")
raw24['hh_id']   = pd.to_numeric(raw24[id_col24], errors='coerce')
raw24['housing_code'] = pd.to_numeric(raw24[q_col24], errors='coerce')
hh24 = (
    raw24.groupby('hh_id')['housing_code']
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan)
    .reset_index()
)
hh24['housing'] = hh24['housing_code'].map(MAP_2024)
hh24 = hh24[['hh_id', 'housing']]
print(f"  housing тархалт:")
print(hh24['housing'].value_counts().to_string())

# ─── 3. OECD + housing нийлүүлэх ────────────
def merge_yr(year, hh):
    oecd = df[df['yr'] == year][['hh_id', 'wt', 'inc_class']].copy()
    m    = oecd.merge(hh, on='hh_id', how='left')
    print(f"\n{year}: OECD={len(oecd)}, housing заагдсан={m['housing'].notna().sum()}")
    return m

m08 = merge_yr(2008, hh08)
m24 = merge_yr(2024, hh24)

# ─── 4. Жинлэсэн % тооцоо ───────────────────
def wpct_table(merged):
    """Орлогын анги × сууцны төрөл – жинлэсэн хувь (мөр нийт = 100%)"""
    result = {}
    for inc in INC_ORDER:
        sub   = merged[merged['inc_class'] == inc]
        total = sub['wt'].sum()
        row   = {}
        for cat in CAT_ORDER:
            w = sub[sub['housing'] == cat]['wt'].sum()
            row[cat] = round(w / total * 100) if total > 0 else np.nan
        row['Нийт'] = 100
        result[inc] = row
    return result

res08 = wpct_table(m08)
res24 = wpct_table(m24)

print("\n=== 2008 ===")
for inc in INC_ORDER:
    print(f"  {inc}: {res08[inc]}")
print("\n=== 2024 ===")
for inc in INC_ORDER:
    print(f"  {inc}: {res24[inc]}")

# ─── 5. DataFrame үүсгэх ────────────────────
cols = ["Орлогын ангилал"] + CAT_ORDER + ["Нийт"]
rows = []

rows.append({c: ("2008 он" if c == "Орлогын ангилал" else "") for c in cols})
for inc in INC_ORDER:
    r = {"Орлогын ангилал": inc}
    for cat in CAT_ORDER + ["Нийт"]:
        v = res08[inc].get(cat, "")
        r[cat] = f"{int(v)}%" if isinstance(v, (int, float)) and not np.isnan(v) else ""
    rows.append(r)

rows.append({c: ("2024 он" if c == "Орлогын ангилал" else "") for c in cols})
for inc in INC_ORDER:
    r = {"Орлогын ангилал": inc}
    for cat in CAT_ORDER + ["Нийт"]:
        v = res24[inc].get(cat, "")
        r[cat] = f"{int(v)}%" if isinstance(v, (int, float)) and not np.isnan(v) else ""
    rows.append(r)

result_df = pd.DataFrame(rows, columns=cols)
print("\n=== ХҮСНЭГТ ===")
print(result_df.to_string(index=False))

# ─── 6. Excel хэвлэх ────────────────────────
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    result_df.to_excel(writer, sheet_name='Хүснэгт17', index=False)

    wb = writer.book
    ws = writer.sheets['Хүснэгт17']

    thin        = Side(style='thin')
    border      = Border(top=thin, bottom=thin, left=thin, right=thin)
    header_fill = PatternFill("solid", fgColor="1F4E79")
    year_fill   = PatternFill("solid", fgColor="BDD7EE")
    dund_fill   = PatternFill("solid", fgColor="D6E4F0")

    # Толгой мөр
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.fill      = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = border
    ws.row_dimensions[2].height = 30

    total_cols = len(cols)

    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        label = str(row[0].value) if row[0].value is not None else ""
        for cell in row:
            cell.border    = border
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.font      = Font(name="Arial", size=10)

        if "он" in label:
            for cell in row:
                cell.fill = year_fill
                cell.font = Font(bold=True, name="Arial", size=10)
            ws.merge_cells(
                start_row=row[0].row, start_column=1,
                end_row=row[0].row,   end_column=total_cols
            )
            row[0].alignment = Alignment(horizontal="center", vertical="center")
        elif label == "Дундаж":
            for cell in row:
                cell.fill = dund_fill
                cell.font = Font(bold=True, name="Arial", size=10)
            row[0].alignment = Alignment(horizontal="left", vertical="center")
        else:
            row[0].alignment = Alignment(horizontal="left", vertical="center")

    # Багана өргөн
    ws.column_dimensions['A'].width = 22
    for i in range(2, total_cols + 1):
        ws.column_dimensions[get_column_letter(i)].width = 16

    # Гарчиг
    ws.insert_rows(1)
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=total_cols)
    title_cell = ws.cell(row=1, column=1,
        value="Хүснэгт 17. Хотын өрхийн сууцны төрөл")
    title_cell.font      = Font(bold=True, name="Arial", size=11)
    title_cell.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 22

    # Эх сурвалж
    src_row = ws.max_row + 1
    ws.cell(row=src_row, column=1,
            value="Эх сурвалж: УСХ, судлаачийн тооцоолол").font = Font(
                italic=True, name="Arial", size=9)

print(f"\n✅ Excel хадгалагдлаа: {OUTPUT}")


▶ 2008 household:

  Баганууд: ['identif', 'q0008', 'hh_no', 'v1_dd', 'v1_mm', 'v1_yy', 'v1_res', 'v2_dd', 'v2_mm', 'v2_yy', 'v2_res', 'v3_dd', 'v3_mm', 'v3_yy', 'v3_res']
  id=identif, housing=q1302
  housing тархалт:
housing
Гэр                     5341
Сууцны тусдаа байшин    3464
Нийтийн орон сууц       2304
Бусад                     63

▶ 2024 indiv:
  id=identif, housing=q0904
  housing тархалт:
housing
Гэр                     6581
Сууцны тусдаа байшин    5430
Нийтийн орон сууц       3289
Бусад                    213

2008: OECD=11172, housing заагдсан=11172

2024: OECD=15511, housing заагдсан=15511

=== 2008 ===
  Бага: {'Гэр': 65, 'Нийтийн орон сууц': 7, 'Сууцны тусдаа байшин': 27, 'Бусад': 1, 'Нийт': 100}
  Дундаж: {'Гэр': 42, 'Нийтийн орон сууц': 20, 'Сууцны тусдаа байшин': 37, 'Бусад': 1, 'Нийт': 100}
  Өндөр: {'Гэр': 18, 'Нийтийн орон сууц': 52, 'Сууцны тусдаа байшин': 29, 'Бусад': 0, 'Нийт': 100}

=== 2024 ===
  Бага: {'Гэр': 60, 'Нийтийн орон сууц': 10, 'Сууцны тусдаа ба

In [34]:
"""
Хүснэгт 18. Өрхийн автомашины тоо, орлогын бүлгээр
2008 ба 2024 он

2024 (14_durable.xlsx):
  durable_id=44 → Суудлын автомашин
  durable_id=46 → Мотоцикл
  тоо хэмжээ: q1101

2008 (Durable.xlsx):
  durable_id=31 → Суудлын автомашин
  durable_id=29 → Мотоцикл
  тоо хэмжээ: q1401
"""

import pandas as pd
import numpy as np

# ─────────────────────────────────────────────
OECD_PATH    = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata_OECD_middle_class.xlsx"
DURABLE_2024 = r"C:\Users\b22fa\Desktop\diplom unelgee\2024\14_durable.xlsx"
DURABLE_2008 = r"C:\Users\b22fa\Desktop\diplom unelgee\2008\Durable.xlsx"
OUTPUT       = r"C:\Users\b22fa\Desktop\diplom unelgee\husnegt18_avto.xlsx"

INC_ORDER = ["Бага", "Дундаж", "Өндөр"]

# durable_id тохиргоо
CFG = {
    2024: {"path": DURABLE_2024, "q_col": "q1101",
           "suudal_id": 46, "moto_id": 44},
    2008: {"path": DURABLE_2008, "q_col": "q1401",
           "suudal_id": 31, "moto_id": 29},
}

def find_col(df, keywords):
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in str(col).lower():
                return col
    return None

# ─── 1. OECD файл унших ─────────────────────
df = pd.read_excel(OECD_PATH, sheet_name=0, header=1)

year_col   = find_col(df, ['он', 'year'])
hh_col     = find_col(df, ['өрхийн дугаар', 'identif', 'hhid'])
weight_col = find_col(df, ['hhweight', 'жин'])
baga_col   = find_col(df, ['бага'])
dund_col   = find_col(df, ['дундаж'])
ondor_col  = find_col(df, ['өндөр'])

def get_cls(row):
    try:
        if row[dund_col]  == 1: return "Дундаж"
        if row[baga_col]  == 1: return "Бага"
        if row[ondor_col] == 1: return "Өндөр"
    except: pass
    return np.nan

df['inc_class'] = df.apply(get_cls, axis=1)
df['hh_id']     = pd.to_numeric(df[hh_col],     errors='coerce')
df['wt']        = pd.to_numeric(df[weight_col], errors='coerce')
df['yr']        = pd.to_numeric(df[year_col],   errors='coerce')

# ─── 2. Durable файлаас өрх бүрт тоо авах ──
def load_durable(year):
    cfg  = CFG[year]
    dur  = pd.read_excel(cfg["path"], header=0)
    print(f"\n▶ {year} durable баганууд: {list(dur.columns[:15])}")

    id_col  = find_col(dur, ['identif', 'hhid']) or dur.columns[0]
    dur_col = find_col(dur, ['durable_id', 'durable'])
    q_col   = cfg["q_col"] if cfg["q_col"] in dur.columns else find_col(dur, [cfg["q_col"]])

    print(f"  id={id_col}, durable_id={dur_col}, qty={q_col}")

    dur['hh_id']     = pd.to_numeric(dur[id_col],  errors='coerce')
    dur['dur_id']    = pd.to_numeric(dur[dur_col], errors='coerce')
    dur['qty']       = pd.to_numeric(dur[q_col],   errors='coerce')

    print(f"  durable_id өвөрмөц утгууд: {sorted(dur['dur_id'].dropna().unique().tolist())}")

    # Суудлын машин
    suudal = (
        dur[dur['dur_id'] == cfg["suudal_id"]]
        .groupby('hh_id')['qty']
        .sum()
        .reset_index(name='suudal')
    )
    # Мотоцикл
    moto = (
        dur[dur['dur_id'] == cfg["moto_id"]]
        .groupby('hh_id')['qty']
        .sum()
        .reset_index(name='moto')
    )

    print(f"  Суудлын машинтай өрх: {len(suudal)}, нийт тоо: {suudal['suudal'].sum()}")
    print(f"  Мотоциклтой өрх:      {len(moto)},   нийт тоо: {moto['moto'].sum()}")

    return suudal, moto

suudal24, moto24 = load_durable(2024)
suudal08, moto08 = load_durable(2008)

# ─── 3. OECD + durable нийлүүлэх ────────────
def merge_and_calc(year, suudal, moto):
    oecd = df[df['yr'] == year][['hh_id', 'wt', 'inc_class']].copy()

    # Бүх өрхийг нийлүүлж, тоо байхгүй бол 0
    m = oecd.merge(suudal, on='hh_id', how='left')
    m = m.merge(moto,   on='hh_id', how='left')
    m['suudal'] = m['suudal'].fillna(0)
    m['moto']   = m['moto'].fillna(0)

    print(f"\n{year}: OECD={len(oecd)}, нийлэгдсэн={len(m)}")

    def weighted_mean(sub, col):
        sub = sub[sub['wt'].notna()]
        tw  = sub['wt'].sum()
        return (sub[col] * sub['wt']).sum() / tw if tw > 0 else np.nan

    result = {}
    for inc in INC_ORDER:
        inc_sub = m[m['inc_class'] == inc]
        s = weighted_mean(inc_sub, 'suudal')
        mo = weighted_mean(inc_sub, 'moto')
        result[inc] = {
            'suudal': round(s,  2) if pd.notna(s)  else np.nan,
            'moto':   round(mo, 2) if pd.notna(mo) else np.nan,
        }
        print(f"  {inc}: суудлын={result[inc]['suudal']}, мото={result[inc]['moto']}, n={len(inc_sub)}")
    return result

res08 = merge_and_calc(2008, suudal08, moto08)
res24 = merge_and_calc(2024, suudal24, moto24)

# ─── 4. DataFrame үүсгэх ────────────────────
cols = ["Орлогын ангилал", "Суудлын автомашин", "Мотоцикл"]
rows = []

rows.append({"Орлогын ангилал": "2008 он", "Суудлын автомашин": "", "Мотоцикл": ""})
for inc in INC_ORDER:
    rows.append({
        "Орлогын ангилал":    inc,
        "Суудлын автомашин":  res08[inc]['suudal'] if pd.notna(res08[inc]['suudal']) else "",
        "Мотоцикл":           res08[inc]['moto']   if pd.notna(res08[inc]['moto'])   else "",
    })

rows.append({"Орлогын ангилал": "2024 он", "Суудлын автомашин": "", "Мотоцикл": ""})
for inc in INC_ORDER:
    rows.append({
        "Орлогын ангилал":    inc,
        "Суудлын автомашин":  res24[inc]['suudal'] if pd.notna(res24[inc]['suudal']) else "",
        "Мотоцикл":           res24[inc]['moto']   if pd.notna(res24[inc]['moto'])   else "",
    })

result_df = pd.DataFrame(rows, columns=cols)
print("\n=== ХҮСНЭГТ ===")
print(result_df.to_string(index=False))

# ─── 5. Excel хэвлэх ────────────────────────
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    result_df.to_excel(writer, sheet_name='Хүснэгт18', index=False)

    wb = writer.book
    ws = writer.sheets['Хүснэгт18']

    thin        = Side(style='thin')
    border      = Border(top=thin, bottom=thin, left=thin, right=thin)
    header_fill = PatternFill("solid", fgColor="1F4E79")
    year_fill   = PatternFill("solid", fgColor="BDD7EE")
    dund_fill   = PatternFill("solid", fgColor="D6E4F0")

    # Толгой мөр
    for cell in ws[1]:
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.fill      = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = border

    # Өгөгдлийн мөрүүд
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        label = str(row[0].value) if row[0].value is not None else ""
        for cell in row:
            cell.border    = border
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.font      = Font(name="Arial", size=10)

        if "он" in label:
            for cell in row:
                cell.fill = year_fill
                cell.font = Font(bold=True, name="Arial", size=10)
            ws.merge_cells(
                start_row=row[0].row, start_column=1,
                end_row=row[0].row,   end_column=len(cols)
            )
            row[0].alignment = Alignment(horizontal="center", vertical="center")
        elif label == "Дундаж":
            for cell in row:
                cell.fill = dund_fill
                cell.font = Font(bold=True, name="Arial", size=10)
            row[0].alignment = Alignment(horizontal="left", vertical="center")
        else:
            row[0].alignment = Alignment(horizontal="left", vertical="center")

    # Багана өргөн
    ws.column_dimensions['A'].width = 20
    ws.column_dimensions['B'].width = 20
    ws.column_dimensions['C'].width = 14

    # Гарчиг
    ws.insert_rows(1)
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=len(cols))
    title_cell = ws.cell(row=1, column=1,
        value="Хүснэгт 18. Өрхийн автомашины тоо, орлогын бүлгээр (өрх тус бүрт дундаж)")
    title_cell.font      = Font(bold=True, name="Arial", size=11)
    title_cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[1].height = 28

    # Эх сурвалж
    src_row = ws.max_row + 1
    ws.cell(row=src_row, column=1,
            value="Эх сурвалж: УСХ, судлаачийн тооцоолол").font = Font(
                italic=True, name="Arial", size=9)

print(f"\n✅ Excel хадгалагдлаа: {OUTPUT}")


▶ 2024 durable баганууд: ['identif', 'durable_id', 'q1101', 'q1102', 'q1103', 'q1104']
  id=identif, durable_id=durable_id, qty=q1101
  durable_id өвөрмөц утгууд: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
  Суудлын машинтай өрх: 15513, нийт тоо: 8854.0
  Мотоциклтой өрх:      15513,   нийт тоо: 3593.0

▶ 2008 durable баганууд: ['identif', 'durable_id', 'q1401', 'q1402', 'q1403']
  id=identif, durable_id=durable_id, qty=q1401
  durable_id өвөрмөц утгууд: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42]
  Суудлын машинтай өрх: 11167, нийт тоо: 2019.0
  Мотоциклтой өрх:      11165,   нийт тоо: 1739.0

2008: OECD=11172, нийлэгдсэн=11172
  Бага: суудлын=0.11, мото=0.27, n=4502
  Дундаж: суудлын=0.16, мото=0.08, n=4360
  Өндөр: су

In [35]:
"""
Хүснэгт 10. Малчин ба малгүй өрхүүд орлогын бүлгээр
2008 ба 2024 он

2024 (01_hhold.xlsx):  q0601 → 1=Малчин, 2=Малгүй
2008 (Household.xlsx): q0923 → 1=Малчин, 2=Малгүй
"""

import pandas as pd
import numpy as np
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

# ─────────────────────────────────────────────
OECD_PATH    = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata_OECD_middle_class.xlsx"
HHOLD_2024   = r"C:\Users\b22fa\Desktop\diplom unelgee\2024\01_hhold.xlsx"
HHOLD_2008   = r"C:\Users\b22fa\Desktop\diplom unelgee\2008\Household (1).xlsx"
OUTPUT       = r"C:\Users\b22fa\Desktop\diplom unelgee\husnegt10_malchin.xlsx"

INC_ORDER = ["Бага", "Дундаж", "Өндөр"]

CFG = {
    2024: {"path": HHOLD_2024, "q_col": "q0601"},
    2008: {"path": HHOLD_2008, "q_col": "q0923"},
}

def find_col(df, keywords):
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in str(col).lower():
                return col
    return None

# ─── 1. OECD файл унших ─────────────────────
df = pd.read_excel(OECD_PATH, sheet_name=0, header=1)

year_col   = find_col(df, ['он', 'year'])
hh_col     = find_col(df, ['өрхийн дугаар', 'identif', 'hhid'])
weight_col = find_col(df, ['hhweight', 'жин'])
baga_col   = find_col(df, ['бага'])
dund_col   = find_col(df, ['дундаж'])
ondor_col  = find_col(df, ['өндөр'])

def get_cls(row):
    try:
        if row[dund_col]  == 1: return "Дундаж"
        if row[baga_col]  == 1: return "Бага"
        if row[ondor_col] == 1: return "Өндөр"
    except: pass
    return np.nan

df['inc_class'] = df.apply(get_cls, axis=1)
df['hh_id']     = pd.to_numeric(df[hh_col],     errors='coerce')
df['wt']        = pd.to_numeric(df[weight_col], errors='coerce')
df['yr']        = pd.to_numeric(df[year_col],   errors='coerce')

# ─── 2. Hhold файлаас малчин/малгүй авах ────
def load_hhold(year):
    cfg  = CFG[year]
    hh   = pd.read_excel(cfg["path"], header=0)
    print(f"\n▶ {year} hhold баганууд: {list(hh.columns[:15])}")

    id_col = find_col(hh, ['identif', 'hhid']) or hh.columns[0]
    q_col  = cfg["q_col"] if cfg["q_col"] in hh.columns else find_col(hh, [cfg["q_col"]])

    print(f"  id={id_col}, q_col={q_col}")

    hh['hh_id']   = pd.to_numeric(hh[id_col], errors='coerce')
    hh['mal_raw'] = pd.to_numeric(hh[q_col],  errors='coerce')

    # 1=Малчин, 2=Малгүй
    hh['mal_type'] = hh['mal_raw'].map({1: 'Малчин', 2: 'Малгүй'})

    print(f"  Малчин өрх:  {(hh['mal_type']=='Малчин').sum()}")
    print(f"  Малгүй өрх:  {(hh['mal_type']=='Малгүй').sum()}")
    print(f"  Бусад/хоосон: {hh['mal_type'].isna().sum()}")

    return hh[['hh_id', 'mal_type']]

hhold24 = load_hhold(2024)
hhold08 = load_hhold(2008)

# ─── 3. OECD + hhold нийлүүлж тооцоолох ────
def merge_and_calc(year, hhold):
    oecd = df[df['yr'] == year][['hh_id', 'wt', 'inc_class']].copy()

    m = oecd.merge(hhold, on='hh_id', how='left')
    print(f"\n{year}: OECD={len(oecd)}, нийлэгдсэн={len(m)}")

    result = {}
    for mal in ['Малчин', 'Малгүй']:
        mal_sub = m[m['mal_type'] == mal].copy()
        result[mal] = {}
        total_wt = mal_sub['wt'].sum()
        for inc in INC_ORDER:
            inc_sub = mal_sub[mal_sub['inc_class'] == inc]
            wt_sum  = inc_sub['wt'].sum()
            pct     = (wt_sum / total_wt * 100) if total_wt > 0 else np.nan
            result[mal][inc] = round(pct, 1) if pd.notna(pct) else np.nan
            print(f"  {mal} | {inc}: {result[mal][inc]}%  (n={len(inc_sub)})")

        # Нийт шалгах (100% байх ёстой)
        total_pct = sum(result[mal][i] for i in INC_ORDER if pd.notna(result[mal][i]))
        print(f"  {mal} нийт: {round(total_pct,1)}%")

    return result

res08 = merge_and_calc(2008, hhold08)
res24 = merge_and_calc(2024, hhold24)

# ─── 4. DataFrame үүсгэх ────────────────────
cols = ["Орлогын ангилал",
        "Малчин өрхүүд 2008", "Малчин өрхүүд 2024",
        "Малгүй өрхүүд 2008", "Малгүй өрхүүд 2024"]

rows = []
for inc in INC_ORDER:
    bold = (inc == "Дундаж")
    rows.append({
        "Орлогын ангилал":    inc,
        "Малчин өрхүүд 2008": f"{res08['Малчин'][inc]}%" if pd.notna(res08['Малчин'][inc]) else "",
        "Малчин өрхүүд 2024": f"{res24['Малчин'][inc]}%" if pd.notna(res24['Малчин'][inc]) else "",
        "Малгүй өрхүүд 2008": f"{res08['Малгүй'][inc]}%" if pd.notna(res08['Малгүй'][inc]) else "",
        "Малгүй өрхүүд 2024": f"{res24['Малгүй'][inc]}%" if pd.notna(res24['Малгүй'][inc]) else "",
    })

# Нийт мөр
rows.append({
    "Орлогын ангилал":    "Нийт",
    "Малчин өрхүүд 2008": "100%",
    "Малчин өрхүүд 2024": "100%",
    "Малгүй өрхүүд 2008": "100%",
    "Малгүй өрхүүд 2024": "100%",
})

result_df = pd.DataFrame(rows, columns=cols)
print("\n=== ХҮСНЭГТ ===")
print(result_df.to_string(index=False))

# ─── 5. Excel хэвлэх ────────────────────────
with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    result_df.to_excel(writer, sheet_name='Хүснэгт10', index=False, startrow=3)

    wb = writer.book
    ws = writer.sheets['Хүснэгт10']

    thin        = Side(style='thin')
    thick       = Side(style='medium')
    border      = Border(top=thin, bottom=thin, left=thin, right=thin)
    header_fill = PatternFill("solid", fgColor="1F4E79")
    sub_fill    = PatternFill("solid", fgColor="BDD7EE")
    dund_fill   = PatternFill("solid", fgColor="D6E4F0")
    total_fill  = PatternFill("solid", fgColor="E2EFDA")

    # ── Гарчиг мөр (1-р мөр) ──
    ws.merge_cells('A1:E1')
    title = ws.cell(row=1, column=1,
        value="Хүснэгт 10. Малчин ба малгүй өрхүүд орлогын бүлгээр")
    title.font      = Font(bold=True, italic=True, name="Arial", size=11)
    title.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[1].height = 22

    # ── Дэд толгой: Малчин өрхүүд / Малгүй өрхүүд (2-р мөр) ──
    ws.merge_cells('B2:C2')
    h1 = ws.cell(row=2, column=2, value="Малчин өрхүүд")
    h1.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    h1.fill      = header_fill
    h1.alignment = Alignment(horizontal="center", vertical="center")
    h1.border    = border

    ws.merge_cells('D2:E2')
    h2 = ws.cell(row=2, column=4, value="Малгүй өрхүүд")
    h2.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    h2.fill      = header_fill
    h2.alignment = Alignment(horizontal="center", vertical="center")
    h2.border    = border

    ws.cell(row=2, column=1).border = border

    # ── Он толгой (3-р мөр) ──
    year_headers = ["", "2008 он", "2024 он", "2008 он", "2024 он"]
    for c, val in enumerate(year_headers, start=1):
        cell = ws.cell(row=3, column=c, value=val)
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.fill      = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border    = border
    ws.row_dimensions[3].height = 18

    # ── Өгөгдлийн мөрүүд (4-р мөрөөс) ──
    # startrow=3 тул өгөгдөл 5-р мөрөөс эхэлнэ (header=4)
    for row in ws.iter_rows(min_row=4, max_row=ws.max_row):
        label = str(row[0].value) if row[0].value is not None else ""
        for cell in row:
            cell.border    = border
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.font      = Font(name="Arial", size=10)
        row[0].alignment = Alignment(horizontal="left", vertical="center")

        if label == "Дундаж":
            for cell in row:
                cell.fill = dund_fill
                cell.font = Font(bold=True, name="Arial", size=10)
            row[0].alignment = Alignment(horizontal="left", vertical="center")
        elif label == "Нийт":
            for cell in row:
                cell.fill = total_fill
                cell.font = Font(bold=True, name="Arial", size=10)
            row[0].alignment = Alignment(horizontal="left", vertical="center")

    # Автомат толгой мөрийг (startrow=3-аас үүссэн) арилгах
    # 4-р мөр бол pandas-ын header → устгах
    ws.delete_rows(4)

    # ── Багана өргөн ──
    ws.column_dimensions['A'].width = 18
    ws.column_dimensions['B'].width = 14
    ws.column_dimensions['C'].width = 14
    ws.column_dimensions['D'].width = 14
    ws.column_dimensions['E'].width = 14

    # ── Эх сурвалж ──
    src_row = ws.max_row + 1
    ws.cell(row=src_row, column=1,
            value="Эх сурвалж: УСХ, судлаачийн тооцоолол").font = Font(
                italic=True, name="Arial", size=9)

print(f"\n✅ Excel хадгалагдлаа: {OUTPUT}")


▶ 2024 hhold баганууд: ['identif', 'years', 'hh_no', 'h12', 'identif22', 'cover_visit_count', 'v1_dd', 'v1_mm', 'v1_yy', 'v1_res', 'v2_dd', 'v2_mm', 'v2_yy', 'v2_res', 'v3_dd']
  id=identif, q_col=q0601
  Малчин өрх:  6011
  Малгүй өрх:  9502
  Бусад/хоосон: 0

▶ 2008 hhold баганууд: ['identif', 'q0008', 'hh_no', 'v1_dd', 'v1_mm', 'v1_yy', 'v1_res', 'v2_dd', 'v2_mm', 'v2_yy', 'v2_res', 'v3_dd', 'v3_mm', 'v3_yy', 'v3_res']
  id=identif, q_col=q0923q
  Малчин өрх:  999
  Малгүй өрх:  3316
  Бусад/хоосон: 6857

2008: OECD=11172, нийлэгдсэн=11172
  Малчин | Бага: 63.8%  (n=639)
  Малчин | Дундаж: 26.8%  (n=268)
  Малчин | Өндөр: 9.4%  (n=92)
  Малчин нийт: 100.0%
  Малгүй | Бага: 62.9%  (n=2071)
  Малгүй | Дундаж: 28.5%  (n=949)
  Малгүй | Өндөр: 8.6%  (n=296)
  Малгүй нийт: 100.0%

2024: OECD=15511, нийлэгдсэн=15511
  Малчин | Бага: 64.6%  (n=3684)
  Малчин | Дундаж: 30.8%  (n=2044)
  Малчин | Өндөр: 4.6%  (n=282)
  Малчин нийт: 100.0%
  Малгүй | Бага: 19.4%  (n=2348)
  Малгүй | Дундаж: 

In [36]:
"""
Хүснэгт. Бизнестэй ба бизнесгүй өрхүүд орлогын бүлгээр
2008 ба 2024 он

2024 (08_enterprise.xlsx): identif байвал = Бизнестэй
2008 (Enterprise.xlsx):    identif байвал = Бизнестэй
OECD файлд байгаа бүх өрх — enterprise-д байхгүй бол = Бизнесгүй
"""

import pandas as pd
import numpy as np
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

# ─────────────────────────────────────────────
OECD_PATH      = r"C:\Users\b22fa\Desktop\diplom unelgee\alldata_OECD_middle_class.xlsx"
ENTERPRISE_2024 = r"C:\Users\b22fa\Desktop\diplom unelgee\2024\08_enterprise.xlsx"
ENTERPRISE_2008 = r"C:\Users\b22fa\Desktop\diplom unelgee\2008\Enterprise.xlsx"
OUTPUT          = r"C:\Users\b22fa\Desktop\diplom unelgee\husnegt_business.xlsx"

INC_ORDER = ["Бага", "Дундаж", "Өндөр"]

def find_col(df, keywords):
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in str(col).lower():
                return col
    return None

# ─── 1. OECD файл унших ─────────────────────
df = pd.read_excel(OECD_PATH, sheet_name=0, header=1)

year_col   = find_col(df, ['он', 'year'])
hh_col     = find_col(df, ['өрхийн дугаар', 'identif', 'hhid'])
weight_col = find_col(df, ['hhweight', 'жин'])
baga_col   = find_col(df, ['бага'])
dund_col   = find_col(df, ['дундаж'])
ondor_col  = find_col(df, ['өндөр'])

def get_cls(row):
    try:
        if row[dund_col]  == 1: return "Дундаж"
        if row[baga_col]  == 1: return "Бага"
        if row[ondor_col] == 1: return "Өндөр"
    except: pass
    return np.nan

df['inc_class'] = df.apply(get_cls, axis=1)
df['hh_id']     = pd.to_numeric(df[hh_col],     errors='coerce')
df['wt']        = pd.to_numeric(df[weight_col], errors='coerce')
df['yr']        = pd.to_numeric(df[year_col],   errors='coerce')

# ─── 2. Enterprise файлаас бизнестэй өрхийн ID авах ──
def load_enterprise(year, path):
    ent = pd.read_excel(path, header=0)
    print(f"\n▶ {year} enterprise баганууд: {list(ent.columns[:15])}")

    id_col = find_col(ent, ['identif', 'hhid'])
    if id_col is None:
        id_col = ent.columns[0]
    print(f"  id багана: {id_col}")

    ent['hh_id'] = pd.to_numeric(ent[id_col], errors='coerce')

    # Давхардлыг арилгаж, бизнестэй өрхийн ID цуглуулах
    biz_ids = set(ent['hh_id'].dropna().unique())
    print(f"  Бизнестэй өрх (өвөрмөц): {len(biz_ids)}")
    return biz_ids

biz_ids_24 = load_enterprise(2024, ENTERPRISE_2024)
biz_ids_08 = load_enterprise(2008, ENTERPRISE_2008)

# ─── 3. OECD өрхөд бизнес төрөл тогтоох ────
def assign_biz_type(year, biz_ids):
    oecd = df[df['yr'] == year][['hh_id', 'wt', 'inc_class']].copy()

    oecd['biz_type'] = oecd['hh_id'].apply(
        lambda x: 'Бизнестэй' if x in biz_ids else 'Бизнесгүй'
    )

    print(f"\n{year}: Нийт өрх={len(oecd)}")
    print(f"  Бизнестэй: {(oecd['biz_type']=='Бизнестэй').sum()}")
    print(f"  Бизнесгүй: {(oecd['biz_type']=='Бизнесгүй').sum()}")
    return oecd

oecd24 = assign_biz_type(2024, biz_ids_24)
oecd08 = assign_biz_type(2008, biz_ids_08)

# ─── 4. Хувь тооцоолох ──────────────────────
def calc_pct(oecd, year):
    result = {}
    for biz in ['Бизнестэй', 'Бизнесгүй']:
        biz_sub  = oecd[oecd['biz_type'] == biz].copy()
        total_wt = biz_sub['wt'].sum()
        result[biz] = {}
        for inc in INC_ORDER:
            inc_sub = biz_sub[biz_sub['inc_class'] == inc]
            wt_sum  = inc_sub['wt'].sum()
            pct     = (wt_sum / total_wt * 100) if total_wt > 0 else np.nan
            result[biz][inc] = round(pct, 1) if pd.notna(pct) else np.nan
            print(f"  {year} | {biz} | {inc}: {result[biz][inc]}%  (n={len(inc_sub)})")

        total_pct = sum(result[biz][i] for i in INC_ORDER if pd.notna(result[biz][i]))
        print(f"  {year} | {biz} нийт: {round(total_pct, 1)}%")
    return result

print("\n── 2008 тооцоолол ──")
res08 = calc_pct(oecd08, 2008)
print("\n── 2024 тооцоолол ──")
res24 = calc_pct(oecd24, 2024)

# ─── 5. DataFrame үүсгэх ────────────────────
cols = ["Орлогын ангилал",
        "Бизнестэй өрхүүд 2008", "Бизнестэй өрхүүд 2024",
        "Бизнесгүй өрхүүд 2008", "Бизнесгүй өрхүүд 2024"]

rows = []
for inc in INC_ORDER:
    rows.append({
        "Орлогын ангилал":        inc,
        "Бизнестэй өрхүүд 2008":  f"{res08['Бизнестэй'][inc]}%" if pd.notna(res08['Бизнестэй'][inc]) else "",
        "Бизнестэй өрхүүд 2024":  f"{res24['Бизнестэй'][inc]}%" if pd.notna(res24['Бизнестэй'][inc]) else "",
        "Бизнесгүй өрхүүд 2008":  f"{res08['Бизнесгүй'][inc]}%" if pd.notna(res08['Бизнесгүй'][inc]) else "",
        "Бизнесгүй өрхүүд 2024":  f"{res24['Бизнесгүй'][inc]}%" if pd.notna(res24['Бизнесгүй'][inc]) else "",
    })

rows.append({
    "Орлогын ангилал":        "Нийт",
    "Бизнестэй өрхүүд 2008":  "100%",
    "Бизнестэй өрхүүд 2024":  "100%",
    "Бизнесгүй өрхүүд 2008":  "100%",
    "Бизнесгүй өрхүүд 2024":  "100%",
})

result_df = pd.DataFrame(rows, columns=cols)
print("\n=== ХҮСНЭГТ ===")
print(result_df.to_string(index=False))

# ─── 6. Excel хэвлэх ────────────────────────
with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    result_df.to_excel(writer, sheet_name='Бизнес', index=False, startrow=3)

    wb = writer.book
    ws = writer.sheets['Бизнес']

    thin        = Side(style='thin')
    border      = Border(top=thin, bottom=thin, left=thin, right=thin)
    header_fill = PatternFill("solid", fgColor="1F4E79")
    dund_fill   = PatternFill("solid", fgColor="D6E4F0")
    total_fill  = PatternFill("solid", fgColor="E2EFDA")

    # ── Гарчиг (1-р мөр) ──
    ws.merge_cells('A1:E1')
    title = ws.cell(row=1, column=1,
        value="Хүснэгт. Бизнестэй ба бизнесгүй өрхүүд орлогын бүлгээр")
    title.font      = Font(bold=True, italic=True, name="Arial", size=11)
    title.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[1].height = 22

    # ── Дэд толгой (2-р мөр) ──
    ws.merge_cells('B2:C2')
    h1 = ws.cell(row=2, column=2, value="Бизнестэй өрхүүд")
    h1.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    h1.fill      = header_fill
    h1.alignment = Alignment(horizontal="center", vertical="center")
    h1.border    = border

    ws.merge_cells('D2:E2')
    h2 = ws.cell(row=2, column=4, value="Бизнесгүй өрхүүд")
    h2.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    h2.fill      = header_fill
    h2.alignment = Alignment(horizontal="center", vertical="center")
    h2.border    = border

    ws.cell(row=2, column=1).border = border

    # ── Он толгой (3-р мөр) ──
    for c, val in enumerate(["", "2008 он", "2024 он", "2008 он", "2024 он"], start=1):
        cell = ws.cell(row=3, column=c, value=val)
        cell.font      = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.fill      = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border    = border
    ws.row_dimensions[3].height = 18

    # ── Өгөгдлийн мөрүүд ──
    for row in ws.iter_rows(min_row=4, max_row=ws.max_row):
        label = str(row[0].value) if row[0].value is not None else ""
        for cell in row:
            cell.border    = border
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.font      = Font(name="Arial", size=10)
        row[0].alignment = Alignment(horizontal="left", vertical="center")

        if label == "Дундаж":
            for cell in row:
                cell.fill = dund_fill
                cell.font = Font(bold=True, name="Arial", size=10)
            row[0].alignment = Alignment(horizontal="left", vertical="center")
        elif label == "Нийт":
            for cell in row:
                cell.fill = total_fill
                cell.font = Font(bold=True, name="Arial", size=10)
            row[0].alignment = Alignment(horizontal="left", vertical="center")

    # Pandas автомат толгой мөр устгах
    ws.delete_rows(4)

    # ── Багана өргөн ──
    ws.column_dimensions['A'].width = 18
    ws.column_dimensions['B'].width = 18
    ws.column_dimensions['C'].width = 18
    ws.column_dimensions['D'].width = 18
    ws.column_dimensions['E'].width = 18

    # ── Эх сурвалж ──
    src_row = ws.max_row + 1
    ws.cell(row=src_row, column=1,
            value="Эх сурвалж: УСХ, судлаачийн тооцоолол").font = Font(
                italic=True, name="Arial", size=9)

print(f"\n✅ Excel хадгалагдлаа: {OUTPUT}")


▶ 2024 enterprise баганууд: ['identif', 'en', 'q0703', 'q0704', 'q0705', 'q0706', 'q0707_01', 'q0707_02', 'q0707_03', 'q0707_04', 'q0707_05', 'q0707_06', 'q0707_07', 'q0707_08', 'q0707_09']
  id багана: identif
  Бизнестэй өрх (өвөрмөц): 1498

▶ 2008 enterprise баганууд: ['identif', 'en', 'q1002c', 'q1003', 'q1004', 'q1005q', 'q1006q', 'q1007q', 'q1008_1', 'q1008_2', 'q1009_01', 'q1009_02', 'q1009_03', 'q1009_04', 'q1009_05']
  id багана: identif
  Бизнестэй өрх (өвөрмөц): 2157

2024: Нийт өрх=15511
  Бизнестэй: 1498
  Бизнесгүй: 14013

2008: Нийт өрх=11172
  Бизнестэй: 2157
  Бизнесгүй: 9015

── 2008 тооцоолол ──
  2008 | Бизнестэй | Бага: 30.6%  (n=673)
  2008 | Бизнестэй | Дундаж: 38.1%  (n=817)
  2008 | Бизнестэй | Өндөр: 31.2%  (n=667)
  2008 | Бизнестэй нийт: 99.9%
  2008 | Бизнесгүй | Бага: 40.5%  (n=3829)
  2008 | Бизнесгүй | Дундаж: 39.7%  (n=3543)
  2008 | Бизнесгүй | Өндөр: 19.8%  (n=1643)
  2008 | Бизнесгүй нийт: 100.0%

── 2024 тооцоолол ──
  2024 | Бизнестэй | Бага: 22.2